# XGuard — Unknown-Threat Discovery under Compound Distribution Shift in X-ray Security Inspection

One notebook, two platforms: **Tier 1 on Colab Pro (A100)** and **Tiers 2–3 on Kaggle (free GPU)**. Everything that does not need a GPU happens on a CPU runtime, so no GPU time is spent on downloads, audits, CLIP loading or tables.

## Colab Pro (Tier 1: core with 3 seeds + component ablation): `MODE = "auto"`, so you never edit the code
You only choose the **runtime type** and click **Run all**. The notebook decides which step to run:

| You open | Data pack on Drive? | It runs |
|---|---|---|
| **CPU** runtime | no | **prepare**: Kaggle download → verification → leakage audit → pack on Drive (add the Colab secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` first) |
| **A100** runtime | yes, test not passed yet | **test**: every tier-1 code path on real data, plus the GPU-hours / compute-unit estimate. The runtime is released at the end. |
| **A100** runtime | yes, test passed | **full**: the tier-1 experiments. Resumable: after a disconnect, just click Run all again. The runtime is released when the work is done or after 20 idle minutes. |
| **CPU** runtime | yes | **report**: merges Colab + Kaggle results into tables, figures and gates |
| **A100** runtime | **no** | stops at once and **releases the GPU** (prepare must run on a CPU runtime) |

If an error ever stops the notebook: **Runtime → Disconnect and delete runtime** releases the GPU immediately.

## Kaggle (Tiers 2–3: LOCO first, then the supporting ablations, baselines, zero-shot, in-domain, LODO, encoders, YOLO / RT-DETR)
1. **CPU session.** Add Input: `khanhbtq99/sixray`, `witsarutsarai/sixray-p1`, `banan41k/xray-contraband-detection`, `newashish/hixray`, `alirzaei/opixray-yolo`, and optionally your OPIXray OL lists. Set `MODE="prepare"` and choose *Save & Run All*. The pack is written to the output.
2. **GPU session** (Accelerator **GPU T4×2/P100**, Internet **on**). Add Input: the output of step 1. Set `MODE="test"`, then `MODE="full"`, using *Save & Run All* so the run continues when the browser is closed. Each session lasts at most 12 h; add the previous output as input and run again to continue.
3. Download `xguard_results_full.zip` and put it into `MyDrive/xguard/kaggle_results/` for step 4 above.

Kaggle uses batch 4 with gradient accumulation 2, which gives the same effective batch of 8 as Colab, so the results are comparable.

| Key | Kaggle dataset | Verified content (real Kaggle run) |
|---|---|---|
| sixray | `khanhbtq99/sixray` | Roboflow export with augmented copies and **5 classes (no hammer)**; collapsed to one copy per original image ID, IDs never shared across splits |
| sixray-p1 | `witsarutsarai/sixray-p1` (first part only on Colab) | clean bags for the false-alarm metrics |
| pidray | `banan41k/xray-contraband-detection` | PIDray; test 18,220 = 9,482 easy + 3,733 hard + 5,005 hidden (matches the paper) |
| hixray | `newashish/hixray` | official 36,295 / 9,069 |
| opixray | `alirzaei/opixray-yolo` | official test of 1,776 split into val + test, merged back here |

**Honesty rules in the code:**
- No target data is used for training, thresholds or checkpoint choice.
- Exact, harmonized and hierarchical AP are reported separately.
- Leakage exclusions are frozen before any evaluation.
- Metrics that do not apply show `n/a` with a reason.
- Fingerprints prevent stale results from being reused.
- Oracle-vocabulary zero-shot runs are labelled as references only.
- Methods without public code are not re-implemented.
- The model is a two-stage R50-FPN, not DETR, for compute reasons; state this in the paper.


## 0 · Configuration
Set `MODE` (`prepare` → `test` → `full`) and `TIERS`. Nothing else needs editing.

In [ ]:
# ==============================================================================
# 0. CONFIGURATION  (edit this cell only)
# ==============================================================================
import os, sys, json, time, math, random, re, glob, shutil, pickle, hashlib, warnings, itertools
import contextlib, io, gc, copy, traceback, platform, subprocess, inspect
from pathlib import Path
from collections import defaultdict, Counter, OrderedDict
from types import SimpleNamespace
import numpy as np

warnings.filterwarnings("ignore")
T_SESSION_START = time.time()
CODE_VERSION = "xguard-v2.0"          # bump when you change model/training code on purpose (part of every run fingerprint)

# ------------------------------------------------------------------------------
# WHERE / WHAT TO RUN
#   Colab Pro A100 : Tier 1  = core (base vs XGuard, 3 seeds, full test sets) + component ablation
#   Kaggle GPU     : Tier 2  = LOCO, prototype / unknown-generation / SIRA / occlusion ablations, baselines, zero-shot
#                    Tier 3  = in-domain, LODO, encoder, #sources, YOLO / RT-DETR
#
# MODE
#   "prepare" : CPU runtime/session (no GPU cost). Data -> verification -> leakage audit -> compact pack
#               (+ CLIP text embeddings + COCO weights cached inside the pack). Colab: pack on Google Drive.
#               Kaggle: pack in /kaggle/working (attach this notebook's output as input to the GPU sessions).
#   "test"    : GPU. Every stage on real data with a few iterations + measured time / compute-unit estimate.
#   "full"    : GPU. The real experiments of this platform's tiers (resumable, fingerprinted).
#   "report"  : CPU. Merges Colab + Kaggle results -> all tables, figures, release gates (no GPU cost).
#   "auto" (default) -- just pick the runtime type and press Run all:
#        no pack yet + CPU  -> prepare          no pack yet + GPU -> stop at once and RELEASE the GPU
#        pack + GPU, test not passed -> test    pack + GPU, test passed -> full          pack + CPU -> report
# ------------------------------------------------------------------------------
MODE = "auto"          # "auto" decides by itself (see the table below); or force "prepare" / "test" / "full" / "report"
TIERS = None           # None = this platform's default (Colab [1], Kaggle [2, 3]); or e.g. [2]
SMOKE = os.environ.get("XGUARD_SMOKE", "0") == "1"   # tiny CPU self-test used by the author; leave it alone
if os.environ.get("XGUARD_MODE"):
    MODE = os.environ["XGUARD_MODE"]
if os.environ.get("XGUARD_TIERS"):
    TIERS = [int(t) for t in os.environ["XGUARD_TIERS"].split(",")]

# ---- platform & paths -----------------------------------------------------------
PLATFORM = "colab" if "google.colab" in sys.modules or os.path.exists("/content/sample_data") else \
           "kaggle" if os.path.exists("/kaggle") else "local"
if os.environ.get("XGUARD_PLATFORM"):
    PLATFORM = os.environ["XGUARD_PLATFORM"]
if TIERS is None:
    TIERS = [1] if PLATFORM in ("colab", "local") else [2, 3]
AUTO_DISCONNECT = True          # Colab: release the GPU runtime when the work is done (results are on Drive)
IDLE_DISCONNECT_MIN = 20        # Colab: release the GPU runtime if it sits idle this long (crash / stuck cell)

DRIVE_DIR = "/content/drive/MyDrive/xguard"      # Colab: pack + all run outputs live here (survive disconnects)


def mount_drive():
    """Mount Google Drive BEFORE anything is written under /content/drive (Colab refuses to mount over files)."""
    if os.path.ismount("/content/drive"):
        return                                     # already mounted (cell re-run)
    if os.path.isdir("/content/drive") and os.listdir("/content/drive"):
        # local leftovers from a run that wrote here before mounting: move them away (they are NOT on Drive)
        bak = f"/content/drive_local_leftover_{int(time.time())}"
        shutil.move("/content/drive", bak)
        print(f"moved local leftovers {bak} (not your Drive files) so Drive can be mounted")
    from google.colab import drive
    drive.mount("/content/drive")


if PLATFORM == "colab":
    mount_drive()


def _find_kaggle_pack():
    for pat in ["/kaggle/input/*/pack_v2", "/kaggle/input/*/*/pack_v2", "/kaggle/input/*/*/*/pack_v2", "/kaggle/input/*/*/*/*/pack_v2"]:
        hits = [h for h in glob.glob(pat) if os.path.exists(os.path.join(h, "meta.pkl"))]
        if hits:
            return Path(hits[0])
    return Path("/kaggle/working/pack_v2")


if PLATFORM == "colab":
    INPUT_ROOT = "/content/raw"                   # raw Kaggle downloads (prepare mode only, local SSD)
    PACK_LOCAL = Path("/content/pack")            # pack copied from Drive to local SSD for fast loading
    PACK_DRIVE = Path(DRIVE_DIR) / "pack_v2"
    WORK = Path(DRIVE_DIR) / "work"
    CACHE = Path("/content/cache")
    PACK_SCOPE = "colab"                          # SIXray complete + val/test of the other datasets (tier 1 needs no more)
elif PLATFORM == "kaggle":
    INPUT_ROOT = "/kaggle/input"
    PACK_LOCAL = Path("/tmp/xguard_pack")         # /tmp: not counted in the 20 GB output limit
    PACK_DRIVE = Path("/kaggle/working/pack_v2") if MODE == "prepare" else _find_kaggle_pack()   # auto: found pack or working
    WORK = Path("/kaggle/working/xguard")
    CACHE = Path("/tmp/xguard_cache")
    PACK_SCOPE = "all"                            # tiers 2-3 also train on PIDray / HiXray / OPIXray
else:
    INPUT_ROOT = os.environ.get("XGUARD_INPUT", "./input")
    PACK_LOCAL = Path(os.environ.get("XGUARD_PACK", "./pack"))
    PACK_DRIVE = Path("./pack_drive")
    WORK = Path(os.environ.get("XGUARD_WORK", "./xguard_work"))
    CACHE = WORK / "cache"
    PACK_SCOPE = os.environ.get("XGUARD_PACK_SCOPE", "all")
if os.environ.get("XGUARD_PACK_DRIVE"):
    PACK_DRIVE = Path(os.environ["XGUARD_PACK_DRIVE"])
if os.environ.get("XGUARD_INPUT"):
    INPUT_ROOT = os.environ["XGUARD_INPUT"]
if os.environ.get("XGUARD_WORK"):
    WORK = Path(os.environ["XGUARD_WORK"])
if os.environ.get("XGUARD_PACK"):
    PACK_LOCAL = Path(os.environ["XGUARD_PACK"])
# Report mode: run folders of the OTHER platform. Colab: unzip the Kaggle "xguard_results_full.zip" into
# MyDrive/xguard/kaggle_results/ (done automatically if the zip is placed there). Kaggle: attach the Colab results.
EXTRA_RESULT_DIRS = [Path(DRIVE_DIR) / "kaggle_results"] if PLATFORM == "colab" else \
    [Path(p) for p in glob.glob("/kaggle/input/*/colab_results")] if PLATFORM == "kaggle" else \
    [Path(p) for p in os.environ.get("XGUARD_EXTRA_RESULTS", "").split(",") if p]
CACHE = Path(os.environ.get("XGUARD_CACHE", str(CACHE)))


def _has_gpu():
    try:
        return subprocess.run(["nvidia-smi", "-L"], capture_output=True, timeout=20).returncode == 0
    except Exception:
        return False


HAS_GPU = _has_gpu()
if MODE == "auto":
    _has_pack = (PACK_DRIVE / "meta.pkl").exists()
    _test_ok = (WORK / "logs" / "test_passed.json").exists()
    if not _has_pack:
        if HAS_GPU:
            _msg = ("No data pack yet and this runtime has a GPU. Step 1 (prepare) needs NO GPU: "
                    "Runtime -> Change runtime type -> CPU, then Run all again. The GPU is being released now.")
            print("⛔ " + _msg)
            if PLATFORM == "colab":
                from google.colab import runtime as _rt
                _rt.unassign()
            raise RuntimeError(_msg)
        MODE = "prepare"
    elif HAS_GPU:
        MODE = "full" if _test_ok else "test"
    else:
        MODE = "report"
    print(f"AUTO -> MODE = '{MODE}'  (pack: {'yes' if _has_pack else 'no'}, GPU: {'yes' if HAS_GPU else 'no'}, "
          f"test passed: {'yes' if _test_ok else 'no'})")
if PLATFORM == "kaggle" and MODE == "prepare":
    PACK_DRIVE = Path("/kaggle/working/pack_v2")
if MODE in ("test", "full") and not HAS_GPU and not SMOKE:
    raise RuntimeError(f"MODE='{MODE}' needs a GPU. Colab: Runtime -> Change runtime type -> A100 GPU. Kaggle: Accelerator -> GPU.")

# ---- datasets (Kaggle slugs used by the Colab "prepare" step) ---------------------
#   key -> (kaggle slug, single file to download or None for the whole dataset)
KAGGLE_DATASETS = {
    "sixray":    ("khanhbtq99/sixray", None),                         # Roboflow export of SIXray positives (boxes)
    "sixray-p1": ("witsarutsarai/sixray-p1", "JPEGImage.tar.gz00"),   # ONLY the first part: clean bags (negatives)
    "pidray":    ("banan41k/xray-contraband-detection", None),       # PIDray (COCO json, official easy/hard/hidden)
    "hixray":    ("newashish/hixray", None),
    "opixray":   ("alirzaei/opixray-yolo", None),
}
OPIXRAY_OL_DIR = str(Path(DRIVE_DIR) / "opixray_ol")   # put test_knife{,-1,-2,-3}.txt here (optional, enables P6)
USE_SIXRAY_NEGATIVES = True     # download 1 part of SIXRAY_P1 (~4 GB) to get clean bags for false-alarm metrics

# Kaggle-platform overrides (ignored on Colab): absolute dataset roots, None = auto-discovery by name/content
DATASET_ROOTS = {"sixray": None, "opixray": None, "hixray": None, "pidray": None, "clcxray": None}
FORCE_FORMAT = {"sixray": None, "opixray": None, "hixray": None, "pidray": None, "clcxray": None}
RESUME_FROM_INPUT = True
EXTRACT_ARCHIVES = False
SIXRAY_TAR_EXTRACT = "auto"
MERGE_VAL_INTO_TEST = {"opixray"}   # alirzaei/opixray-yolo cut the OFFICIAL test set (1,776) into val 888 + test 888

# ---- training / evaluation settings -----------------------------------------------
C = SimpleNamespace(
    IMG=800, EPOCHS_MAIN=12, EPOCHS_ABL=8, BATCH=8, ACCUM=1, LR=0.01, WARMUP_ITERS=500, WEIGHT_DECAY=1e-4,
    MAX_TRAIN_PER_DS=12000, MAX_VAL=600, MAX_EVAL_MAIN=None, MAX_EVAL_ABL=3000, N_NEG_TRAIN=500, N_NEG_EVAL=500,
    SEEDS_MAIN=[0, 1, 2], SEEDS_ABL=[0], PILOT_GATE=True, PILOT_FORCE=False, DEADLINE=True, PSEUDO_TOPK=100, PSEUDO_PER_IMG=8, VOS_QUEUE=1000, VOS_MIN=100, VOS_SAMPLES=1000, MIN_DRIVE_GB=2.0, WORKERS=min(12, os.cpu_count() or 2),
    INIT="coco", TRAINABLE_LAYERS=3, EMB_DIM=256, AMP=True, GRAD_CLIP=10.0,
    SCORE_THR=0.01, DETS_PER_IMG=100, UNK_DETS_PER_IMG=50, BOOTSTRAP=1000,
    USE_SEMANTIC_ALIASES=True, DUP_HAMMING=4, DUP_MAD=3.0, AUDIT_MAX_PER_DS=None, KEEP_FULL_CKPT=False,
    LOSS_W=dict(sira=0.5, occ=0.5, rel=0.2, proto=0.2, unknown=0.5, ent=0.1, cons=0.5, risk=0.5, rec=0.1, threat=0.1),
    VIEW2_DET=True,          # detection loss on view 2 only if it is NOT occluded (occluded boxes are noisy labels)
    OCC_P=0.5, OCC_FRAC=(0.15, 0.5),
    CLIP_HF_NAME="openai/clip-vit-base-patch32", REQUIRE_TEXT=None,   # None -> required in "full" mode
    MAX_NAN_FRAC=0.02, VAL_EVERY=1, PACK_LONG=800, PACK_QUALITY=95,
    REFERRAL_BUDGETS=(0.05, 0.10), OCC_SWEEP=(0.0, 0.1, 0.3, 0.5, 0.7), LOCO_MIN_OBJ=30,
    EXTERNAL_RESULTS=None, SEED=0, MAX_SESSION_HOURS=23.5 if PLATFORM == "colab" else 11.4,
    TEST_ITERS=None, TEST_EVAL=None, TEST_VAL=None,
    CU_PER_HOUR={"A100": 11.8, "L4": 4.8, "T4": 1.8, "V100": 4.9},   # Colab compute units/hour (verify on your Resources page)
)
if PLATFORM == "kaggle":      # T4 / P100 have 16 GB: same EFFECTIVE batch (8) as Colab via accumulation -> comparable runs
    C.BATCH, C.ACCUM = 4, 2
if MODE == "test":
    C.TEST_ITERS, C.TEST_EVAL, C.TEST_VAL, C.BOOTSTRAP = 30, 120, 60, 100
    C.SEEDS_MAIN, C.SEEDS_ABL = [0], [0]
# DEADLINE plan (paper due): most valuable runs first; anything unfinished simply stays out of the tables.
#   occlusion fair control -> unknown-supervision baselines -> seed 1 -> RT-DETR -> zero-shot -> seed 2
C.DEADLINE_RUNS = ["six_base_s0", "six_xguard_s0", "six_base-occaug_s0",
                   "unkF-pseudo_s0", "unkF-vos_s0", "unkF-gaussian_s0",          # published unknown-supervision recipes, same protocol
                   "six_base_s1", "six_xguard_s1", "six_base-occaug_s1",
                   "six_rtdetr-l_s0",                                           # DETR-family closed-set detector
                   "six_gdino_s0", "six_owlvit_s0",                             # zero-shot open-vocabulary transformers
                   "six_base_s2", "six_xguard_s2", "six_base-occaug_s2"]
if C.DEADLINE:
    C.PILOT_GATE = False          # the deadline plan is fixed in advance: no early stop after seed 0
if C.REQUIRE_TEXT is None:
    C.REQUIRE_TEXT = MODE == "full"
if SMOKE:   # tiny CPU self-test
    C.__dict__.update(IMG=160, BATCH=2, MAX_TRAIN_PER_DS=24, MAX_VAL=8, N_NEG_TRAIN=2, N_NEG_EVAL=4, WORKERS=0,
                      INIT="none", WARMUP_ITERS=2, BOOTSTRAP=30, TEST_ITERS=3, TEST_EVAL=10, TEST_VAL=8, PACK_LONG=200,
                      LOCO_MIN_OBJ=1, MAX_EVAL_ABL=10, REQUIRE_TEXT=False)

for sub in ["runs", "tables", "figures", "audit", "logs"]:
    (WORK / sub).mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)
os.environ["TORCH_HOME"] = str((PACK_LOCAL if MODE != "prepare" else PACK_DRIVE) / "torch")   # weights travel inside the pack
os.environ.setdefault("HF_HOME", str(CACHE / "hf"))
jdump_cfg = {k: v for k, v in vars(C).items()}
json.dump(dict(MODE=MODE, TIERS=TIERS, PLATFORM=PLATFORM, CODE_VERSION=CODE_VERSION, **jdump_cfg),
          open(WORK / f"config_{MODE}.json", "w"), indent=1, default=str)
print(f"MODE={MODE}  TIERS={TIERS}  PLATFORM={PLATFORM}  WORK={WORK}  PACK={PACK_LOCAL}")


## 1 · Environment
Drive mount (Colab), GPU check (stops if there is no GPU), bf16/TF32 on A100, helpers.

In [ ]:
# ==============================================================================
# 1. ENVIRONMENT, IMPORTS, GPU SETTINGS, HELPERS
# ==============================================================================
if PLATFORM == "colab":
    from google.colab import drive      # already mounted in the config cell (needed here for flush_and_unmount)

import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader, Sampler
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, Markdown

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu"
_CPU_BF16 = DEVICE.type != "cuda" and os.environ.get("XGUARD_CPU_BF16") == "1"   # developer check only: bf16 autocast on CPU
if DEVICE.type != "cuda" and not _CPU_BF16:
    C.AMP = False
# A100/L4/H100 -> bfloat16 (no loss scaling needed); T4/P100/V100 -> float16 + GradScaler
AMP_DTYPE = torch.bfloat16 if (_CPU_BF16 or (DEVICE.type == "cuda" and torch.cuda.is_bf16_supported())) else torch.float16
if _CPU_BF16:   # emulate CUDA: these ops have no bf16/fp16 kernel on the GPU, so make them fail on CPU too
    def _strict(fn):
        def g(x, *a, **k):
            if torch.is_tensor(x) and x.dtype in (torch.bfloat16, torch.float16):
                raise RuntimeError(f"{fn.__name__}: Low precision dtypes not supported. Got {x.dtype}")
            return fn(x, *a, **k)
        return g
    torch.linalg.inv, torch.quantile = _strict(torch.linalg.inv), _strict(torch.quantile)
USE_SCALER = C.AMP and AMP_DTYPE == torch.float16
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
print("python", sys.version.split()[0], "| torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| device", DEVICE, GPU_NAME, "| amp", AMP_DTYPE if C.AMP else "off")
if MODE in ("test", "full") and DEVICE.type != "cuda" and not SMOKE:
    raise RuntimeError("No GPU! Colab: Runtime -> Change runtime type -> A100 GPU.  Kaggle: Settings -> Accelerator -> GPU. "
                       "(Your previous run trained on CPU: 1 epoch took 10.5 hours.)")


def release_runtime(reason):
    """Colab: flush Drive and give the GPU back so no compute units are spent on an idle runtime."""
    if PLATFORM != "colab" or not AUTO_DISCONNECT:
        return
    print(f"⏏  releasing the Colab runtime: {reason} (all results are on Drive)")
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    from google.colab import runtime
    runtime.unassign()


if PLATFORM == "colab" and MODE in ("test", "full") and DEVICE.type == "cuda" and AUTO_DISCONNECT:
    import threading

    def _gpu_watchdog():
        idle = 0
        while True:
            time.sleep(60)
            try:
                u = int(subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu", "--format=csv,noheader,nounits"],
                                       capture_output=True, text=True, timeout=20).stdout.strip().split("\n")[0])
            except Exception:
                continue
            idle = idle + 1 if u < 3 else 0
            if idle >= IDLE_DISCONNECT_MIN:
                release_runtime(f"GPU idle for {idle} min (finished, crashed or stuck)")
                return
    threading.Thread(target=_gpu_watchdog, daemon=True).start()
    print(f"GPU idle watchdog on: the runtime is released after {IDLE_DISCONNECT_MIN} idle minutes")


def has_internet(timeout=3):
    import socket
    try:
        socket.create_connection(("huggingface.co", 443), timeout=timeout).close()
        return True
    except Exception:
        return False


INTERNET = os.environ.get("XGUARD_INTERNET") == "1" or (False if SMOKE else has_internet())
print("Internet:", INTERNET)

try:
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    HAVE_COCO = True
except Exception:
    HAVE_COCO = False
    if INTERNET:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycocotools"], check=False)
        try:
            from pycocotools.coco import COCO
            from pycocotools.cocoeval import COCOeval
            HAVE_COCO = True
        except Exception:
            pass
print("pycocotools:", HAVE_COCO)


def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


def rng_state():
    return dict(py=random.getstate(), np=np.random.get_state(), torch=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None)


def set_rng_state(s):
    random.setstate(s["py"]); np.random.set_state(s["np"]); torch.set_rng_state(s["torch"])
    if s.get("cuda") is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(s["cuda"])


def elapsed_h():
    return (time.time() - T_SESSION_START) / 3600.0


def time_left_h():
    return C.MAX_SESSION_HOURS - elapsed_h()


def stable_hash(s, mod=10 ** 9):
    return int(hashlib.md5(str(s).encode()).hexdigest()[:12], 16) % mod


def sha(obj, n=16):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:n]


def atomic_write(path, writer):
    """write to <path>.tmp, then rename: a full Drive / disconnect never leaves a half-written file behind."""
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    try:
        writer(tmp); os.replace(tmp, path)
    except BaseException:
        try:
            tmp.unlink()
        except Exception:
            pass
        raise


def drive_free_gb():
    """free space of the Drive account (Colab), None where unknown."""
    try:
        if PLATFORM == "colab" and Path("/content/drive/MyDrive").exists():
            u = shutil.disk_usage("/content/drive/MyDrive")
            if 1e9 < u.total < 1e14 and u.free <= u.total:        # only trust a plausible Drive quota
                return u.free / 1e9
    except Exception:
        pass
    return None


def jdump(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = str(path) + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=1, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))
    os.replace(tmp, path)


def jload(path, default=None):
    try:
        return json.load(open(path))
    except Exception:
        return default


def environment_info():
    info = dict(python=sys.version.split()[0], torch=torch.__version__, torchvision=torchvision.__version__,
                cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version() if DEVICE.type == "cuda" else None,
                gpu=GPU_NAME, amp=str(AMP_DTYPE) if C.AMP else "off", platform=PLATFORM, code_version=CODE_VERSION)
    try:
        import transformers
        info["transformers"] = transformers.__version__
    except Exception:
        info["transformers"] = None
    try:
        info["git_commit"] = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, timeout=5).stdout.strip() or "n/a"
    except Exception:
        info["git_commit"] = "n/a"
    return info


ENV_INFO = environment_info()
jdump(ENV_INFO, WORK / "logs" / "environment.json")
seed_everything(C.SEED)


## 2 · Data staging
`prepare`: Kaggle API → local SSD (parallel downloads). `test`/`full`: copy the pack from Drive to local SSD.

In [ ]:
# ==============================================================================
# 2. DATA STAGING
#   Colab "prepare" (CPU runtime): Kaggle API -> /content/raw (local SSD) -> discovery / verification / audit
#                                  -> compact pack on Google Drive (images resized to PACK_LONG, JPEG q95,
#                                  boxes kept in ORIGINAL coordinates, so metrics are unchanged).
#   Colab "test"/"full" (GPU)    : copy + untar the pack from Drive to local SSD (a few minutes, no Kaggle download).
#   Kaggle "prepare" (CPU session): datasets attached as inputs -> same pipeline -> pack in /kaggle/working/pack_v2.
#   Kaggle "test"/"full" (GPU)   : attach the prepare notebook's OUTPUT as input; the pack is found automatically.
# ==============================================================================
PACK_READY = False
PACK_EVENTS = {}


def disk_free_gb(p):
    p = Path(p)
    while not p.exists():
        p = p.parent
    return shutil.disk_usage(p).free / 1e9


def _extract(k, shard):
    marker = PACK_LOCAL / f".done_{k}"
    if not marker.exists():
        subprocess.run(["tar", "-xf", str(shard), "-C", str(PACK_LOCAL)], check=True)
        marker.write_text("ok")
    PACK_EVENTS[k].set()


def restore_pack(first=("sixray",)):
    """Copy the pack to local SSD. The source-training dataset (SIXray) is extracted FIRST so training can start
    immediately; the other datasets are extracted by a background thread while the GPU is already busy."""
    if not (PACK_DRIVE / "meta.pkl").exists():
        return (PACK_LOCAL / "meta.pkl").exists()
    import threading
    PACK_LOCAL.mkdir(parents=True, exist_ok=True)
    shutil.copy(PACK_DRIVE / "meta.pkl", PACK_LOCAL / "meta.pkl")
    for extra in ["clip_text.pt", "torch"]:
        src = PACK_DRIVE / extra
        if src.is_dir():
            shutil.copytree(src, PACK_LOCAL / extra, dirs_exist_ok=True)
        elif src.exists():
            shutil.copy(src, PACK_LOCAL / extra)
    shards = {f.stem.replace("images_", ""): f for f in sorted(PACK_DRIVE.glob("images_*.tar"))}
    for k in shards:
        PACK_EVENTS[k] = threading.Event()
    t0 = time.time()
    for k in [k for k in first if k in shards]:
        _extract(k, shards[k])
    print(f"pack {PACK_DRIVE}: {', '.join(first)} ready in {(time.time() - t0) / 60:.1f} min; "
          f"{len(shards) - len(first)} more shards extracting in the background")

    def bg():
        for k, f in shards.items():
            if k not in first:
                _extract(k, f)
    threading.Thread(target=bg, daemon=True).start()
    return True


def wait_pack(ds):
    ev = PACK_EVENTS.get(ds)
    if ev is not None and not ev.is_set():
        t0 = time.time(); print(f"   waiting for the {ds} images to finish extracting…", end=" ")
        ev.wait(); print(f"{time.time() - t0:.0f}s")


def wait_pack_all():
    for k in list(PACK_EVENTS):
        wait_pack(k)


def resume_previous_outputs():
    """Kaggle: copy finished runs from previous versions of this notebook attached as input."""
    if PLATFORM != "kaggle":
        return
    n = 0
    for pat in ["/kaggle/input/*/xguard", "/kaggle/input/*/*/xguard", "/kaggle/input/*/*/*/xguard"]:
        for cand in glob.glob(pat):
            for rd_root in Path(cand).glob("runs*"):
                for rd in rd_root.iterdir():
                    dst = WORK / rd_root.name / rd.name
                    if rd.is_dir() and not dst.exists():
                        shutil.copytree(rd, dst); n += 1
    if n:
        print(f"[resume] copied {n} finished run folders from the previous notebook output")


def _set_kaggle_token(tok):
    """New-style Kaggle API token (KGAT_...): env var + ~/.kaggle/access_token (what the kaggle CLI reads)."""
    tok = tok.strip()
    os.environ["KAGGLE_API_TOKEN"] = tok
    d = Path.home() / ".kaggle"; d.mkdir(parents=True, exist_ok=True)
    (d / "access_token").write_text(tok); os.chmod(d / "access_token", 0o600)


def kaggle_auth():
    """Accepts BOTH Kaggle credential styles, never from the code itself:
       new : Colab secret KAGGLE_API_TOKEN (value starts with KGAT_)
       old : Colab secrets KAGGLE_USERNAME + KAGGLE_KEY, or MyDrive/xguard/kaggle.json"""
    if os.environ.get("KAGGLE_API_TOKEN") or (os.environ.get("KAGGLE_KEY") and os.environ.get("KAGGLE_USERNAME")):
        return "environment"
    secrets = {}
    try:
        from google.colab import userdata
        for name in ["KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"]:
            try:
                v = userdata.get(name)
                if v:
                    secrets[name] = v
            except Exception:
                pass
    except Exception:
        pass
    tok = secrets.get("KAGGLE_API_TOKEN") or (secrets.get("KAGGLE_KEY") if str(secrets.get("KAGGLE_KEY", "")).startswith("KGAT_") else None)
    if tok:
        _set_kaggle_token(tok); return "Colab secret (API token)"
    if secrets.get("KAGGLE_USERNAME") and secrets.get("KAGGLE_KEY"):
        os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = secrets["KAGGLE_USERNAME"], secrets["KAGGLE_KEY"]
        return "Colab secrets (username + key)"
    for p in [Path(DRIVE_DIR) / "kaggle.json", Path.home() / ".kaggle" / "kaggle.json"]:
        if p.exists():
            d = json.load(open(p)); os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = d["username"], d["key"]
            return str(p)
    if (Path.home() / ".kaggle" / "access_token").exists():
        return "~/.kaggle/access_token"
    raise RuntimeError("Kaggle credentials not found. Colab: key icon (Secrets) -> add a secret named KAGGLE_API_TOKEN "
                       "whose value is your token (KGAT_...) from kaggle.com -> Settings -> API -> Create New Token, "
                       "switch 'Notebook access' ON, then Run all again.")


def kaggle_download(key, slug, file=None):
    out = Path(INPUT_ROOT) / key
    if (out / ".done").exists():
        return f"{key}: cached"
    out.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    base = ["kaggle", "datasets", "download"]
    extra = ["-p", str(out)] + (["-f", file] if file else [])
    r = subprocess.run(base + [slug] + extra, capture_output=True, text=True)
    if r.returncode != 0:     # older kaggle CLI needs -d
        r = subprocess.run(base + ["-d", slug] + extra, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"kaggle download failed for {slug}: {r.stderr[-500:]}")
    for z in list(out.glob("*.zip")):
        subprocess.run(["unzip", "-q", "-o", str(z), "-d", str(out)], check=True)
        z.unlink()
    (out / ".done").write_text(slug)
    size = sum(f.stat().st_size for f in out.rglob("*") if f.is_file()) / 1e9
    return f"{key}: {slug}{' / ' + file if file else ''} -> {size:.1f} GB in {(time.time() - t0) / 60:.1f} min"


if MODE in ("test", "full", "report"):
    resume_previous_outputs()
    PACK_READY = restore_pack()
    if not PACK_READY and PLATFORM == "colab":
        raise RuntimeError(f"No data pack at {PACK_DRIVE}. First run this notebook once with MODE='prepare' on a CPU "
                           "runtime (Runtime -> Change runtime type -> CPU); it downloads the Kaggle datasets and builds the pack.")
elif MODE == "prepare":
    if PLATFORM == "colab":
        if (PACK_DRIVE / "meta.pkl").exists():
            print(f"⚠️  a pack already exists at {PACK_DRIVE}. Delete that folder to rebuild it; otherwise switch MODE to 'test'.")
        print(f"free disk: {disk_free_gb('/content'):.0f} GB local, Drive folder {DRIVE_DIR}")
        print("Kaggle credentials:", kaggle_auth())
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kaggle"], check=False)   # new tokens need a recent CLI
        jobs = {k: v for k, v in KAGGLE_DATASETS.items() if k != "sixray-p1" or USE_SIXRAY_NEGATIVES}
        from concurrent.futures import ThreadPoolExecutor as _TPE
        par = len(jobs) if disk_free_gb("/content") > 80 else 1   # zip + extracted copy need ~2x space while unzipping
        print(f"downloading {len(jobs)} datasets ({'parallel' if par > 1 else 'sequential: low disk'})")
        with _TPE(par) as ex:                            # network-bound: download datasets in parallel
            for msg in ex.map(lambda kv: kaggle_download(kv[0], *kv[1]), jobs.items()):
                print("  ", msg)
        if os.path.isdir(OPIXRAY_OL_DIR) and glob.glob(os.path.join(OPIXRAY_OL_DIR, "test_knife*.txt")):
            shutil.copytree(OPIXRAY_OL_DIR, Path(INPUT_ROOT) / "opixray-ol", dirs_exist_ok=True)
            print("   OPIXray occlusion lists copied from", OPIXRAY_OL_DIR)
        else:
            print(f"   (optional) OPIXray OL1-3 lists not found in {OPIXRAY_OL_DIR} -> P6 occlusion-level table will be empty")
if MODE == "report":     # merge the other platform's results (zip files dropped into EXTRA_RESULT_DIRS are unpacked)
    for d in EXTRA_RESULT_DIRS:
        for z in Path(d).glob("*.zip") if Path(d).exists() else []:
            out = Path(d) / z.stem
            if not out.exists():
                shutil.unpack_archive(str(z), str(out)); print("unpacked", z)
NEED_DISCOVERY = not PACK_READY
print(f"PACK_READY={PACK_READY}  NEED_DISCOVERY={NEED_DISCOVERY}")


## 3 · Dataset discovery (prepare only)
Folder tree, annotation format, splits and occlusion lists of every downloaded dataset.

In [ ]:
# ==============================================================================
# 3. DATASET DISCOVERY (raw inputs only; skipped when a pack exists) + FORMAT-AGNOSTIC PARSERS
#    Supported: COCO-JSON (PIDray, CLCXray), Pascal-VOC XML (SIXray), per-line TXT
#    "img cls x1 y1 x2 y2" (OPIXray, HiXray), YOLO txt (+data.yaml/classes.txt), CSV.
# ==============================================================================
import xml.etree.ElementTree as ET
import tarfile
from concurrent.futures import ThreadPoolExecutor

DS_KEYS = ["sixray", "opixray", "hixray", "pidray", "clcxray"]
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
ANN_EXT = {".xml", ".json", ".txt", ".csv", ".yaml", ".yml", ".names"}
NEG_DATASETS = {"sixray"}   # images without boxes in these datasets are used as clean (negative) bags
ARCHIVE_EXT = (".zip", ".tar", ".gz", ".tgz", ".rar", ".7z")

# Class-name fingerprints: identify a Kaggle mirror by its CONTENT when its folder name does not say which
# dataset it is (e.g. "xray-contraband-detection" = PIDray). >= 2 hits of the signature classes are required.
FINGERPRINT = OrderedDict(
    pidray={"handcuffs", "baton", "sprayer", "bullet", "powerbank"},
    hixray={"portable_charger_1", "portable_charger_2", "mobile_phone", "laptop", "tablet", "cosmetic", "water", "nonmetallic_lighter"},
    opixray={"folding_knife", "straight_knife", "utility_knife", "multitool_knife"},
    clcxray={"swiss_army_knife", "cans", "carton_drinks", "glass_bottle", "plastic_bottle", "vacuum_cup", "spray_cans", "tin", "blade", "dagger"},
    sixray={"gun", "knife", "wrench", "pliers", "scissors", "hammer"},
)
# Used ONLY for YOLO mirrors that ship no data.yaml / classes.txt (official class order; verify in the printout!)
YOLO_DEFAULT_NAMES = {
    "sixray": ["Gun", "Knife", "Wrench", "Pliers", "Scissors", "Hammer"],
    "opixray": ["Folding_Knife", "Straight_Knife", "Scissor", "Utility_Knife", "Multi-tool_Knife"],
    "hixray": ["Portable_Charger_1", "Portable_Charger_2", "Mobile_Phone", "Laptop", "Tablet", "Cosmetic", "Water", "Nonmetallic_Lighter"],
    "pidray": ["Baton", "Pliers", "Hammer", "Powerbank", "Scissors", "Wrench", "Gun", "Bullet", "Sprayer", "HandCuffs", "Knife", "Lighter"],
}


# ---------- resume: bring finished runs from a previous notebook version ----------
def resume_from_input():
    if PLATFORM != "kaggle" or not RESUME_FROM_INPUT or not os.path.isdir(INPUT_ROOT):
        return
    n = 0
    for cand in glob.glob(os.path.join(INPUT_ROOT, "**", "xguard", "runs"), recursive=True)[:5]:
        if str(WORK) in cand:
            continue
        for rd in Path(cand).iterdir():
            dst = WORK / "runs" / rd.name
            if rd.is_dir() and not dst.exists():
                shutil.copytree(rd, dst); n += 1
        for extra in ["audit", "cache", "extracted"]:
            src = Path(cand).parent / extra
            if src.exists():
                shutil.copytree(src, WORK / extra, dirs_exist_ok=True)
    if n:
        print(f"[resume] copied {n} run folders from previous output")




# ---------- archive extraction (optional; Kaggle normally auto-extracts .zip uploads) ----------
EXTRACT_DIR = Path(os.environ.get("XGUARD_EXTRACT_DIR", "/tmp/xguard_extract"))   # /tmp is not limited to 20 GB like /kaggle/working


def list_archives(root, limit=20, max_depth=2):
    """Archives in the top levels of an input (cheap: does not walk million-file trees)."""
    out = []
    for dirpath, dirnames, files in os.walk(root):
        if len(Path(dirpath).relative_to(root).parts) >= max_depth:
            dirnames[:] = []
        out += [os.path.join(dirpath, f) for f in files if f.lower().endswith(ARCHIVE_EXT)]
        if len(out) >= limit:
            break
    return out


def maybe_extract(root):
    arcs = list_archives(root)
    if not arcs:
        return root
    if not EXTRACT_ARCHIVES:
        print(f"   ⚠️  {root} contains archive(s) {[os.path.basename(a) for a in arcs[:3]]} that Kaggle did not extract. "
              "Use an already-extracted mirror, or set EXTRACT_ARCHIVES=True (extracts to /tmp).")
        return root
    out = EXTRACT_DIR / Path(root).name
    if not (out / ".done").exists():
        out.mkdir(parents=True, exist_ok=True)
        for a in arcs:
            print("extracting", a)
            if a.lower().endswith((".rar", ".7z")):
                subprocess.run(["7z", "x", "-y", f"-o{out}", a], check=False)
            else:
                shutil.unpack_archive(a, str(out))
        (out / ".done").write_text("ok")
    return str(out)


# ---------- multi-part tar streams (e.g. SIXRAY_P1/P2/P3 = JPEGImage.tar.gz00 ... gz15) ----------
MULTIPART = re.compile(r"^(?P<base>.+?\.(?:tar\.gz|tgz|tar|tar\.bz2|tar\.xz))[._-]?(?:part)?[._-]?(?P<num>\d{1,3})$", re.I)


def multipart_sets(roots):
    parts = defaultdict(list)
    for r in roots:
        for dirpath, dirnames, files in os.walk(r):
            if len(Path(dirpath).relative_to(r).parts) >= 2:
                dirnames[:] = []
            for f in files:
                m = MULTIPART.match(f)
                if m:
                    parts[m.group("base")].append((int(m.group("num")), os.path.join(dirpath, f)))
    return {b: [p for _, p in sorted(v)] for b, v in parts.items()}


class ConcatReader(io.RawIOBase):
    """Reads several files as one continuous byte stream (split archives), without copying them."""
    def __init__(self, paths):
        self.paths, self.i, self.f = paths, 0, open(paths[0], "rb")

    def readable(self):
        return True

    def readinto(self, b):
        while True:
            n = self.f.readinto(b)
            if n:
                return n
            self.f.close(); self.i += 1
            if self.i >= len(self.paths):
                return 0
            self.f = open(self.paths[self.i], "rb")


def stream_extract(paths, out_dir, want_stems, n_neg, neg_prefix="n"):
    """Stream through a (split) tar archive once and write only the wanted files:
    images referenced by annotations (want_stems) + the first n_neg clean bags (name starts with neg_prefix)."""
    out_dir.mkdir(parents=True, exist_ok=True)
    want = set(want_stems); got_pos = got_neg = seen = 0; t0 = time.time()
    try:
      with tarfile.open(fileobj=io.BufferedReader(ConcatReader(paths), 1 << 22), mode="r|*") as tf:
        for m in tf:
            seen += 1
            if seen % 100000 == 0:
                print(f"   … {seen} members scanned, {got_pos} annotated + {got_neg} clean extracted ({(time.time() - t0) / 60:.1f} min)")
            if not m.isfile():
                continue
            b = os.path.basename(m.name); st, ext = os.path.splitext(b.lower())
            if ext not in IMG_EXT:
                continue
            is_pos = st in want
            is_neg = (not is_pos) and got_neg < n_neg and st.startswith(neg_prefix)
            if is_pos or is_neg:
                with open(out_dir / b, "wb") as f:
                    f.write(tf.extractfile(m).read())
                if is_pos:
                    want.discard(st); got_pos += 1
                else:
                    got_neg += 1
            if not want and got_neg >= n_neg:
                break
    except (tarfile.ReadError, EOFError, OSError) as e:   # only some parts downloaded -> stream ends early
        print(f"   archive stream ended early ({type(e).__name__}): partial download, keeping what was extracted")
    print(f"   extracted {got_pos} annotated + {got_neg} clean images in {(time.time() - t0) / 60:.1f} min; "
          f"{len(want)} annotated images not found in the archive")


def prepare_multipart(key, roots):
    """If a dataset is shipped as split tar parts, extract what is needed into /kaggle/working (persists across sessions)."""
    sets = multipart_sets(roots)
    if not sets or SIXRAY_TAR_EXTRACT is False:
        return roots
    out = CACHE / "extracted" / f"{key}_from_tar"
    imgs, anns = index_files(roots)
    have = {os.path.splitext(os.path.basename(p))[0].lower() for p in imgs}
    ann_stems = {os.path.splitext(os.path.basename(a))[0].lower() for a in anns if a.lower().endswith((".xml", ".txt"))}
    want = {s for s in ann_stems - have if key != "sixray" or re.match(r"^p\d+", s)}
    n_neg = (C.N_NEG_TRAIN + C.N_NEG_EVAL) if key in NEG_DATASETS else 0
    marker = out / f".done_{stable_hash(','.join(sorted(want)) + str(n_neg))}"   # new annotations -> extract again
    if marker.exists():
        return roots + [str(out)]
    if out.exists():
        want -= {os.path.splitext(f)[0].lower() for f in os.listdir(out)}
        cache = CACHE / f"index_{stable_hash(str(out))}.pkl"
        if cache.exists():
            cache.unlink()
    print(f"[{key}] split archive {[os.path.basename(p) for p in list(sets.values())[0][:3]]}… ({sum(len(v) for v in sets.values())} parts): "
          f"streaming {len(want)} annotated images + {n_neg} clean bags into {out}")
    if not want and key == "sixray":
        print("   ⚠️  no SIXray box annotations attached -> only clean bags are extracted. Attach an annotated SIXray "
              "(e.g. khanhbtq99/sixray, or SIXray-D / official positive XMLs uploaded as a dataset named '*sixray*').")
    n_have_neg = sum(1 for f in os.listdir(out) if f.lower().startswith("n")) if out.exists() else 0
    for base, paths in sets.items():
        stream_extract(paths, out, want, max(0, n_neg - n_have_neg))
    marker.write_text("ok")
    return roots + [str(out)]


# ---------- locate dataset roots (one Kaggle dataset = one root; a dataset may span several roots) ----------
def kaggle_dataset_dirs():
    """/kaggle/input/<slug>  and the newer layout /kaggle/input/datasets/<owner>/<slug>."""
    out = []
    if not os.path.isdir(INPUT_ROOT):
        return out
    for d in sorted(os.scandir(INPUT_ROOT), key=lambda e: e.name):
        if not d.is_dir():
            continue
        if d.name in ("datasets", "competitions"):
            for o in sorted(os.scandir(d.path), key=lambda e: e.name):
                if o.is_dir():
                    out += [x.path for x in sorted(os.scandir(o.path), key=lambda e: e.name) if x.is_dir()] if d.name == "datasets" else [o.path]
        elif d.name not in ("models", "notebooks"):
            out.append(d.path)
    # skip previous XGuard outputs (resume inputs) and weight-only inputs
    return [r for r in out if not glob.glob(os.path.join(r, "**", "xguard", "runs"), recursive=True)[:1]]


def name_key(path):
    n = re.sub(r"[^a-z0-9]", "", Path(path).name.lower())
    hits = [k for k in DS_KEYS if k in n]
    return hits[0] if len(hits) == 1 else None


def find_dataset_roots():
    groups, unnamed = defaultdict(list), []
    for k, v in DATASET_ROOTS.items():
        if v:
            groups[k] = [v] if isinstance(v, str) else list(v)
    manual = {os.path.normpath(p) for v in groups.values() for p in v}
    for r in kaggle_dataset_dirs():
        if os.path.normpath(r) in manual:
            continue
        k = name_key(r)
        if k and not DATASET_ROOTS.get(k):
            groups[k].append(r)
        elif not k:
            unnamed.append(r)
    groups = {k: [maybe_extract(r) for r in v] for k, v in groups.items()}
    return groups, [maybe_extract(r) for r in unnamed]


def print_tree(root, max_depth=3, max_children=8):
    root = Path(root)
    print(f"📁 {root}")

    def rec(p, depth, prefix):
        try:
            entries = sorted(os.scandir(p), key=lambda e: (not e.is_dir(), e.name))
        except Exception:
            return
        dirs = [e for e in entries if e.is_dir()]
        files = [e for e in entries if e.is_file()]
        ext = Counter(Path(f.name).suffix.lower() for f in files)
        if files:
            ex = ", ".join(f"{v}×{k or '(none)'}" for k, v in ext.most_common(5))
            print(f"{prefix}  {len(files)} files [{ex}]  e.g. {files[0].name}")
        if depth >= max_depth:
            if dirs:
                print(f"{prefix}  … {len(dirs)} sub-folders")
            return
        for d in dirs[:max_children]:
            print(f"{prefix}├─ {d.name}/")
            rec(d.path, depth + 1, prefix + "│  ")
        if len(dirs) > max_children:
            print(f"{prefix}└─ … +{len(dirs) - max_children} more folders")
    rec(root, 0, "")


def index_files(roots):
    if isinstance(roots, (list, tuple)):
        imgs, anns = [], []
        for r in roots:
            a, b = index_files(r); imgs += a; anns += b
        return imgs, anns
    root = roots
    cache = CACHE / f"index_{stable_hash(root)}.pkl"
    if cache.exists():
        return pickle.load(open(cache, "rb"))
    imgs, anns = [], []
    for dirpath, _, files in os.walk(root):
        for f in files:
            ext = os.path.splitext(f)[1].lower()
            if ext in IMG_EXT:
                imgs.append(os.path.join(dirpath, f))
            elif ext in ANN_EXT:
                anns.append(os.path.join(dirpath, f))
    out = (imgs, anns)
    if len(imgs) > 20000:
        print(f"   indexed {len(imgs)} images under {root}")
    pickle.dump(out, open(cache, "wb"))
    return out


# ---------- split / subset inference from a path ----------
SPLIT_WORDS = [("train", "train"), ("trainval", "train"), ("valid", "val"), ("validation", "val"), ("val", "val"), ("test", "test"),
               ("easy", "test"), ("hard", "test"), ("hidden", "test")]


def _rel(path, root):
    roots = root if isinstance(root, (list, tuple)) else [root]
    for r in roots:
        if os.path.normpath(path).startswith(os.path.normpath(r) + os.sep):
            return os.path.relpath(path, r)
    return os.path.basename(path)


def infer_split(path, root):
    rel = _rel(path, root).lower()
    toks = re.split(r"[\\/_\-\.\s]+", rel)
    split, subsets = None, []
    for t in toks:
        for w, s in SPLIT_WORDS:
            if t == w or (t.startswith(w) and w in ("train", "test") and len(t) <= len(w) + 6):
                split = s
        if t in ("easy", "hard", "hidden"):
            subsets.append(t)
    return split, subsets


class ImageIndex:
    def __init__(self, paths, root):
        self.root = root
        self.by_name, self.by_stem = defaultdict(list), defaultdict(list)
        self.by_abs = {}
        for p in paths:
            b = os.path.basename(p)
            self.by_name[b.lower()].append(p)
            self.by_stem[os.path.splitext(b)[0].lower()].append(p)
            self.by_abs[os.path.normpath(p).lower()] = p

    def resolve(self, name, hint=None):
        if name is None:
            return None
        name = str(name).strip().replace("\\", "/")
        if os.path.isabs(name) and os.path.normpath(name).lower() in self.by_abs:
            return self.by_abs[os.path.normpath(name).lower()]
        roots = self.root if isinstance(self.root, (list, tuple)) else [self.root]
        bases = ([os.path.dirname(hint), os.path.dirname(os.path.dirname(hint))] if hint else []) + list(roots)
        for base in bases:
            q = os.path.normpath(os.path.join(base, name)).lower()
            if q in self.by_abs:
                return self.by_abs[q]
        b = os.path.basename(name).lower()
        c = self.by_name.get(b) or self.by_stem.get(os.path.splitext(b)[0]) or self.by_stem.get(b)
        if not c:
            return None
        if len(c) == 1 or hint is None:
            return c[0]
        hs, _ = infer_split(hint, self.root)
        hparts = set(re.split(r"[\\/]", _rel(hint, self.root).lower()))

        def score(p):
            ps, _ = infer_split(p, self.root)
            parts = set(re.split(r"[\\/]", _rel(p, self.root).lower()))
            return (ps == hs) * 10 + len(parts & hparts)
        return max(c, key=score)


# ---------- annotation parsers: each returns list of dict(img, path, split, subsets, objs=[(cls,x1,y1,x2,y2)], w, h) ----------
def _safe_float(x):
    try:
        return float(x)
    except Exception:
        return None


def parse_coco(files, idx, root):
    out = []
    for jf in files:
        try:
            d = json.load(open(jf))
        except Exception:
            continue
        if not (isinstance(d, dict) and "images" in d and "annotations" in d):
            continue
        cats = {c["id"]: c["name"] for c in d.get("categories", [])}
        split, subsets = infer_split(jf, root)
        by_img = defaultdict(list)
        for a in d["annotations"]:
            if a.get("iscrowd", 0):
                continue
            x, y, w, h = a["bbox"]
            by_img[a["image_id"]].append((cats.get(a["category_id"], str(a["category_id"])), x, y, x + w, y + h))
        for im in d["images"]:
            p = idx.resolve(im["file_name"], jf)
            if p is None:
                continue
            ps, psub = infer_split(p, root)
            out.append(dict(img=os.path.splitext(os.path.basename(p))[0], path=p, split=split or ps,
                            subsets=sorted(set(subsets + psub)), objs=by_img.get(im["id"], []),
                            w=im.get("width"), h=im.get("height"), src=os.path.basename(jf)))
    return out


def parse_voc(files, idx, root):
    out = []
    for xf in files:
        try:
            r = ET.parse(xf).getroot()
        except Exception:
            continue
        objs = []
        for o in r.iter("object"):
            n = o.findtext("name"); bb = o.find("bndbox")
            if n is None or bb is None:
                continue
            v = [_safe_float(bb.findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax")]
            if None in v:
                continue
            objs.append((n.strip(), *v))
        fn = r.findtext("filename") or os.path.splitext(os.path.basename(xf))[0]
        p = idx.resolve(fn, xf) or idx.resolve(os.path.splitext(os.path.basename(xf))[0], xf)
        if p is None:
            continue
        sz = r.find("size")
        w = _safe_float(sz.findtext("width")) if sz is not None else None
        h = _safe_float(sz.findtext("height")) if sz is not None else None
        s1, sub1 = infer_split(xf, root); s2, sub2 = infer_split(p, root)
        out.append(dict(img=os.path.splitext(os.path.basename(p))[0], path=p, split=s1 or s2,
                        subsets=sorted(set(sub1 + sub2)), objs=objs, w=w or None, h=h or None, src="voc"))
    return out


TXT_LINE5 = re.compile(r"^\s*([A-Za-z][\w\-]*)[\s,;]+(-?[\d.]+)[\s,;]+(-?[\d.]+)[\s,;]+(-?[\d.]+)[\s,;]+(-?[\d.]+)\s*$")
TXT_LINE = re.compile(r"^\s*(\S+?)[\s,;]+([A-Za-z][\w\-\s]*?)[\s,;]+(-?[\d.]+)[\s,;]+(-?[\d.]+)[\s,;]+(-?[\d.]+)[\s,;]+(-?[\d.]+)\s*$")


def parse_txt(files, idx, root):
    per = {}
    for tf in files:
        try:
            lines = open(tf, errors="ignore").read().splitlines()
        except Exception:
            continue
        for ln in lines:
            m = TXT_LINE.match(ln)
            if m:
                name, cls = m.group(1), m.group(2).strip()
                v = [float(m.group(i)) for i in range(3, 7)]
            else:
                m = TXT_LINE5.match(ln)
                if not m:
                    continue
                name, cls = None, m.group(1)
                v = [float(m.group(i)) for i in range(2, 6)]
            p = idx.resolve(name, tf) if name else None
            if p is None:   # annotation file named after the image (one txt per image)
                p = idx.resolve(os.path.splitext(os.path.basename(tf))[0], tf)
            if p is None:
                continue
            if p not in per:
                s1, sub1 = infer_split(tf, root); s2, sub2 = infer_split(p, root)
                per[p] = dict(img=os.path.splitext(os.path.basename(p))[0], path=p, split=s1 or s2,
                              subsets=sorted(set(sub1 + sub2)), objs=[], w=None, h=None, src="txt")
            per[p]["objs"].append((cls, *v))
    # images that have an (empty) per-image annotation file are still annotated negatives/positives
    return list(per.values())


def yolo_class_names(anns):
    for f in [a for a in anns if a.lower().endswith((".yaml", ".yml"))]:
        try:
            import yaml
            d = yaml.safe_load(open(f))
            n = d.get("names") if isinstance(d, dict) else None
            if isinstance(n, dict):
                return [n[k] for k in sorted(n)]
            if isinstance(n, list):
                return n
        except Exception:
            pass
    for f in [a for a in anns if os.path.basename(a).lower() in ("classes.txt", "obj.names") or a.lower().endswith(".names")]:
        return [l.strip() for l in open(f) if l.strip()]
    return None


def parse_yolo(files, idx, root, names=None):
    out = []
    for tf in files:
        p = idx.resolve(os.path.splitext(os.path.basename(tf))[0], tf.replace("labels", "images"))
        if p is None:
            continue
        rows = []
        for ln in open(tf, errors="ignore"):
            t = ln.split()
            if len(t) >= 5 and all(_safe_float(x) is not None for x in t[:5]):
                rows.append([float(x) for x in t[:5]])
        s1, sub1 = infer_split(tf, root); s2, sub2 = infer_split(p, root)
        out.append(dict(img=os.path.splitext(os.path.basename(p))[0], path=p, split=s1 or s2,
                        subsets=sorted(set(sub1 + sub2)), objs=[], w=None, h=None, src="yolo", _yolo=rows,
                        _names=names))
    return out


def parse_csv(files, idx, root):
    out = {}
    for cf in files:
        try:
            df = pd.read_csv(cf)
        except Exception:
            continue
        cols = {c.lower().strip(): c for c in df.columns}
        def pick(*ks):
            for k in ks:
                if k in cols:
                    return cols[k]
        fc, cc = pick("filename", "file_name", "image", "img", "image_name", "path"), pick("class", "label", "category", "name")
        bx = [pick("xmin", "x1", "left"), pick("ymin", "y1", "top"), pick("xmax", "x2", "right"), pick("ymax", "y2", "bottom")]
        if fc is None or cc is None or None in bx:
            continue
        for _, r in df.iterrows():
            p = idx.resolve(str(r[fc]), cf)
            if p is None:
                continue
            if p not in out:
                s1, sub1 = infer_split(cf, root); s2, sub2 = infer_split(p, root)
                out[p] = dict(img=os.path.splitext(os.path.basename(p))[0], path=p, split=s1 or s2,
                              subsets=sorted(set(sub1 + sub2)), objs=[], w=None, h=None, src="csv")
            out[p]["objs"].append((str(r[cc]), *[float(r[b]) for b in bx]))
    return list(out.values())


def classify_ann_files(anns, root):
    """Sort annotation files by format using cheap content sniffing."""
    kinds = defaultdict(list)
    for a in anns:
        ext = os.path.splitext(a)[1].lower()
        if ext == ".json":
            try:
                if os.path.getsize(a) > 2e9:
                    continue
                with open(a) as f:
                    head = f.read(4096)
                if '"images"' in head or '"annotations"' in head or '"categories"' in head or os.path.getsize(a) > 4096:
                    kinds["coco"].append(a)
            except Exception:
                pass
        elif ext == ".xml":
            kinds["voc"].append(a)
        elif ext == ".csv":
            kinds["csv"].append(a)
        elif ext == ".txt":
            try:
                with open(a, errors="ignore") as f:
                    lines = [l.strip() for l in itertools.islice(f, 20) if l.strip()]
            except Exception:
                continue
            if not lines:
                kinds["txt_empty"].append(a); continue
            if any(TXT_LINE.match(l) or TXT_LINE5.match(l) for l in lines):
                kinds["txt"].append(a)
            elif all(len(l.split()) >= 5 and all(_safe_float(x) is not None for x in l.split()[:5]) for l in lines):
                kinds["yolo"].append(a)
            elif all(len(l.split()) == 1 for l in lines):
                kinds["idlist"].append(a)
    # cheap confirmation for voc (must contain <object> or <annotation>)
    kinds["voc"] = [x for x in kinds["voc"] if "<annotation" in open(x, errors="ignore").read(400) or
                    "<object" in open(x, errors="ignore").read()]
    return kinds


def apply_idlists(recs, idlists, root):
    """ImageSets-style id lists and OPIXray test_knife-{1,2,3}.txt.
    - occlusion lists tag OL1/OL2/OL3 (these images belong to the official test set)
    - split lists only fill in a split that folders / annotation files did not already give
      (a mirror's own random split never overrides the official one)
    - renamed mirror files (e.g. test_034894_0001.jpg) are matched through their numeric id (034894)."""
    by_stem = {r["img"].lower(): r for r in recs}
    core = defaultdict(list)
    for r in recs:
        for d in re.findall(r"\d{5,}", r["img"]):
            core[d].append(r)
    by_core = {d: v[0] for d, v in core.items() if len(v) == 1}
    used = []
    for lf in idlists:
        ids = [os.path.splitext(os.path.basename(l.strip().split()[0]))[0].lower()
               for l in open(lf, errors="ignore") if l.strip()]
        rs = [by_stem.get(i) or by_core.get(i) for i in ids]
        rs = [r for r in rs if r is not None]
        if len(rs) < max(5, 0.5 * len(ids)):
            continue
        nm = os.path.basename(lf).lower()
        s, _ = infer_split(lf, root)
        ol = re.search(r"test[\w]*?[-_](\d)\.txt$", nm)
        n_set = 0
        for r in rs:
            if ol:
                r["subsets"] = sorted(set(r["subsets"] + [f"OL{ol.group(1)}"]))
            elif r["split"] is None and (s or "trainval" in nm):
                r["split"] = "train" if "trainval" in nm else s; n_set += 1
        used.append(nm if (ol or n_set) else f"{nm} (ignored: official split kept)")
    return used


def read_sizes(recs):
    todo = [r for r in recs if not r.get("w") or not r.get("h")]
    def f(r):
        try:
            with Image.open(r["path"]) as im:
                r["w"], r["h"] = im.size
        except Exception:
            r["w"] = r["h"] = None
    with ThreadPoolExecutor(16) as ex:
        list(ex.map(f, todo))


def load_dataset(key, roots):
    roots = roots if isinstance(roots, (list, tuple)) else [roots]
    root = roots if len(roots) > 1 else roots[0]
    imgs, anns = index_files(roots)
    idx = ImageIndex(imgs, root)
    kinds = classify_ann_files(anns, root)
    order = ["coco", "voc", "txt", "csv", "yolo"]
    fmt = FORCE_FORMAT.get(key) if key else None
    fmt = fmt or next((k for k in order if kinds.get(k)), None)
    yolo_names, yolo_note = None, None
    if fmt == "yolo":
        yolo_names = yolo_class_names(anns)
        if yolo_names is None and key in YOLO_DEFAULT_NAMES:
            yolo_names = YOLO_DEFAULT_NAMES[key]
            yolo_note = f"no data.yaml/classes.txt found -> ASSUMED official class order {yolo_names}. VERIFY!"
    parsers = dict(coco=parse_coco, voc=parse_voc, txt=parse_txt, csv=parse_csv,
                   yolo=lambda f, i, r: parse_yolo(f, i, r, yolo_names))
    recs = parsers[fmt](kinds[fmt], idx, root) if fmt in parsers else []
    if fmt == "coco" and not recs:
        for alt in ["voc", "txt", "csv", "yolo"]:
            if kinds.get(alt) and alt != "yolo":
                fmt, recs = alt, parsers[alt](kinds[alt], idx, root)
                if recs:
                    break
    # dedupe: same file listed twice, or the same image present in two attached mirrors (same file name)
    seen, by_base = {}, {}
    for r in recs:
        b = os.path.basename(r["path"]).lower()
        if r["path"] in seen or (b in by_base and by_base[b] != r["path"]):
            t = seen.get(r["path"]) or seen[by_base[b]]
            t["objs"] += [o for o in r["objs"] if o not in t["objs"]]
            t["subsets"] = sorted(set(t["subsets"] + r["subsets"]))
        else:
            seen[r["path"]] = r; by_base[b] = r["path"]
    recs = list(seen.values())
    used_lists = apply_idlists(recs, kinds.get("idlist", []), root)
    read_sizes(recs)
    for r in recs:   # YOLO normalized -> absolute
        if "_yolo" in r:
            nm = r.pop("_names") or []
            for c, cx, cy, bw, bh in r.pop("_yolo"):
                W, H = r["w"] or 1, r["h"] or 1
                r["objs"].append((nm[int(c)] if int(c) < len(nm) else f"class{int(c)}",
                                  (cx - bw / 2) * W, (cy - bh / 2) * H, (cx + bw / 2) * W, (cy + bh / 2) * H))
    # negatives (clean bags) = images never mentioned by any annotation
    annotated = {os.path.basename(r["path"]).lower() for r in recs}
    negs, nb = [], set()
    if key in NEG_DATASETS:
        for p in imgs:
            b = os.path.basename(p).lower()
            if b not in annotated and b not in nb:
                negs.append(p); nb.add(b)
    example = None
    if kinds.get(fmt):
        try:
            example = open(kinds[fmt][0], errors="ignore").read(600)
        except Exception:
            pass
    classes = Counter(canon_name(o[0]) for r in recs for o in r["objs"])
    info = dict(roots=roots, format=fmt, n_image_files=len(imgs), n_ann_files={k: len(v) for k, v in kinds.items()},
                n_annotated_images=len(recs), n_unannotated_images=len(negs), idlists_used=used_lists,
                classes=dict(classes.most_common()), yolo_note=yolo_note, example_annotation=example,
                archives=[a for r in roots for a in list_archives(r, 5)])
    return recs, negs, info


def canon_name(n):
    return re.sub(r"[^a-z0-9]", "", str(n).lower())


def fingerprint(classes):
    got = {canon(c) for c in classes} if "canon" in globals() else set()
    raw = {canon_name(c) for c in classes}
    for k, sig in FINGERPRINT.items():
        sig_raw = {canon_name(x) for x in sig}
        hits = len(sig_raw & raw) + len(sig & got)
        if k == "sixray":
            if len(sig_raw & raw) >= 4 and not (raw - sig_raw - {"scissor"}):
                return k
        elif hits >= 2:
            return k
    return None


# canonical-name helper is defined in the ontology cell; a light version is enough for fingerprinting here
_CANON_LITE = {"scissor": "scissors", "multitoolknife": "multitool_knife", "foldingknife": "folding_knife",
               "straightknife": "straight_knife", "utilityknife": "utility_knife", "handcuff": "handcuffs"}


def canon(n):
    k = canon_name(n)
    return _CANON_LITE.get(k, re.sub(r"[^a-z0-9]+", "_", str(n).lower()).strip("_"))


def report(k, d, secs):
    print(f"\n[{k}] format={d['format']}  image files={d['n_image_files']}  annotated={d['n_annotated_images']}  "
          f"unannotated={d['n_unannotated_images']}  ann-files={d['n_ann_files']}  id-lists={d['idlists_used']}  ({secs:.0f}s)")
    print(f"   classes: {d['classes']}")
    if d.get("yolo_note"):
        print("   ⚠️ ", d["yolo_note"])
    if d["example_annotation"]:
        print("   example annotation snippet:\n   " + d["example_annotation"][:300].replace("\n", "\n   "))


if NEED_DISCOVERY:
    resume_from_input()
    GROUPS, UNNAMED = find_dataset_roots()
    print("Name-matched Kaggle inputs:", json.dumps(GROUPS, indent=1))
    print("Other inputs (identified by content):", UNNAMED)

    RAW, RAW_NEG, DISCOVERY = {}, {}, {}
    for k, roots in GROUPS.items():
        for r in roots:
            print("\n" + "=" * 90 + f"\n{k.upper()}"); print_tree(r)
        t0 = time.time()
        roots = GROUPS[k] = prepare_multipart(k, roots)
        RAW[k], RAW_NEG[k], DISCOVERY[k] = load_dataset(k, roots)
        report(k, DISCOVERY[k], time.time() - t0)

    for r in UNNAMED:   # e.g. "xray-contraband-detection" -> fingerprinted as PIDray
        t0 = time.time()
        recs, negs, info = load_dataset(None, [r])
        k = fingerprint(info["classes"]) if recs else None
        if k is None:
            print(f"\n[unidentified input] {r}: {len(recs)} annotated images, classes={list(info['classes'])[:10]} -> ignored "
                  f"(set DATASET_ROOTS if this is one of {DS_KEYS})")
            continue
        if RAW.get(k):
            print(f"\n[{k}] also found in {r} (content match) -> ignored because {GROUPS[k]} is already used")
            continue
        print("\n" + "=" * 90 + f"\n{k.upper()}  (identified by class names)"); print_tree(r)
        if k in NEG_DATASETS:
            recs, negs, info = load_dataset(k, [r])
        RAW[k], RAW_NEG[k], DISCOVERY[k] = recs, negs, info
        GROUPS[k] = [r]
        report(k, info, time.time() - t0)

    for k in list(RAW):
        if not RAW[k]:
            print(f"   ❌ no annotations parsed for {k} ({GROUPS[k]}). "
                  + ("It only contains archives -> attach an extracted mirror or set EXTRACT_ARCHIVES=True. " if DISCOVERY[k]["archives"] else "")
                  + f"Set FORCE_FORMAT['{k}'] / DATASET_ROOTS['{k}'] and re-run.")
            RAW.pop(k); RAW_NEG.pop(k)
    DATASET_ROOTS_FOUND = {k: GROUPS[k] for k in RAW}
    missing = [k for k in DS_KEYS if k not in RAW]
    if missing:
        print("\n⚠️  not available (runs that need them are skipped):", missing)
    if "opixray" in RAW and not any("OL" in s for r in RAW["opixray"] for s in r["subsets"]):
        print("⚠️  OPIXray occlusion lists test_knife-{1,2,3}.txt not found -> P6 (OL1/OL2/OL3, ORD) will be empty. "
              "Upload these 4 small txt files as a Kaggle dataset whose name contains 'opixray' (e.g. opixray-ol-lists).")
    jdump(DISCOVERY, WORK / "dataset_discovery.json")
else:
    print('pack found -> discovery skipped (records, audit and verification are loaded from the pack)')


## 4 · Ontology
Exact / harmonized / hierarchical tracks, reviewed aliases, superclasses, attributes.

In [ ]:
# ==============================================================================
# 4. UNIFIED ONTOLOGY (three evaluation tracks, never mixed):
#      EXACT        : same canonical class name only
#      HARMONIZED   : + documented hyponym aliases (e.g. folding_knife IS-A knife)
#      HIERARCHICAL : threat superclass (sharp / firearm / blunt_tool / ...)
#    An object is UNKNOWN for an experiment iff it is not known under the HARMONIZED track.
# ==============================================================================
CANON = {
    "gun": "gun", "pistol": "gun", "knife": "knife", "knives": "knife", "wrench": "wrench", "pliers": "pliers",
    "plier": "pliers", "scissors": "scissors", "scissor": "scissors", "hammer": "hammer",
    "handcuffs": "handcuffs", "handcuff": "handcuffs", "baton": "baton", "sprayer": "sprayer", "powerbank": "powerbank",
    "lighter": "lighter", "bullet": "bullet",
    "foldingknife": "folding_knife", "straightknife": "straight_knife", "utilityknife": "utility_knife",
    "multitoolknife": "multitool_knife", "multiknife": "multitool_knife",
    "portablecharger1": "portable_charger_1", "portablecharger2": "portable_charger_2", "mobilephone": "mobile_phone",
    "laptop": "laptop", "tablet": "tablet", "cosmetic": "cosmetic", "water": "water",
    "nonmetalliclighter": "nonmetallic_lighter",
    "blade": "blade", "dagger": "dagger", "swissarmyknife": "swiss_army_knife", "cans": "cans",
    "cartondrinks": "carton_drinks", "glassbottle": "glass_bottle", "plasticbottle": "plastic_bottle",
    "vacuumcup": "vacuum_cup", "spraycans": "spray_cans", "tin": "tin",
}


def canon(name):
    k = re.sub(r"[^a-z0-9]", "", str(name).lower())
    return CANON.get(k, re.sub(r"[^a-z0-9]+", "_", str(name).lower()).strip("_"))


# Explicit hyponym aliases (child IS-A parent per the dataset papers' class definitions). Reviewed:
#  * OPIXray knife sub-types and CLCXray dagger / Swiss-army knife are knives;
#  * HiXray "Portable_Charger_1/2" are power banks (PIDray "powerbank");
#  * HiXray "Nonmetallic_Lighter" is a lighter.
#  REMOVED after review: blade -> knife (CLCXray blade = razor blade) and spray_cans -> sprayer
#  (aerosol cans vs. PIDray sprayer); these are left UNKNOWN / superclass-only.
SEM_ALIAS = {"folding_knife": "knife", "straight_knife": "knife", "utility_knife": "knife", "multitool_knife": "knife",
             "swiss_army_knife": "knife", "dagger": "knife",
             "portable_charger_1": "powerbank", "portable_charger_2": "powerbank",
             "nonmetallic_lighter": "lighter"}

SUPER = {  # threat superclass (level 3 of the hierarchy)
    "firearm": ["gun", "bullet"],
    "sharp": ["knife", "folding_knife", "straight_knife", "utility_knife", "multitool_knife", "swiss_army_knife",
              "dagger", "blade", "scissors"],
    "blunt_tool": ["wrench", "pliers", "hammer", "baton"],
    "restraint": ["handcuffs"],
    "pressurized": ["sprayer", "spray_cans"],
    "fire_hazard": ["lighter", "nonmetallic_lighter"],
    "battery_electronic": ["powerbank", "portable_charger_1", "portable_charger_2", "mobile_phone", "laptop", "tablet"],
    "liquid_container": ["water", "cosmetic", "cans", "carton_drinks", "glass_bottle", "plastic_bottle",
                         "vacuum_cup", "tin"],
}
SUPER_OF = {c: s for s, cs in SUPER.items() for c in cs}

ATTR_GROUPS = OrderedDict(
    shape=["elongated", "pointed_edge", "jointed", "rectangular", "cylindrical", "curved", "compact", "ring_shaped"],
    material=["metallic", "dense", "plastic", "liquid", "circuitry", "battery_cell", "glass"],
    function=["cutting", "gripping", "striking", "projectile", "ignition", "restraint", "pressurized",
              "power_storage", "containing", "communication"],
)
ATTRS = [a for g in ATTR_GROUPS.values() for a in g]
_knife = {"elongated", "pointed_edge", "metallic", "cutting"}
CLASS_ATTR = {
    "gun": {"jointed", "curved", "metallic", "dense", "projectile"},
    "bullet": {"cylindrical", "compact", "metallic", "dense", "projectile"},
    "knife": _knife, "straight_knife": _knife, "dagger": _knife | {"dense"}, "blade": {"rectangular", "compact", "metallic", "cutting"},
    "folding_knife": _knife | {"jointed"}, "utility_knife": _knife | {"rectangular", "plastic"},
    "multitool_knife": _knife | {"jointed", "gripping"}, "swiss_army_knife": _knife | {"jointed", "compact"},
    "scissors": {"elongated", "pointed_edge", "jointed", "ring_shaped", "metallic", "cutting"},
    "wrench": {"elongated", "metallic", "dense", "gripping", "striking"},
    "pliers": {"elongated", "jointed", "metallic", "gripping"},
    "hammer": {"elongated", "metallic", "dense", "striking"},
    "baton": {"elongated", "cylindrical", "dense", "striking"},
    "handcuffs": {"ring_shaped", "jointed", "metallic", "restraint"},
    "sprayer": {"cylindrical", "metallic", "liquid", "pressurized", "containing"},
    "spray_cans": {"cylindrical", "metallic", "liquid", "pressurized", "containing"},
    "lighter": {"rectangular", "compact", "metallic", "liquid", "ignition"},
    "nonmetallic_lighter": {"rectangular", "compact", "plastic", "liquid", "ignition"},
    "powerbank": {"rectangular", "dense", "circuitry", "battery_cell", "power_storage"},
    "portable_charger_1": {"rectangular", "dense", "circuitry", "battery_cell", "power_storage"},
    "portable_charger_2": {"rectangular", "compact", "circuitry", "battery_cell", "power_storage"},
    "mobile_phone": {"rectangular", "compact", "circuitry", "battery_cell", "communication"},
    "laptop": {"rectangular", "dense", "circuitry", "battery_cell", "communication"},
    "tablet": {"rectangular", "circuitry", "battery_cell", "communication"},
    "water": {"cylindrical", "plastic", "liquid", "containing"},
    "cosmetic": {"cylindrical", "compact", "plastic", "liquid", "containing"},
    "cans": {"cylindrical", "metallic", "liquid", "containing"}, "tin": {"cylindrical", "metallic", "containing"},
    "carton_drinks": {"rectangular", "liquid", "containing"},
    "glass_bottle": {"cylindrical", "glass", "liquid", "containing"},
    "plastic_bottle": {"cylindrical", "plastic", "liquid", "containing"},
    "vacuum_cup": {"cylindrical", "metallic", "liquid", "containing"},
}
TEXT_NAME = {c: c.replace("_", " ") for c in list(CLASS_ATTR)}
TEXT_NAME.update(powerbank="power bank", portable_charger_1="portable charger", portable_charger_2="portable charger",
                 nonmetallic_lighter="non-metallic lighter", multitool_knife="multi-tool knife")
# global order used to number classes in every experiment
GLOBAL_ORDER = ["gun", "knife", "wrench", "pliers", "scissors", "hammer", "handcuffs", "baton", "sprayer", "powerbank",
                "lighter", "bullet", "folding_knife", "straight_knife", "utility_knife", "multitool_knife",
                "portable_charger_1", "portable_charger_2", "mobile_phone", "laptop", "tablet", "cosmetic", "water",
                "nonmetallic_lighter", "blade", "dagger", "swiss_army_knife", "cans", "carton_drinks", "glass_bottle",
                "plastic_bottle", "vacuum_cup", "spray_cans", "tin"]


def attr_vector(c):
    return np.array([1.0 if a in CLASS_ATTR.get(c, set()) else 0.0 for a in ATTRS], dtype=np.float32)


def superclass(c):
    return SUPER_OF.get(c, SUPER_OF.get(SEM_ALIAS.get(c, ""), "other"))


def resolve_known(c, known, track="harmonized"):
    """track='exact': same name only. track='harmonized': + SEM_ALIAS. Returns the known class or None."""
    if c in known:
        return c
    if track == "harmonized" and C.USE_SEMANTIC_ALIASES and SEM_ALIAS.get(c) in known:
        return SEM_ALIAS[c]
    return None


def order_classes(cs):
    cs = sorted(set(cs), key=lambda c: (GLOBAL_ORDER.index(c) if c in GLOBAL_ORDER else 999, c))
    return cs


ONTOLOGY_VERSION = sha(dict(canon=CANON, alias=SEM_ALIAS if C.USE_SEMANTIC_ALIASES else {}, super=SUPER,
                            attrs={k: sorted(v) for k, v in CLASS_ATTR.items()}))
print("ontology version", ONTOLOGY_VERSION)


# CLIP prompts (shared by the prepare step, which caches the embeddings, and the model)
TEXT_PROMPTS = ["an x-ray scan of a {}", "a {} inside an x-ray baggage image", "a photo of a {}"]
THREAT_PROMPTS = ["an x-ray scan of a dangerous prohibited item", "a threat object hidden in luggage", "a prohibited item"]


def all_text_prompts():
    return [t.format(TEXT_NAME.get(c, c.replace("_", " "))) for c in GLOBAL_ORDER for t in TEXT_PROMPTS] + THREAT_PROMPTS


def clip_encode_text(model, tokenizer, prompts, batch=64):
    """One L2-normalised CLIP text embedding per prompt, independent of the transformers version:
    text_model -> pooled (EOS) output -> text_projection. (get_text_features returns a per-TOKEN tensor in some
    transformers versions, which silently produced (tokens x 512) "embeddings".)"""
    import torch, torch.nn.functional as F
    outs = []
    with torch.no_grad():
        for i in range(0, len(prompts), batch):
            t = tokenizer(prompts[i:i + batch], padding=True, return_tensors="pt")
            o = model.text_model(input_ids=t["input_ids"], attention_mask=t.get("attention_mask"))
            pooled = o.pooler_output if getattr(o, "pooler_output", None) is not None else o[1]
            e = model.text_projection(pooled)
            assert e.dim() == 2 and e.shape[0] == len(prompts[i:i + batch]), f"bad CLIP text output {tuple(e.shape)}"
            outs.append(F.normalize(e.float(), dim=-1))
    return torch.cat(outs)


def valid_text_cache(cache):
    """Keep only well-formed entries (1-D vectors of the same size)."""
    vecs = {k: v for k, v in cache.items() if k != "__meta__" and torch.is_tensor(v) and v.dim() == 1}
    if vecs:
        d = Counter(v.shape[0] for v in vecs.values()).most_common(1)[0][0]
        vecs = {k: v for k, v in vecs.items() if v.shape[0] == d}
    return vecs


## 5 · Manifest & data integrity
Lineage (original image ids), official-size verification, split metadata, class presence audit.

In [ ]:
# ==============================================================================
# 5. IMMUTABLE MANIFEST + DATA INTEGRITY
#   * canonical labels, cleaned boxes, subsets (OL1-3, easy/hard/hidden, negative)
#   * LINEAGE: every image is traced to its ORIGINAL image id (Roboflow names such as
#     "P00031_jpg.rf.<hash>.jpg" -> P00031). Augmented copies are collapsed to ONE copy per id and an id
#     may live in only one split (test > val > train), so no original image leaks across splits.
#   * split origin per dataset (official / mirror / derived) + disjointness asserts
#   * verification against the official dataset sizes (dataset_verification.csv)
#   * class_presence_audit.csv (decides which LOCO folds are valid)
# ==============================================================================
OFFICIAL = {   # sizes published by the dataset papers / official repositories
    "pidray":  dict(train=29457, test=18220, easy=9482, hard=3733, hidden=5005),
    "hixray":  dict(train=36295, test=9069),
    "opixray": dict(train=7109, test=1776),
    "sixray":  dict(positives=8929),
}
SPLIT_ORIGIN = {"pidray": "official (COCO json per split)", "hixray": "official (train/ test/ folders)",
                "opixray": "official, reconstructed (mirror val 888 + test 888 = official test 1,776)",
                "sixray": "MIRROR split (Roboflow export khanhbtq99/sixray), NOT an official SIXray detection split",
                "clcxray": "mirror"}


class StopExecution(Exception):
    def _render_traceback_(self):
        return []


def lineage_id(ds, img):
    s = os.path.basename(str(img))
    if ds == "sixray":
        m = re.match(r"^([PN]\d+)", s, re.I)
        if m:
            return m.group(1).upper()
    if ds == "opixray":
        m = re.search(r"(\d{5,})", s)
        if m:
            return m.group(1)
    s = re.sub(r"_(jpg|jpeg|png)\.rf\.[0-9a-f]+$", "", os.path.splitext(s)[0], flags=re.I)
    return s


def hash_split(key, test_frac=0.2):
    return "test" if stable_hash(key) % 1000 < test_frac * 1000 else "train"


def collapse_lineage(k, recs, log):
    """One record per original id; an id lives in one split only (priority test > val > train)."""
    prio = {"test": 0, "val": 1, "train": 2, None: 3}
    groups = defaultdict(list)
    for r in recs:
        groups[r["lineage"]].append(r)
    keep, n_cross, n_copies = [], 0, 0
    for lid, rs in groups.items():
        splits = {r["split"] for r in rs}
        if len(splits) > 1:
            n_cross += 1
        best = min(splits, key=lambda s: prio.get(s, 3))
        cand = sorted([r for r in rs if r["split"] == best], key=lambda r: stable_hash(r["path"]))
        n_copies += len(rs) - 1
        keep.append(cand[0])
    log.update(files=len(recs), unique_ids=len(groups), dropped_copies=n_copies, ids_in_several_splits=n_cross,
               max_copies_per_id=max((len(v) for v in groups.values()), default=0))
    return keep


def build_records():
    REC, split_policy, lineage_log = {}, {}, {}
    for k, raw in RAW.items():
        recs, dropped = [], 0
        for r in raw:
            W, H = r.get("w"), r.get("h")
            if not W or not H:
                continue
            bx, cl = [], []
            for (c, x1, y1, x2, y2) in r["objs"]:
                x1, x2 = sorted((x1, x2)); y1, y2 = sorted((y1, y2))
                x1, y1 = max(0.0, x1), max(0.0, y1); x2, y2 = min(float(W), x2), min(float(H), y2)
                if x2 - x1 < 2 or y2 - y1 < 2:
                    dropped += 1; continue
                bx.append([x1, y1, x2, y2]); cl.append(canon(c))
            recs.append(dict(ds=k, img=r["img"], path=r["path"], orig_path=r["path"], split=r["split"],
                             subsets=list(r["subsets"]), w=int(W), h=int(H), boxes=np.array(bx, np.float32).reshape(-1, 4),
                             cls=cl, neg=len(bx) == 0, lineage=lineage_id(k, r["img"])))
        policy = []
        if k in MERGE_VAL_INTO_TEST and any(r["split"] == "val" for r in recs):
            nv = sum(r["split"] == "val" for r in recs)
            for r in recs:
                if r["split"] == "val":
                    r["split"] = "test"
            policy.append(f"mirror val ({nv}) + test merged back into the official test set")
        if k == "sixray":   # Roboflow images whose label file is empty are NOT clean bags: drop them
            n0 = len(recs); recs = [r for r in recs if not (r["neg"] and r["lineage"].startswith("P"))]
            if n0 - len(recs):
                policy.append(f"{n0 - len(recs)} positive-id images with empty labels dropped")
        lineage_log[k] = {}
        recs = collapse_lineage(k, recs, lineage_log[k])
        if lineage_log[k]["dropped_copies"] or lineage_log[k]["ids_in_several_splits"]:
            policy.append(f"lineage: {lineage_log[k]['files']} files -> {lineage_log[k]['unique_ids']} original ids "
                          f"({lineage_log[k]['dropped_copies']} augmented/duplicate copies dropped, "
                          f"{lineage_log[k]['ids_in_several_splits']} ids appeared in >1 split)")
        splits = Counter(r["split"] for r in recs)
        if splits.get("test", 0) == 0:
            for r in recs:
                r["split"] = hash_split(k + r["lineage"])
            policy.append("no test split shipped -> deterministic md5 split 80/20 by original id (CUSTOM split)")
        else:
            for r in recs:
                if r["split"] is None:
                    r["split"] = hash_split(k + r["lineage"])
            policy.append("train/test split as shipped")
        if not any(r["split"] == "val" for r in recs):
            for r in recs:
                if r["split"] == "train" and stable_hash("val" + k + r["lineage"]) % 10 == 0:
                    r["split"] = "val"
            policy.append("val = 10% of train by original id (thresholds / checkpoint selection ONLY)")
        # clean (negative) bags = SIXray N*.jpg from the official archive part
        negs = sorted([p for p in RAW_NEG.get(k, []) if k != "sixray" or re.match(r"^N\d+", os.path.basename(p), re.I)],
                      key=lambda p: stable_hash(p))
        n_need = C.N_NEG_TRAIN + C.N_NEG_EVAL
        neg_recs = [dict(ds=k, img=os.path.splitext(os.path.basename(p))[0], path=p, orig_path=p, split=None,
                         subsets=["negative"], w=None, h=None, boxes=np.zeros((0, 4), np.float32), cls=[], neg=True,
                         lineage=lineage_id(k, p)) for p in negs[:n_need]]
        if neg_recs:
            read_sizes(neg_recs)
            for i, r in enumerate(neg_recs):
                r["split"] = "train" if i < C.N_NEG_TRAIN else "test"
            neg_recs = [r for r in neg_recs if r["w"]]
            policy.append(f"{len(neg_recs)} clean bags (train {C.N_NEG_TRAIN} / test {C.N_NEG_EVAL})")
        recs += neg_recs
        for r in recs:
            r["uid"] = f"{k}/{hashlib.md5(r['orig_path'].encode()).hexdigest()[:14]}"
            if r["neg"] and "negative" not in r["subsets"]:
                r["subsets"].append("negative")
        REC[k] = recs
        split_policy[k] = policy
        print(f"[{k}] records={len(recs)} dropped-degenerate-boxes={dropped}  " + " | ".join(policy))
    return REC, split_policy, lineage_log


def split_metadata():
    meta, rows = {}, []
    for k, recs in REC.items():
        ids = {sp: {r["lineage"] for r in recs if r["split"] == sp} for sp in ["train", "val", "test"]}
        assert ids["train"].isdisjoint(ids["test"]), f"{k}: train/test share original ids"
        assert ids["val"].isdisjoint(ids["test"]), f"{k}: val/test share original ids"
        assert ids["train"].isdisjoint(ids["val"]), f"{k}: train/val share original ids"
        meta[k] = dict(dataset=k, source=DISCOVERY.get(k, {}).get("roots"), split_origin=SPLIT_ORIGIN.get(k, "as shipped"),
                       policy=SPLIT_POLICY.get(k), lineage=LINEAGE_LOG.get(k),
                       n_ids={sp: len(v) for sp, v in ids.items()}, ids_sha={sp: sha(sorted(v)) for sp, v in ids.items()})
        obs = Counter(r["split"] for r in recs if not r["neg"])
        sub = Counter(s for r in recs if r["split"] == "test" for s in r["subsets"])
        observed = dict(train=obs["train"] + obs["val"], test=obs["test"], positives=sum(obs.values()), **sub)
        for key, off in OFFICIAL.get(k, {}).items():
            got = observed.get(key, 0)
            rows.append(dict(dataset=k, quantity=key, official=off, observed=got, diff=got - off,
                             status="match" if got == off else ("close (<0.1%)" if abs(got - off) <= max(3, 0.001 * off) else "MISMATCH")))
    jdump(meta, WORK / "audit" / "split_metadata.json")
    ver = pd.DataFrame(rows)
    ver.to_csv(WORK / "audit" / "dataset_verification.csv", index=False)
    return meta, ver


def class_presence_audit():
    rows = []
    for k, recs in REC.items():
        classes = sorted({c for r in recs for c in r["cls"]})
        for c in classes:
            cnt = {sp: sum(r["cls"].count(c) for r in recs if r["split"] == sp) for sp in ["train", "val", "test"]}
            img = {sp: sum(c in r["cls"] for r in recs if r["split"] == sp) for sp in ["train", "val", "test"]}
            rows.append(dict(dataset=k, cls=c, superclass=superclass(c), train_objects=cnt["train"], val_objects=cnt["val"],
                             test_objects=cnt["test"], train_images=img["train"], test_images=img["test"],
                             valid_loco=bool(k == "sixray" and cnt["train"] >= C.LOCO_MIN_OBJ and cnt["test"] >= C.LOCO_MIN_OBJ)))
    df = pd.DataFrame(rows)
    df.to_csv(WORK / "audit" / "class_presence_audit.csv", index=False)
    return df


if PACK_READY:
    _meta = pickle.load(open(PACK_LOCAL / "meta.pkl", "rb"))
    REC, DISCOVERY, SPLIT_POLICY, LINEAGE_LOG = _meta["REC"], _meta["DISCOVERY"], _meta["SPLIT_POLICY"], _meta["LINEAGE_LOG"]
    EXCLUDED, AUDIT_SUMMARY = _meta["EXCLUDED"], _meta.get("AUDIT_SUMMARY", {})
    for r in (r for v in REC.values() for r in v):
        r["path"] = str(PACK_LOCAL / r["pack_rel"])
    DATASET_ROOTS_FOUND = {k: DISCOVERY[k].get("roots") for k in REC}
    print(f"loaded pack {_meta['pack_id']} built {_meta['built']}: " + ", ".join(f"{k}={len(v)}" for k, v in REC.items()))
else:
    REC, SPLIT_POLICY, LINEAGE_LOG = build_records()
    EXCLUDED, AUDIT_SUMMARY = {}, {}
ALL_REC = [r for k in REC for r in REC[k]]
UID = {r["uid"]: r for r in ALL_REC}
if PACK_READY:   # audits of the FULL datasets, computed once in the prepare step
    SPLIT_META, VERIFICATION, CLASS_AUDIT = _meta["SPLIT_META"], _meta["VERIFICATION"], _meta["CLASS_AUDIT"]
else:
    SPLIT_META, VERIFICATION = split_metadata()
    CLASS_AUDIT = class_presence_audit()
print("\nVerification against official dataset sizes:")
display(VERIFICATION)
display(CLASS_AUDIT)
display(pd.DataFrame(LINEAGE_LOG).T)


def dataset_stats():
    S, rows = {}, []
    for k, recs in REC.items():
        d = dict(root=DATASET_ROOTS_FOUND.get(k), format=DISCOVERY[k]["format"], split_policy=SPLIT_POLICY[k],
                 split_origin=SPLIT_ORIGIN.get(k), splits={})
        for sp in ["train", "val", "test"]:
            rs = [r for r in recs if r["split"] == sp and not r.get("excluded")]
            cls_cnt = Counter(c for r in rs for c in r["cls"])
            areas = np.array([((b[2] - b[0]) * (b[3] - b[1])) / (r["w"] * r["h"]) for r in rs for b in r["boxes"]] or [np.nan])
            sub = Counter(s for r in rs for s in r["subsets"])
            d["splits"][sp] = dict(images=len(rs), positives=sum(not r["neg"] for r in rs), negatives=sum(r["neg"] for r in rs),
                                   instances=int(sum(cls_cnt.values())), classes=dict(cls_cnt.most_common()),
                                   subsets=dict(sub), rel_box_area_median=float(np.nanmedian(areas)),
                                   img_w_median=float(np.median([r["w"] for r in rs])) if rs else None,
                                   img_h_median=float(np.median([r["h"] for r in rs])) if rs else None)
            for c, n in cls_cnt.items():
                rows.append(dict(dataset=k, split=sp, cls=c, superclass=superclass(c), instances=n))
        S[k] = d
    jdump(S, WORK / "dataset_stats.json")
    return S, pd.DataFrame(rows)


raw_names = defaultdict(set)
for k, d in DISCOVERY.items():
    for n in d.get("raw_class_names", d.get("classes", {})):
        raw_names[k].add(n)
jdump(dict(raw_to_canonical={k: {n: canon(n) for n in sorted(v)} for k, v in raw_names.items()},
           semantic_aliases=SEM_ALIAS if C.USE_SEMANTIC_ALIASES else {}, superclasses=SUPER, attributes=ATTR_GROUPS,
           class_attributes={c: sorted(v) for c, v in CLASS_ATTR.items()}, ontology_version=ONTOLOGY_VERSION),
      WORK / "class_mapping.json")


def write_manifest(excluded=None):
    excluded = excluded or {}
    rows = [dict(uid=r["uid"], dataset=r["ds"], image=r["img"], lineage_id=r["lineage"], source_path=r["orig_path"],
                 split=r["split"], subsets="|".join(r["subsets"]), width=r["w"], height=r["h"], n_boxes=len(r["boxes"]),
                 classes="|".join(sorted(set(r["cls"]))), negative=r["neg"], excluded_reason=excluded.get(r["uid"], ""))
            for r in ALL_REC]
    df = pd.DataFrame(rows)
    df.to_csv(WORK / "audit" / "final_split_manifest.csv", index=False)
    return sha(df[["uid", "split", "excluded_reason"]].values.tolist())


## 6 · Leakage audit
SHA-256 exact duplicates, confirmed near-duplicates, clusters, cross-dataset overlap, frozen exclusions.

In [ ]:
# ==============================================================================
# 6. DATA-LEAKAGE AUDIT (runs once in "prepare"; the exclusion policy is FROZEN in the pack
#    before any model is evaluated -- test performance never influences exclusions)
#   (1) byte-identical duplicates : SHA-256 of the file bytes
#   (2) visual near-duplicates    : 64-bit dHash candidates (LSH, Hamming <= DUP_HAMMING) CONFIRMED by a
#                                   16x16 grey-thumbnail mean |diff| <= DUP_MAD (X-ray bags share backgrounds)
#   (3) annotation conflicts      : same image content with different label sets
#   Duplicates are merged into clusters (union-find). Any val/test member of a cluster that also has a TRAIN
#   member (any dataset) is excluded from evaluation; cross-dataset clusters are reported separately.
#   Files: audit/exact_duplicate_report.csv, near_duplicate_report.csv, cross_dataset_overlap.csv,
#          excluded_images.csv, final_split_manifest.csv
# ==============================================================================
from multiprocessing import Pool


def image_fingerprint(path):
    """-> (sha256 of bytes, 64-bit dHash, 16x16 grey thumbnail bytes)"""
    try:
        data = open(path, "rb").read()
        h = hashlib.sha256(data).hexdigest()
        with Image.open(io.BytesIO(data)) as im:
            try:
                im.draft("L", (64, 64))
            except Exception:
                pass
            g = im.convert("L")
            a = np.asarray(g.resize((9, 8), Image.BILINEAR), dtype=np.int16)
            t = np.asarray(g.resize((16, 16), Image.BILINEAR), dtype=np.uint8).tobytes()
        bits = (a[:, 1:] > a[:, :-1]).flatten()
        return h, int("".join("1" if b else "0" for b in bits), 2), t
    except Exception:
        return None, -1, b""


class UF:
    def __init__(self):
        self.p = {}

    def find(self, x):
        self.p.setdefault(x, x)
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]; x = self.p[x]
        return x

    def union(self, a, b):
        self.p[self.find(a)] = self.find(b)


def run_audit():
    cache = CACHE / "audit_fingerprints.pkl"
    have = pickle.load(open(cache, "rb")) if cache.exists() else {}
    recs = []
    for k, rs in REC.items():
        rs = sorted(rs, key=lambda r: stable_hash(r["uid"]))
        recs += rs[: C.AUDIT_MAX_PER_DS] if C.AUDIT_MAX_PER_DS else rs
    todo = [r["orig_path"] for r in recs if r["orig_path"] not in have]
    if todo:
        t0 = time.time()
        with Pool(max(1, os.cpu_count() or 2)) as pool:
            res = pool.map(image_fingerprint, todo, chunksize=64)
        have.update(zip(todo, res))
        pickle.dump(have, open(cache, "wb"))
        print(f"fingerprinted {len(todo)} images in {(time.time() - t0) / 60:.1f} min")
    F_ = {r["uid"]: have[r["orig_path"]] for r in recs if have[r["orig_path"]][0] is not None}
    uf = UF()
    # (1) exact
    by_sha = defaultdict(list)
    for u, (h, _, _) in F_.items():
        by_sha[h].append(u)
    exact_rows = []
    for h, us in by_sha.items():
        if len(us) > 1:
            labels = {u: tuple(sorted(UID[u]["cls"])) for u in us}
            for u in us[1:]:
                uf.union(us[0], u)
            exact_rows.append(dict(sha256=h, n=len(us), uids="|".join(us), datasets="|".join(sorted({UID[u]["ds"] for u in us})),
                                   splits="|".join(sorted({str(UID[u]["split"]) for u in us})),
                                   annotation_conflict=len(set(labels.values())) > 1))
    # (2) near
    H = {u: v[1] for u, v in F_.items() if v[1] >= 0}
    TH = {u: np.frombuffer(v[2], np.uint8).astype(np.int16) for u, v in F_.items() if v[1] >= 0}
    bands = defaultdict(list)
    for u, h in H.items():
        for b in range(4):
            bands[(b, (h >> (16 * b)) & 0xFFFF)].append(u)
    pairs, n_cand = set(), 0
    for lst in bands.values():
        if 1 < len(lst) < 2000:
            for i in range(len(lst)):
                for j in range(i + 1, len(lst)):
                    a, b = lst[i], lst[j]
                    d = bin(H[a] ^ H[b]).count("1")
                    if d <= C.DUP_HAMMING:
                        n_cand += 1
                        mad = float(np.abs(TH[a] - TH[b]).mean())
                        if mad <= C.DUP_MAD and F_[a][0] != F_[b][0]:
                            pairs.add((min(a, b), max(a, b), d, round(mad, 2)))
    near_rows = []
    for a, b, d, mad in pairs:
        uf.union(a, b)
        near_rows.append(dict(uid_a=a, uid_b=b, dataset_a=UID[a]["ds"], dataset_b=UID[b]["ds"], split_a=UID[a]["split"],
                              split_b=UID[b]["split"], hamming=d, thumb_mad=mad,
                              annotation_conflict=sorted(UID[a]["cls"]) != sorted(UID[b]["cls"])))
    # clusters -> exclusions
    clusters = defaultdict(list)
    for u in list(uf.p):
        clusters[uf.find(u)].append(u)
    excluded, cross_rows = {}, []
    for root, us in clusters.items():
        if len(us) < 2:
            continue
        train_members = [u for u in us if UID[u]["split"] == "train"]
        dss = sorted({UID[u]["ds"] for u in us})
        if len(dss) > 1:
            cross_rows.append(dict(cluster=root, size=len(us), datasets="|".join(dss),
                                   members="|".join(f"{UID[u]['ds']}:{UID[u]['split']}:{UID[u]['img']}" for u in us)))
        if train_members:
            for u in us:
                if UID[u]["split"] in ("val", "test"):
                    excluded[u] = f"duplicate cluster with train image {train_members[0]} ({'cross-dataset' if len(dss) > 1 else 'within-dataset'})"
    pd.DataFrame(exact_rows).to_csv(WORK / "audit" / "exact_duplicate_report.csv", index=False)
    pd.DataFrame(near_rows).to_csv(WORK / "audit" / "near_duplicate_report.csv", index=False)
    pd.DataFrame(cross_rows).to_csv(WORK / "audit" / "cross_dataset_overlap.csv", index=False)
    pd.DataFrame([dict(uid=u, dataset=UID[u]["ds"], split=UID[u]["split"], image=UID[u]["img"], reason=v)
                  for u, v in excluded.items()]).to_csv(WORK / "audit" / "excluded_images.csv", index=False)
    summary = dict(images=len(F_), exact_clusters=len(exact_rows), exact_annotation_conflicts=sum(r["annotation_conflict"] for r in exact_rows),
                   near_candidates=n_cand, near_confirmed=len(near_rows), cross_dataset_clusters=len(cross_rows),
                   excluded_eval_images=len(excluded),
                   excluded_by_dataset=dict(Counter(UID[u]["ds"] for u in excluded)))
    return excluded, summary


if NEED_DISCOVERY:
    EXCLUDED, AUDIT_SUMMARY = run_audit()
for r in ALL_REC:
    r["excluded"] = r["uid"] in EXCLUDED
if NEED_DISCOVERY:
    MANIFEST_HASH = write_manifest(EXCLUDED)
    STATS, CLS_DF = dataset_stats()
else:
    MANIFEST_HASH, STATS = _meta["MANIFEST_HASH"], _meta["STATS"]
print("audit:", json.dumps(AUDIT_SUMMARY, default=str))
print("manifest hash:", MANIFEST_HASH)
display(pd.DataFrame([dict(dataset=k, split_origin=SPLIT_ORIGIN.get(k), **{f"{sp}_img": v["splits"][sp]["images"] for sp in ["train", "val", "test"]},
                           test_subsets=v["splits"]["test"]["subsets"]) for k, v in STATS.items()]))


## 7 · Pack
`prepare` writes the compact pack to Drive and stops here.

In [ ]:
# ==============================================================================
# 7. PACK (prepare mode): one compact copy of every image the experiments can use
#    * long side resized to C.PACK_LONG (= training resolution), JPEG quality C.PACK_QUALITY
#    * boxes, widths and heights stay in ORIGINAL pixel coordinates -> metrics (incl. AP-S/M/L) unchanged
#    * train images capped exactly like the experiment registry (C.MAX_TRAIN_PER_DS, same md5 order);
#      all val / test / clean-bag images are kept
#    * written as one tar per dataset to Google Drive + meta.pkl (records, verification, audit, policy)
#    GPU sessions only copy these shards to local SSD: no Kaggle download, no image decoding of huge PNGs.
# ==============================================================================
def train_cap_order(r):
    return stable_hash("tr" + r["uid"])


def select_for_pack():
    """PACK_SCOPE 'colab': tier 1 trains only on SIXray -> other datasets ship only val/test (smaller, faster restore).
    PACK_SCOPE 'all'  : Kaggle tiers 2-3 also train on PIDray / HiXray / OPIXray."""
    keep = []
    for k, recs in REC.items():
        tr = sorted([r for r in recs if r["split"] == "train" and not r["neg"]], key=train_cap_order)
        if C.MAX_TRAIN_PER_DS:
            tr = tr[: C.MAX_TRAIN_PER_DS]
        if PACK_SCOPE == "colab" and k != "sixray":
            tr = []
        keep += tr + [r for r in recs if r["split"] != "train" or r["neg"]]
    return keep


def precompute_text_and_weights():
    """Done on the CPU so that GPU sessions never download or load anything except the detector weights."""
    os.environ["TORCH_HOME"] = str(PACK_DRIVE / "torch")
    try:
        from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights
        from torchvision.models import ResNet50_Weights
        for w in [FasterRCNN_ResNet50_FPN_Weights.COCO_V1, ResNet50_Weights.IMAGENET1K_V1]:
            torch.hub.load_state_dict_from_url(w.url, progress=False)
        print("   COCO + ImageNet weights cached in the pack")
    except Exception as e:
        print("   ⚠️  could not cache detector weights (GPU session will download them):", str(e)[:120])
    try:
        import transformers
        from transformers import CLIPModel, CLIPTokenizer
        prompts = all_text_prompts()
        tok, m = CLIPTokenizer.from_pretrained(C.CLIP_HF_NAME), CLIPModel.from_pretrained(C.CLIP_HF_NAME).eval()
        cache = dict(zip(prompts, clip_encode_text(m, tok, prompts)))
        cache["__meta__"] = dict(model=C.CLIP_HF_NAME, transformers=transformers.__version__, n=len(prompts))
        torch.save(cache, PACK_DRIVE / "clip_text.pt")
        print(f"   CLIP text embeddings for {len(prompts)} prompts cached in the pack")
    except Exception as e:
        print("   ⚠️  CLIP text embeddings NOT cached:", str(e)[:150])


def pack_one(args):
    src, dst, long_side, q = args
    try:
        if os.path.exists(dst):
            return True
        with Image.open(src) as im:
            W, H = im.size
            s = min(1.0, long_side / max(W, H))
            try:
                im.draft("RGB", (int(W * s) + 1, int(H * s) + 1))
            except Exception:
                pass
            im = im.convert("RGB")
            if s < 1.0:
                im = im.resize((max(1, round(W * s)), max(1, round(H * s))), Image.BILINEAR, reducing_gap=2.0)
            im.save(dst, "JPEG", quality=q, optimize=False)
        return True
    except Exception as e:
        return f"{src}: {e}"


if MODE == "prepare":
    sel = select_for_pack()
    pack_id = sha(dict(manifest=MANIFEST_HASH, long=C.PACK_LONG, q=C.PACK_QUALITY, cap=C.MAX_TRAIN_PER_DS, n=len(sel)), 10)
    jobs = []
    for r in sel:
        rel = f"images/{r['ds']}/{r['uid'].split('/')[1]}.jpg"
        r["pack_rel"] = rel
        (PACK_LOCAL / "images" / r["ds"]).mkdir(parents=True, exist_ok=True)
        jobs.append((r["orig_path"], str(PACK_LOCAL / rel), C.PACK_LONG, C.PACK_QUALITY))
    t0 = time.time()
    with Pool(max(1, os.cpu_count() or 2)) as pool:
        res = pool.map(pack_one, jobs, chunksize=32)
    bad = [x for x in res if x is not True]
    print(f"packed {len(jobs) - len(bad)} images in {(time.time() - t0) / 60:.1f} min ({len(bad)} failed)")
    if bad:
        print("first failures:", bad[:5])
    sel_uids = {r["uid"] for r in sel if r.get("pack_rel")}
    PACKED_REC = {k: [r for r in v if r["uid"] in sel_uids] for k, v in REC.items()}
    PACK_DRIVE.mkdir(parents=True, exist_ok=True)
    if not SMOKE:
        import torch, torch.nn.functional as F
        precompute_text_and_weights()
    for k in PACKED_REC:
        subprocess.run(["tar", "-cf", str(PACK_DRIVE / f"images_{k}.tar"), "-C", str(PACK_LOCAL), f"images/{k}"], check=True)
    meta = dict(REC=PACKED_REC, DISCOVERY=DISCOVERY, SPLIT_POLICY=SPLIT_POLICY, LINEAGE_LOG=LINEAGE_LOG, EXCLUDED=EXCLUDED,
                AUDIT_SUMMARY=AUDIT_SUMMARY, STATS=STATS, VERIFICATION=VERIFICATION, CLASS_AUDIT=CLASS_AUDIT,
                SPLIT_META=SPLIT_META, MANIFEST_HASH=MANIFEST_HASH, ONTOLOGY_VERSION=ONTOLOGY_VERSION,
                pack_id=pack_id, built=time.strftime("%Y-%m-%d %H:%M"), PACK_LONG=C.PACK_LONG, PACK_QUALITY=C.PACK_QUALITY,
                PACK_SCOPE=PACK_SCOPE, PLATFORM=PLATFORM)
    pickle.dump(meta, open(PACK_DRIVE / "meta.pkl", "wb"))
    size = sum(f.stat().st_size for f in PACK_DRIVE.glob("images_*.tar")) / 1e9
    print(f"\n✅ pack {pack_id} written to {PACK_DRIVE} ({size:.1f} GB, {len(sel_uids)} images).")
    if PLATFORM == "kaggle":
        print("NEXT: Save Version (Save & Run All) -> new GPU notebook -> Add Input: THIS notebook's output -> MODE='test'.")
    else:
        print("NEXT: Runtime -> Change runtime type -> A100 GPU, set MODE = 'test', Run all.")
    raise StopExecution()
elif PACK_READY:
    print(f"pack {_meta['pack_id']}: verification / class audit / split metadata are those of the FULL datasets (prepare step)")
    VERIFICATION.to_csv(WORK / "audit" / "dataset_verification.csv", index=False)
    CLASS_AUDIT.to_csv(WORK / "audit" / "class_presence_audit.csv", index=False)
    jdump(SPLIT_META, WORK / "audit" / "split_metadata.json")
    jdump(STATS, WORK / "dataset_stats.json")


## 8 · Data pipeline & augmentations

In [ ]:
# ==============================================================================
# 8. DATA PIPELINE + X-RAY AUGMENTATIONS
#    T_phi (scanner counterfactual): attenuation curve, pseudo-energy channel mixing,
#    colour-map rotation, saturation, contrast, detector saturation, blur, Poisson/Gaussian
#    noise -- geometry untouched, so boxes are unchanged.
#    Occlusion synthesis: x_occ = M*x + (1-M)*B  (replace) or Beer-Lambert overlay x*B.
#    Baselines: RGB jitter, Fourier amplitude mixing (FDA-style), Cutout.
# ==============================================================================
if MODE == "prepare":
    raise StopExecution()   # prepare mode ends after the pack cell
def load_rgb(path, target_long):
    im = Image.open(path)
    try:
        im.draft("RGB", (target_long, target_long))
    except Exception:
        pass
    if im.mode not in ("RGB",):
        if im.mode in ("I;16", "I", "F"):
            a = np.asarray(im, dtype=np.float32); a = (a - a.min()) / max(1e-6, a.max() - a.min())
            im = Image.fromarray((a * 255).astype(np.uint8))
        im = im.convert("RGB")
    return im


class XrayDet(Dataset):
    def __init__(self, recs, cls2id, img_size, train=False, aug_seed=0):
        self.recs, self.cls2id, self.S, self.train, self.aug_seed = recs, cls2id, img_size, train, aug_seed

    def __len__(self):
        return len(self.recs)

    def __getitem__(self, i):
        r = self.recs[i]
        s = self.S / max(r["w"], r["h"])
        nw, nh = max(1, round(r["w"] * s)), max(1, round(r["h"] * s))
        try:
            im = load_rgb(r["path"], self.S).resize((nw, nh), Image.BILINEAR)
            x = torch.from_numpy(np.asarray(im, dtype=np.uint8).copy()).permute(2, 0, 1).float().div_(255)
        except Exception as e:
            print("image read error", r["path"], e)
            x = torch.zeros(3, nh, nw)
        keep = [j for j, c in enumerate(r["cls"]) if self.cls2id.get(c) is not None]
        boxes = torch.from_numpy(r["boxes"][keep] * s).float().reshape(-1, 4)
        labels = torch.tensor([self.cls2id[r["cls"][j]] for j in keep], dtype=torch.int64)
        if self.train and stable_hash(f"{self.aug_seed}-{i}") % 2:   # deterministic flip: exact resume
            x = x.flip(-1)
            if len(boxes):
                boxes = torch.stack([nw - boxes[:, 2], boxes[:, 1], nw - boxes[:, 0], boxes[:, 3]], 1)
        return x, dict(boxes=boxes, labels=labels, idx=i, scale=s)


def collate(b):
    return tuple(zip(*b))


class ResumableSampler(Sampler):
    """Deterministic per-epoch permutation that can start mid-epoch (for time-limited Kaggle sessions)."""
    def __init__(self, n, seed, epoch=0, start=0, weights=None):
        self.n, self.seed, self.epoch, self.start, self.weights = n, seed, epoch, start, weights

    def __iter__(self):
        g = np.random.RandomState(self.seed * 1000 + self.epoch)
        if self.weights is None:
            order = g.permutation(self.n)
        else:
            p = np.asarray(self.weights, np.float64); order = g.choice(self.n, self.n, replace=True, p=p / p.sum())
        return iter(order[self.start:].tolist())

    def __len__(self):
        return self.n - self.start


# ---------------- GPU augmentations (batch lists of CHW tensors in [0,1]) ----------------
def _gauss_blur(x, sigma):
    if sigma < 0.3:
        return x
    k = int(2 * round(3 * sigma) + 1)
    t = torch.arange(k, device=x.device, dtype=x.dtype) - k // 2
    g = torch.exp(-t ** 2 / (2 * sigma ** 2)); g = g / g.sum()
    x = F.conv2d(x[None], g.view(1, 1, 1, k).repeat(3, 1, 1, 1), padding=(0, k // 2), groups=3)
    x = F.conv2d(x, g.view(1, 1, k, 1).repeat(3, 1, 1, 1), padding=(k // 2, 0), groups=3)
    return x[0]


def _hue_rotate(x, deg):
    th = math.radians(deg); c, s = math.cos(th), math.sin(th)
    # matrix maths always in fp32 (linalg.inv has no bf16/fp16 kernel), result cast back to the image dtype
    T = torch.tensor([[0.299, 0.587, 0.114], [0.596, -0.274, -0.322], [0.211, -0.523, 0.312]], device=x.device, dtype=torch.float32)
    R = torch.tensor([[1, 0, 0], [0, c, -s], [0, s, c]], device=x.device, dtype=torch.float32)
    M = (torch.linalg.inv(T) @ R @ T).to(x.dtype)
    return torch.einsum("ij,jhw->ihw", M, x)


def scanner_counterfactual(x, strength=1.0):
    """T_phi: one random virtual scanner per image. x: CHW in [0,1]. Always computed in fp32 outside autocast."""
    with torch.autocast(x.device.type, enabled=False):
        return _scanner_counterfactual(x.float(), strength).to(x.dtype)


def _scanner_counterfactual(x, strength=1.0):
    u = lambda a, b: a + (b - a) * random.random()
    s = strength
    x = x.clamp(0, 1) ** u(1 - 0.4 * s, 1 + 0.6 * s)                                   # attenuation curve
    M = torch.eye(3, device=x.device) + 0.15 * s * torch.randn(3, 3, device=x.device)   # pseudo dual-energy response
    M = M / M.sum(1, keepdim=True).clamp(min=0.3)
    x = torch.einsum("ij,jhw->ihw", M, x)
    x = _hue_rotate(x, u(-40, 40) * s)                                                  # colour mapping
    g = x.mean(0, keepdim=True); x = g + (x - g) * u(1 - 0.5 * s, 1 + 0.5 * s)          # saturation
    m = x.mean(); x = (x - m) * u(1 - 0.3 * s, 1 + 0.3 * s) + m + u(-0.08, 0.08) * s      # contrast / brightness
    sat = u(0.85, 1.0); x = x.clamp(0, sat) / sat                                       # detector saturation
    x = _gauss_blur(x, u(0, 1.5 * s))                                                   # focal-spot / sensor blur
    x = x + u(0, 0.04 * s) * torch.sqrt(x.clamp(min=0)) * torch.randn_like(x) + u(0, 0.01) * torch.randn_like(x)
    return x.clamp(0, 1)


def rgb_jitter(x):
    with torch.autocast(x.device.type, enabled=False):
        return _rgb_jitter(x.float()).to(x.dtype)


def _rgb_jitter(x):
    u = lambda a, b: a + (b - a) * random.random()
    g = x.mean(0, keepdim=True); x = g + (x - g) * u(0.7, 1.3)
    m = x.mean(); x = (x - m) * u(0.8, 1.2) + m + u(-0.1, 0.1)
    return _hue_rotate(x, u(-10, 10)).clamp(0, 1)


def fourier_mix(xs, beta=0.05):
    """FDA-style: swap low-frequency amplitude with another image of the batch."""
    out = []
    for i, x in enumerate(xs):
        y = xs[(i + 1) % len(xs)]
        y = F.interpolate(y[None], size=x.shape[-2:], mode="bilinear", align_corners=False)[0]
        fx, fy = torch.fft.fft2(x.float()), torch.fft.fft2(y.float())
        ax, px, ay = fx.abs(), fx.angle(), fy.abs()
        h, w = x.shape[-2:]; b = max(1, int(min(h, w) * beta))
        ax = torch.fft.fftshift(ax, dim=(-2, -1)); ay = torch.fft.fftshift(ay, dim=(-2, -1))
        ch, cw = h // 2, w // 2
        ax[:, ch - b:ch + b, cw - b:cw + b] = ay[:, ch - b:ch + b, cw - b:cw + b]
        ax = torch.fft.ifftshift(ax, dim=(-2, -1))
        out.append(torch.fft.ifft2(ax * torch.exp(1j * px)).real.clamp(0, 1).to(x.dtype))
    return out


def synth_occlusion(xs, targets, p=0.8, frac=(0.2, 0.6), mode="mixed", rng=None):
    """Occlude random parts of GT objects with clutter from other bags. Returns images + visibility maps (1=visible).
    rng: a random.Random for deterministic evaluation-time occlusion (occlusion-ratio sweep)."""
    rng = rng or random
    outs, vis = [], []
    for i, (x, t) in enumerate(zip(xs, targets)):
        x = x.clone(); v = torch.ones(1, *x.shape[-2:], device=x.device)
        src = xs[(i + 1) % len(xs)]
        H, W = x.shape[-2:]
        for b in t["boxes"]:
            if rng.random() > p:
                continue
            x1, y1, x2, y2 = [int(c) for c in b.tolist()]
            bw, bh = max(2, x2 - x1), max(2, y2 - y1)
            area = rng.uniform(*frac) * bw * bh
            ar = rng.uniform(0.3, 3.0)
            ow, oh = int(min(bw, max(2, math.sqrt(area * ar)))), int(min(bh, max(2, math.sqrt(area / ar))))
            ox, oy = rng.randint(x1, max(x1, x2 - ow)), rng.randint(y1, max(y1, y2 - oh))
            ox, oy = min(ox, W - ow), min(oy, H - oh)
            if ow < 2 or oh < 2 or ox < 0 or oy < 0:
                continue
            sh, sw = src.shape[-2:]
            sx, sy = rng.randint(0, max(0, sw - ow)), rng.randint(0, max(0, sh - oh))
            patch = src[:, sy:sy + oh, sx:sx + ow]
            if patch.shape[-2:] != (oh, ow):
                patch = F.interpolate(patch[None], size=(oh, ow), mode="bilinear", align_corners=False)[0]
            if mode == "cutout":
                x[:, oy:oy + oh, ox:ox + ow] = 0.5; v[:, oy:oy + oh, ox:ox + ow] = 0.0
            elif mode == "replace" or (mode == "mixed" and rng.random() < 0.5):
                x[:, oy:oy + oh, ox:ox + ow] = patch; v[:, oy:oy + oh, ox:ox + ow] = 0.0
            else:   # Beer-Lambert: overlapping objects multiply transmitted intensity
                dens = patch * rng.uniform(0.3, 0.9)
                x[:, oy:oy + oh, ox:ox + ow] = x[:, oy:oy + oh, ox:ox + ow] * dens
                v[:, oy:oy + oh, ox:ox + ow] = torch.minimum(v[:, oy:oy + oh, ox:ox + ow], dens.mean(0, keepdim=True))
        outs.append(x); vis.append(v)
    return outs, vis


## 9 · XGuard model
SIRA · OER · STPB · CUS · RCR, baselines, ablation variants.

In [ ]:
# ==============================================================================
# 9. XGUARD MODEL
#    Two-stage detector (ResNet-50-FPN + RPN + RoI head).  NOTE: the proposal names a
#    DINO/Deformable-DETR decoder; on a single Kaggle T4/P100 a DETR-family model does not
#    converge within the session budget, so the five modules are implemented as
#    detector-agnostic plug-ins on RoI (query-equivalent) embeddings. State this in the paper.
#
#    SIRA : Scanner-Invariant Representation Adapter (frequency-conditioned gate on FPN
#           features + counterfactual-scanner consistency)
#    OER  : Occlusion Evidence Router (token reliability r_i, attention bias lambda*log r_i,
#           synthetic evidence removal + consistency + visibility supervision of r_i)
#    STPB : Semantic Threat Prototype Bank (visual + CLIP-text + attribute prototypes,
#           superclass hierarchy, generic threat prototype)
#    CUS  : Counterfactual Unknown Synthesis (shape/material/function factor mixing)
#    RCR  : Risk-Calibrated Referral head  R_i = P(detection incorrect | z_i)
# ==============================================================================
if MODE == "prepare":
    raise StopExecution()   # prepare mode ends after the pack cell
from torchvision.models.detection.backbone_utils import _resnet_fpn_extractor
from torchvision.models.detection.rpn import AnchorGenerator, RPNHead, RegionProposalNetwork
from torchvision.models.detection.transform import GeneralizedRCNNTransform
from torchvision.models.detection.faster_rcnn import TwoMLPHead
from torchvision.models.detection import _utils as det_utils
from torchvision.ops import MultiScaleRoIAlign, box_iou, batched_nms, nms, roi_align
from torchvision.ops.misc import FrozenBatchNorm2d
from torchvision.ops.feature_pyramid_network import FeaturePyramidNetwork, LastLevelMaxPool
from torchvision.models.detection.image_list import ImageList

DINO_R50_URL = "https://dl.fbaipublicfiles.com/dino/dino_resnet50_pretrain/dino_resnet50_pretrain.pth"
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]


def stage_local_weights():
    """If pretrained .pth files were attached as a Kaggle dataset, put them where torch.hub looks (offline use)."""
    dst = Path(os.environ["TORCH_HOME"]) / "hub" / "checkpoints"; dst.mkdir(parents=True, exist_ok=True)
    if PLATFORM != "kaggle":
        return
    for p in glob.glob(os.path.join(INPUT_ROOT, "**", "*.pth"), recursive=True)[:50]:
        n = os.path.basename(p)
        if any(s in n for s in ["fasterrcnn", "resnet50", "dino_resnet50", "fcos", "retinanet"]) and not (dst / n).exists():
            try:
                os.symlink(p, dst / n)
            except Exception:
                shutil.copy(p, dst / n)


stage_local_weights()


def default_rpn():
    anchor = AnchorGenerator(((32,), (64,), (128,), (256,), (512,)), ((0.5, 1.0, 2.0),) * 5)
    return RegionProposalNetwork(anchor, RPNHead(256, 3), 0.7, 0.3, 256, 0.5, dict(training=2000, testing=1000),
                                 dict(training=2000, testing=1000), 0.7)


class ClipRN50FPN(nn.Module):
    """CLIP-family encoder (open_clip RN50 visual tower) + FPN."""
    def __init__(self, tl):
        super().__init__()
        import open_clip
        m, _, _ = open_clip.create_model_and_transforms("RN50", pretrained="openai")
        self.v = m.visual.float()
        del self.v.attnpool
        layers = ["layer4", "layer3", "layer2", "layer1"][:tl]
        for n, p in self.v.named_parameters():
            p.requires_grad_(any(n.startswith(l) for l in layers))
        self.fpn = FeaturePyramidNetwork([256, 512, 1024, 2048], 256, extra_blocks=LastLevelMaxPool())
        self.out_channels = 256

    def forward(self, x):
        v = self.v
        if hasattr(v, "stem"):
            x = v.stem(x)
        else:
            x = v.act1(v.bn1(v.conv1(x))); x = v.act2(v.bn2(v.conv2(x))); x = v.act3(v.bn3(v.conv3(x))); x = v.avgpool(x)
        c2 = v.layer1(x); c3 = v.layer2(c2); c4 = v.layer3(c3); c5 = v.layer4(c4)
        return self.fpn(OrderedDict([("0", c2), ("1", c3), ("2", c4), ("3", c5)]))


def build_backbone_rpn(init, tl):
    from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
    from torchvision.models import resnet50, ResNet50_Weights
    order = [init] + [x for x in ["imagenet", "none"] if x != init]
    for ini in order:
        if ini != "none" and not INTERNET and not any(Path(os.environ["TORCH_HOME"]).rglob("*.pth")):
            print(f"[init] no internet and no local weights -> skip '{ini}'"); continue
        try:
            if ini == "coco":
                m = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1, trainable_backbone_layers=tl)
                return m.backbone, m.rpn, ini
            if ini == "clip":
                return ClipRN50FPN(tl), default_rpn(), ini
            r = resnet50(weights=None, norm_layer=FrozenBatchNorm2d if ini != "none" else nn.BatchNorm2d)
            if ini == "imagenet":
                r.load_state_dict(ResNet50_Weights.IMAGENET1K_V1.get_state_dict(progress=False), strict=False)
            elif ini == "dino":
                r.load_state_dict(torch.hub.load_state_dict_from_url(DINO_R50_URL, map_location="cpu"), strict=False)
            return _resnet_fpn_extractor(r, tl if ini != "none" else 5), default_rpn(), ini
        except Exception as e:
            print(f"[init] '{ini}' failed ({type(e).__name__}: {str(e)[:120]}) -> fallback")
    raise RuntimeError("could not build backbone")


# ---------------------------- text prototypes (CLIP text encoder) ----------------------------
# CLIP text embeddings are computed ONCE in the prepare step (CPU) and shipped inside the pack: GPU sessions never load CLIP
_TEXT_CACHE = PACK_LOCAL / "clip_text.pt" if (PACK_LOCAL / "clip_text.pt").exists() else WORK / "logs" / "clip_text.pt"
TEXT_INFO = dict(source=None, transformers=None, available=False, embedding_sha=None, error=None)


def _as_text_tensor(e):
    """CLIPModel.get_text_features returns a Tensor in most transformers versions and a ModelOutput in some.
    Never use `a or b` on tensors (ambiguous truth value)."""
    if torch.is_tensor(e):
        return e
    for name in ("text_embeds", "pooler_output"):
        v = getattr(e, name, None)
        if v is not None:
            return v
    return e[0]


def clip_text_embed(prompts):
    raw = torch.load(_TEXT_CACHE) if _TEXT_CACHE.exists() else {}
    cache = valid_text_cache(raw)
    bad = len([k for k in raw if k != "__meta__"]) - len(cache)
    if bad:
        print(f"[text] {bad} malformed cached CLIP embeddings (old pack) -> recomputing once")
    todo = [p for p in prompts if p not in cache]
    if todo:
        src = C.CLIP_HF_NAME if INTERNET or os.path.isdir(C.CLIP_HF_NAME) else None
        if src is None:
            TEXT_INFO["error"] = "no internet and no local CLIP folder"
            return None
        try:
            import transformers
            from transformers import CLIPModel, CLIPTokenizer
            tok, m = CLIPTokenizer.from_pretrained(src), CLIPModel.from_pretrained(src).eval()
            with torch.no_grad():
                # recompute EVERY prompt the cache should hold, so the repaired cache is complete
                full = sorted(set(todo) | set(all_text_prompts()))
                cache.update(zip(full, clip_encode_text(m, tok, full)))
            meta = dict(model=src, transformers=transformers.__version__, n=len(cache), repaired=bool(bad))
            torch.save({**cache, "__meta__": meta}, _TEXT_CACHE)
            try:                                   # write the repaired cache back into the pack on Drive
                if PACK_DRIVE.exists() and PACK_DRIVE != _TEXT_CACHE.parent:
                    torch.save({**cache, "__meta__": meta}, PACK_DRIVE / "clip_text.pt")
            except Exception:
                pass
            TEXT_INFO.update(source=src, transformers=transformers.__version__)
            del m
        except Exception as e:
            TEXT_INFO["error"] = f"{type(e).__name__}: {str(e)[:200]}"
            print("[text] CLIP text encoder unavailable:", TEXT_INFO["error"]); return None
    out = torch.stack([cache[p] for p in prompts])
    TEXT_INFO.update(cache_meta=raw.get("__meta__")) if isinstance(raw.get("__meta__"), dict) else None
    TEXT_INFO.update(available=True, embedding_sha=sha(out.numpy().round(5).tolist()), source=TEXT_INFO["source"] or C.CLIP_HF_NAME + " (cache)")
    return out


PROMPTS = TEXT_PROMPTS


def text_prototypes(known):
    ps = [t.format(TEXT_NAME.get(c, c.replace("_", " "))) for c in known for t in PROMPTS]
    thr = THREAT_PROMPTS
    e = clip_text_embed(ps + thr)
    if e is None:
        return None
    assert e.dim() == 2, f"CLIP text embeddings must be (prompts x dim), got {tuple(e.shape)}"
    cls = F.normalize(e[:len(ps)].view(len(known), len(PROMPTS), -1).mean(1), dim=-1)
    return dict(cls=cls, threat=F.normalize(e[len(ps):].mean(0), dim=-1))


# ---------------------------- modules ----------------------------
class SIRAAdapter(nn.Module):
    """Frequency-conditioned scanner invariance: F_L = LP(F), F_H = F - F_L,
    G = sigmoid(MLP([F_L, F_H])), F_inv = G*F_H + (1-G)*F_L;  out = F + gamma*proj(F_inv)."""
    def __init__(self, ch=256):
        super().__init__()
        self.gate = nn.Sequential(nn.Conv2d(2 * ch, ch // 4, 1), nn.GELU(), nn.Conv2d(ch // 4, ch, 1))
        self.proj = nn.Conv2d(ch, ch, 1)
        nn.init.zeros_(self.proj.weight); nn.init.zeros_(self.proj.bias)
        self.gamma = nn.Parameter(torch.tensor(1.0))
        self.last_gate = None

    def forward(self, x):
        L = F.avg_pool2d(x, 5, 1, 2, count_include_pad=False)
        H = x - L
        G = torch.sigmoid(self.gate(torch.cat([L, H], 1)))
        self.last_gate = G.detach().mean().item() if not self.training else None
        return x + self.gamma * self.proj(G * H + (1 - G) * L)


class OER(nn.Module):
    """Occlusion Evidence Router on the 7x7 RoI token grid: A(q,k_i) = q.k_i/sqrt(d) + lambda*log(r_i+eps)."""
    def __init__(self, ch=256, heads=4, n_tok=49):
        super().__init__()
        self.h = heads
        self.rel = nn.Sequential(nn.Linear(ch, 64), nn.GELU(), nn.Linear(64, 1))
        self.norm = nn.LayerNorm(ch)
        self.qkv = nn.Linear(ch, 3 * ch)
        self.proj = nn.Linear(ch, ch)
        self.pos = nn.Parameter(torch.zeros(1, n_tok, ch))
        self.lam = nn.Parameter(torch.tensor(1.0))
        self.gamma = nn.Parameter(torch.tensor(0.0))   # identity at initialisation: OER cannot hurt the pretrained head at step 0

    def forward(self, x):
        N, Cc, H, W = x.shape
        t = x.flatten(2).transpose(1, 2)                                   # N, HW, C
        r = torch.sigmoid(self.rel(t).float())                             # N, HW, 1  evidence reliability
        q, k, v = self.qkv(self.norm(t + self.pos)).chunk(3, -1)
        sh = lambda a: a.view(N, H * W, self.h, Cc // self.h).transpose(1, 2)
        q, k, v = sh(q), sh(k), sh(v)
        att = (q @ k.transpose(-1, -2)).float() / math.sqrt(Cc // self.h)
        att = att + self.lam * torch.log(r + 1e-4).transpose(1, 2).unsqueeze(1)   # bias per key token
        o = (att.softmax(-1).to(v.dtype) @ v).transpose(1, 2).reshape(N, H * W, Cc)
        t = t + self.gamma * self.proj(o)
        return t.transpose(1, 2).reshape(N, Cc, H, W), r.squeeze(-1)


def pair_iou(a, b):
    lt, rb = torch.max(a[:, :2], b[:, :2]), torch.min(a[:, 2:], b[:, 2:])
    inter = (rb - lt).clamp(min=0).prod(1)
    area = lambda x: (x[:, 2] - x[:, 0]).clamp(min=0) * (x[:, 3] - x[:, 1]).clamp(min=0)
    return inter / (area(a) + area(b) - inter).clamp(min=1e-6)


def bce_prob(p, t):
    """BCE on probabilities, autocast-safe."""
    p = p.float().clamp(1e-4, 1 - 1e-4)
    return -(t * p.log() + (1 - t) * (1 - p).log()).mean()


def mlp(i, o, h=128):
    return nn.Sequential(nn.Linear(i, h), nn.GELU(), nn.Linear(h, h), nn.GELU(), nn.Linear(h, o))


STPB_ALIASES = {"full": "vta", "text": "t", "attr": "a", "visual": "v"}


class STPB(nn.Module):
    """Prototype bank. mode = "none" (plain linear classifier) or any subset of "v" (visual), "t" (CLIP text),
    "a" (attribute) -> p_c = normalize(sum_k w_k p_c^k), w = softmax(learned). The superclass (hierarchy) loss is
    applied for every non-"none" mode so that the prototype-source ablation only varies the sources."""
    def __init__(self, known, D, mode, text=None):
        super().__init__()
        K, NA = len(known), len(ATTRS)
        mode = STPB_ALIASES.get(mode, mode)
        self.mode = mode
        A = torch.tensor(np.stack([attr_vector(c) for c in known]))
        self.register_buffer("A", A)
        sup = sorted({superclass(c) for c in known})
        M = torch.zeros(len(sup), K)
        for i, c in enumerate(known):
            M[sup.index(superclass(c)), i] = 1
        self.register_buffer("Msup", M / M.sum(1, keepdim=True).clamp(min=1))
        self.register_buffer("sup_of", torch.tensor([sup.index(superclass(c)) for c in known]))
        self.n_sup = len(sup)
        self.attr_head = nn.Linear(D, NA)
        self.threat = nn.Parameter(F.normalize(torch.randn(D), dim=0))
        self.register_buffer("ema", torch.zeros(K, D)); self.register_buffer("ema_n", torch.zeros(K))
        self.text_available = text is not None
        self.requested = set(mode) if mode != "none" else set()
        self.use_text = "t" in self.requested and text is not None
        self.use_attr = "a" in self.requested
        self.use_vis = "v" in self.requested or (self.requested and not (self.use_text or self.use_attr))
        if mode == "none":
            self.lin = nn.Linear(D, K)
            return
        self.P_V = nn.Parameter(F.normalize(torch.randn(K, D), dim=1))
        if self.use_text:
            self.register_buffer("T", text["cls"].float()); self.register_buffer("T_thr", text["threat"].float())
            self.text_proj = nn.Linear(self.T.shape[1], D)
        if self.use_attr:
            self.attr_emb = nn.Parameter(torch.randn(NA, D) * 0.1)
        self.mix = nn.Parameter(torch.zeros(3))
        self.log_scale = nn.Parameter(torch.tensor(math.log(16.0)))

    def sources(self):
        return "".join(c for c, u in [("v", self.use_vis), ("t", self.use_text), ("a", self.use_attr)] if u) or "linear"

    def prototypes(self):
        if self.mode == "none":
            return F.normalize(self.lin.weight, dim=1)
        comps = []
        if self.use_vis:
            comps.append(F.normalize(self.P_V, dim=1))
        if self.use_text:
            comps.append(F.normalize(self.text_proj(self.T), dim=1))
        if self.use_attr:
            An = (self.A + 1e-3) / (self.A + 1e-3).sum(1, keepdim=True)
            comps.append(F.normalize(An @ self.attr_emb, dim=1))
        w = self.mix[:len(comps)].softmax(0)
        return F.normalize(sum(wi * ci for wi, ci in zip(w, comps)), dim=1)

    def mix_weights(self):
        names = [n for n, u in [("visual", self.use_vis), ("text", self.use_text), ("attribute", self.use_attr)] if u]
        return dict(zip(names, self.mix[:len(names)].softmax(0).tolist())) if self.mode != "none" else {}

    def threat_proto(self):
        t = F.normalize(self.threat, dim=0)
        if self.mode != "none" and self.use_text:
            t = t + F.normalize(self.text_proj(self.T_thr[None])[0], dim=0)
        return F.normalize(t, dim=0)

    def logits(self, z_raw, P=None):
        if self.mode == "none":
            return self.lin(z_raw)
        P = self.prototypes() if P is None else P
        return self.log_scale.exp().clamp(max=50) * F.normalize(z_raw, dim=-1) @ P.t().to(z_raw.dtype)

    def sup_logits(self, z_raw, P):
        return self.log_scale.exp().clamp(max=50) * F.normalize(z_raw.float(), dim=-1) @ F.normalize(self.Msup @ P.float(), dim=1).t()


def different_class_pairs(y, n):
    """n index pairs (i, j) with y[i] != y[j] GUARANTEED; None if fewer than 2 classes are present."""
    cls = y.unique()
    if len(cls) < 2:
        return None
    i = torch.randint(0, len(y), (n,), device=y.device)
    ci = torch.searchsorted(cls, y[i])
    cj = cls[(ci + torch.randint(1, len(cls), (n,), device=y.device)) % len(cls)]
    j = torch.empty_like(i)
    for c in cls:
        m = cj == c
        if m.any():
            members = (y == c).nonzero().squeeze(1)
            j[m] = members[torch.randint(0, len(members), (int(m.sum()),), device=y.device)]
    assert bool((y[i] != y[j]).all())
    return i, j


class CUS(nn.Module):
    """Counterfactual Unknown Synthesis: z -> (s: shape, m: material, g: function/geometry);
    each factor supervised by its own attribute group; z_U = Q([s_A, m_B, lam*g_A+(1-lam)*g_B])."""
    def __init__(self, D, d=64):
        super().__init__()
        self.d = d
        self.P, self.Q = nn.Linear(D, 3 * d), nn.Linear(3 * d, D)
        self.gidx = [[ATTRS.index(a) for a in g] for g in ATTR_GROUPS.values()]
        self.heads = nn.ModuleList([nn.Linear(d, len(g)) for g in self.gidx])

    def losses(self, z, A_rows):
        h = self.P(z); parts = h.split(self.d, 1)
        rec = (1 - F.cosine_similarity(self.Q(h), z.detach(), dim=-1)).mean()
        att = sum(F.binary_cross_entropy_with_logits(hd(p), A_rows[:, gi]) for hd, p, gi in zip(self.heads, parts, self.gidx))
        return rec, att / 3

    def synth(self, z, y, n):
        """Different-class factor mixing; returns None when the batch holds < 2 classes."""
        pr = different_class_pairs(y, n)
        if pr is None:
            return None
        i, j = pr
        s, m, g = self.P(z).split(self.d, 1)
        lam = torch.rand(len(i), 1, device=z.device)
        h = torch.cat([s[i], m[j], lam * g[i] + (1 - lam) * g[j]], 1)
        return self.Q(h)


class XGuard(nn.Module):
    def __init__(self, known, mcfg, init="coco", tl=3, img_size=640, D=256):
        super().__init__()
        self.known, self.K, self.m = list(known), len(known), dict(mcfg)
        self.backbone, self.rpn, self.init_used = build_backbone_rpn(init, tl)
        self.transform = GeneralizedRCNNTransform(img_size, img_size, IMAGENET_MEAN, IMAGENET_STD)
        self.sira = SIRAAdapter(256) if self.m["sira_freq"] else None
        self.roi_pool = MultiScaleRoIAlign(["0", "1", "2", "3"], 7, 2)
        self.oer = OER(256) if self.m["oer"] else None
        self.box_head = TwoMLPHead(256 * 49, 1024)
        self.reg, self.bg, self.embed = nn.Linear(1024, 4), nn.Linear(1024, 1), nn.Linear(1024, D)
        mode = STPB_ALIASES.get(self.m["stpb"], self.m["stpb"])
        text = text_prototypes(self.known) if "t" in mode and mode != "none" and not FORCE_NO_TEXT else None
        if "t" in mode and mode != "none" and text is None and C.REQUIRE_TEXT and not FORCE_NO_TEXT:
            raise RuntimeError(f"CLIP text prototypes required for STPB '{mode}' but unavailable: {TEXT_INFO['error']}")
        self.stpb = STPB(self.known, D, mode, text)
        self.diag = defaultdict(list)
        self.cus = CUS(D) if self.m["unk_gen"] == "cus" else None
        self.unk_head = mlp(D + 2, 1) if self.m["unk_gen"] != "none" else None
        self.unk_skipped = 0
        self.risk_head = mlp(D + 4, 1) if self.m["rcr"] else None
        self.box_coder = det_utils.BoxCoder((10.0, 10.0, 5.0, 5.0))
        self.matcher = det_utils.Matcher(0.5, 0.5, False)
        self.sampler = det_utils.BalancedPositiveNegativeSampler(512, 0.25)
        self.tau_U_formula = None

    # ---------------- shared pieces ----------------
    def features(self, xs, targets=None):
        il, tt = self.transform(list(xs), targets)
        f = self.backbone(il.tensors)
        if self.sira is not None:
            f = OrderedDict((k, self.sira(v)) for k, v in f.items())
        return il, tt, f

    def head(self, feats, boxes, sizes):
        x = self.roi_pool(feats, boxes, sizes)
        r = None
        if self.oer is not None:
            x, r = self.oer(x)
        h = self.box_head(x)
        z = self.embed(h)
        return dict(h=h, z=z, fg=self.stpb.logits(z), bgl=self.bg(h), reg=self.reg(h), r=r)

    def open_scores(self, out, P=None):
        probs = torch.cat([out["bgl"], out["fg"]], 1).float().softmax(1)
        o = 1 - probs[:, 0]
        pfg = probs[:, 1:]
        zn = F.normalize(out["z"].float(), dim=-1)
        P = self.stpb.prototypes().float() if P is None else P.float()
        s_k = (zn @ P.t()).max(1).values.add(1).div(2)
        s_t = (zn @ self.stpb.threat_proto().float()).add(1).div(2)
        if self.unk_head is not None:
            u = torch.sigmoid(self.unk_head(torch.cat([zn, s_k[:, None], s_t[:, None]], 1).to(out["z"].dtype)).float()).squeeze(1)
            U = o * u
        elif self.m["stpb"] != "none":
            U = o * s_t * (1 - s_k)                       # U_i = o_i * s_t * (1 - s_k)
        else:
            cond = pfg / pfg.sum(1, keepdim=True).clamp(min=1e-6)
            U = o * (1 - cond.max(1).values)              # objectness x MSP baseline
        ent = -(pfg.clamp(min=1e-8) * pfg.clamp(min=1e-8).log()).sum(1)
        return probs, o, s_k, s_t, U, ent

    def energy_entropy(self, out):
        """Standard uncertainty baselines: energy = -logsumexp(known logits); normalised entropy of p(known|fg)."""
        fg = out["fg"].float()
        E = -torch.logsumexp(fg, 1)
        q = fg.softmax(1)
        H = -(q * q.clamp(min=1e-8).log()).sum(1) / math.log(max(2, fg.shape[1]))
        return E, H

    def risk(self, z, conf, U, ent, area):
        x = torch.cat([F.normalize(z.float(), dim=-1), conf[:, None], U[:, None], ent[:, None], area[:, None]], 1)
        return self.risk_head(x.to(z.dtype)).float().squeeze(1)

    def sample(self, proposals, targets):
        P, L, RT, MG = [], [], [], []
        for p, t in zip(proposals, targets):
            gt, gl = t["boxes"], t["labels"]
            p = torch.cat([p, gt]) if len(gt) else p
            if len(gt) == 0:
                lab = torch.zeros(len(p), dtype=torch.int64, device=p.device); mg = torch.zeros_like(p)
            else:
                idx = self.matcher(box_iou(gt, p)); cl = idx.clamp(min=0)
                lab = gl[cl] + 1
                lab[idx == self.matcher.BELOW_LOW_THRESHOLD] = 0
                lab[idx == self.matcher.BETWEEN_THRESHOLDS] = -1
                mg = gt[cl]
            pos, neg = self.sampler([lab])
            keep = torch.where(pos[0] | neg[0])[0]
            P.append(p[keep]); L.append(lab[keep]); MG.append(mg[keep])
        RT = self.box_coder.encode(MG, P)
        return P, torch.cat(L), torch.cat(RT), torch.cat(MG)

    # ---------------- training ----------------
    def forward_train(self, xs, targets, xs2=None, vis2=None):
        W = C.LOSS_W
        B = len(xs)
        two = xs2 is not None
        tg = [dict(boxes=t["boxes"], labels=t["labels"]) for t in targets]
        il, tt, feats = self.features(list(xs) + (list(xs2) if two else []), tg + ([dict(t) for t in tg] if two else []))
        # detection loss on view 1, and on view 2 only if view 2 is NOT occluded (occluded boxes are noisy labels)
        det_on_v2 = two and C.VIEW2_DET and not self.m.get("occ")
        nd = 2 * B if det_on_v2 else B
        if nd < len(tt):
            il_d = ImageList(il.tensors[:nd], il.image_sizes[:nd])
            feats_d = OrderedDict((k, v[:nd]) for k, v in feats.items())
            tt_d = tt[:nd]
        else:
            il_d, feats_d, tt_d = il, feats, tt
        proposals, rpn_l = self.rpn(il_d, feats_d, tt_d)
        props, lab, rt, mg = self.sample(proposals, tt_d)
        out = self.head(feats_d, props, il_d.image_sizes)
        logits = torch.cat([out["bgl"], out["fg"]], 1).float()
        L = dict(rpn_obj=rpn_l["loss_objectness"], rpn_box=rpn_l["loss_rpn_box_reg"])
        L["cls"] = F.cross_entropy(logits, lab)
        pos = lab > 0
        L["box"] = F.smooth_l1_loss(out["reg"][pos].float(), rt[pos].float(), beta=1 / 9, reduction="sum") / max(1, lab.numel())
        z, y = out["z"].float(), lab - 1
        zp, yp = z[pos], y[pos]
        st = self.stpb
        P = st.prototypes().float()
        # ---------- STPB ----------
        if self.m["stpb"] != "none" and pos.any():
            L["proto"] = F.binary_cross_entropy_with_logits(st.attr_head(zp), st.A[yp])
            thr = st.threat_proto().float()
            zb = F.normalize(z[lab == 0][:256], dim=-1)
            L["threat"] = (1 - F.normalize(zp, dim=-1) @ thr).mean() + (F.relu(zb @ thr)).mean()
            if st.n_sup > 1:
                L["proto"] = L["proto"] + F.cross_entropy(st.sup_logits(zp, P).float(), st.sup_of[yp])
            with torch.no_grad():
                for c in yp.unique():
                    zc = F.normalize(zp[yp == c], dim=-1).mean(0)
                    st.ema[c] = F.normalize(0.99 * st.ema[c] + 0.01 * zc if st.ema_n[c] > 0 else zc, dim=0)
                    st.ema_n[c] += 1
            if st.use_vis:
                seen = st.ema_n > 0
                L["proto"] = L["proto"] + (1 - (F.normalize(st.P_V[seen], dim=1) * st.ema[seen]).sum(1)).mean()
        # ---------- unknown discovery: cus | gaussian | mixup | slerp ----------
        gen = {"random": "gaussian"}.get(self.m["unk_gen"], self.m["unk_gen"])
        if self.unk_head is not None and pos.sum() >= 2:
            n1 = int((torch.cat([torch.full((len(p),), i) for i, p in enumerate(props)]) < B)[pos.cpu()].sum())
            zr, yr = zp[:max(2, n1)], yp[:max(2, n1)]
            n = min(256, 2 * len(zr))
            zu = None
            if gen == "cus":
                rec, att = self.cus.losses(zr, st.A[yr])
                L["rec"] = rec; L["proto"] = L.get("proto", 0) + att
                zu = self.cus.synth(zr, yr, n)
            elif gen == "gaussian":   # Gaussian feature noise around known embeddings
                zu = F.normalize(zr, dim=-1)[torch.randint(0, len(zr), (n,), device=zr.device)] + 0.5 * torch.randn(n, zr.shape[1], device=zr.device) / math.sqrt(zr.shape[1])   # noise norm ~0.5
            elif gen in ("mixup", "slerp"):
                pr = different_class_pairs(yr, n)
                if pr is not None:
                    a_, b_ = F.normalize(zr[pr[0]], dim=-1), F.normalize(zr[pr[1]], dim=-1)
                    lam = torch.rand(len(pr[0]), 1, device=zr.device) * 0.4 + 0.3
                    if gen == "mixup":
                        zu = lam * a_ + (1 - lam) * b_
                    else:             # manifold-aware: spherical interpolation stays on the unit hypersphere
                        om = torch.acos((a_ * b_).sum(1, keepdim=True).clamp(-0.9999, 0.9999))
                        zu = (torch.sin((1 - lam) * om) * a_ + torch.sin(lam * om) * b_) / torch.sin(om).clamp(min=1e-4)
            elif gen == "pseudo":   # ORE / OW-DETR-style: high-objectness proposals that match no label = real unknown candidates  # @fp-neutral
                pu = []  # @fp-neutral
                for i_ in range(B):  # @fp-neutral
                    pr_, gb_ = proposals[i_][: C.PSEUDO_TOPK], tt_d[i_]["boxes"]  # proposals are sorted by objectness  # @fp-neutral
                    if len(pr_) and len(gb_):  # @fp-neutral
                        pr_ = pr_[box_iou(pr_, gb_).max(1).values < 0.3]  # @fp-neutral
                    pu.append(pr_[: C.PSEUDO_PER_IMG].detach())  # @fp-neutral
                if sum(len(p_) for p_ in pu) >= 2:  # @fp-neutral
                    zu = self.head(feats_d, pu, il_d.image_sizes)["z"]  # @fp-neutral
            elif gen == "vos":   # VOS-style: class-conditional Gaussians (shared covariance), lowest-likelihood samples  # @fp-neutral
                q_ = getattr(self, "_vos_q", None) or {}  # @fp-neutral
                with torch.no_grad(), torch.autocast(zr.device.type, enabled=False):  # @fp-neutral
                    zf_ = F.normalize(zr.detach().float(), dim=-1)  # @fp-neutral
                    for c_ in yr.unique().tolist():  # @fp-neutral
                        q_[c_] = torch.cat([q_.get(c_, zf_[:0]), zf_[yr == c_]])[-C.VOS_QUEUE:]  # @fp-neutral
                    self._vos_q = q_  # @fp-neutral
                    if len(q_) >= 2 and min(len(v_) for v_ in q_.values()) >= C.VOS_MIN:  # @fp-neutral
                        mus_ = {c_: v_.mean(0) for c_, v_ in q_.items()}  # @fp-neutral
                        X_ = torch.cat([v_ - mus_[c_] for c_, v_ in q_.items()])  # @fp-neutral
                        cov_ = X_.t() @ X_ / len(X_) + 1e-4 * torch.eye(X_.shape[1], device=X_.device)  # @fp-neutral
                        mvn_ = [torch.distributions.MultivariateNormal(mus_[c_], covariance_matrix=cov_) for c_ in q_]  # @fp-neutral
                        cand_ = torch.cat([m_.sample((C.VOS_SAMPLES,)) for m_ in mvn_])  # @fp-neutral
                        lp_ = torch.stack([m_.log_prob(cand_) for m_ in mvn_], 1).max(1).values  # @fp-neutral
                        zu = cand_[lp_.argsort()[:n]]  # @fp-neutral
            if zu is None or len(zu) == 0:
                self.unk_skipped += 1
            else:
                zun, zrn = F.normalize(zu.float(), dim=-1), F.normalize(zr.float(), dim=-1)
                thr = st.threat_proto().float()
                sk_r, st_r = (zrn @ P.t()).max(1).values.float(), (zrn @ thr).float()
                sk_u, st_u = (zun @ P.t()).max(1).values.float(), (zun @ thr).float()
                if gen in ("cus", "slerp"):   # keep threat-like counterfactuals unlike every known prototype
                    keep = (sk_u < torch.quantile(sk_r.detach().float(), 0.25)) & (st_u > torch.quantile(st_r.detach().float(), 0.10))
                    if keep.sum() < 4:
                        keep = sk_u <= sk_u.median()
                    zun, sk_u, st_u = zun[keep], sk_u[keep], st_u[keep]
                if len(zun) == 0:
                    self.unk_skipped += 1
                else:
                    self._last_synth = zun.detach()
                    self.diag["synth_max_known_cos"].append(float(sk_u.mean()))
                    self.diag["real_max_known_cos"].append(float(sk_r.mean()))
                    f_u = torch.cat([zun, ((sk_u + 1) / 2)[:, None], ((st_u + 1) / 2)[:, None]], 1)
                    f_r = torch.cat([zrn, ((sk_r + 1) / 2)[:, None], ((st_r + 1) / 2)[:, None]], 1)
                    lu, lr = self.unk_head(f_u.to(out["z"].dtype)).float(), self.unk_head(f_r.to(out["z"].dtype)).float()
                    L["unknown"] = F.binary_cross_entropy_with_logits(lu, torch.ones_like(lu)) + \
                        F.binary_cross_entropy_with_logits(lr, torch.zeros_like(lr))
                    lk = st.logits(zun.to(out["z"].dtype)).float()
                    L["ent"] = -(lk.log_softmax(1)).mean()   # push synthetic unknowns away from every known class
        # ---------- invariance / occlusion consistency on GT RoIs ----------
        if two:
            gtb = [t["boxes"] for t in tt]
            has = sum(len(b) for b in gtb[:B]) > 0
            if self.m["sira_cf"]:
                w_lvl = []
                for k in ["0", "1", "2", "3"]:
                    f1, f2 = feats[k][:B].detach().float(), feats[k][B:].float()
                    cs = F.cosine_similarity(f1, f2, dim=1)[:, None]
                    if vis2 is not None:
                        v = F.interpolate(self._pad_vis(vis2, il), size=cs.shape[-2:], mode="bilinear", align_corners=False)
                        w_lvl.append(((1 - cs) * v).sum() / v.sum().clamp(min=1))
                    else:
                        w_lvl.append((1 - cs).mean())
                L["sira"] = sum(w_lvl) / len(w_lvl)
            if has:
                o_gt = self.head(feats, gtb, il.image_sizes)
                n1 = sum(len(b) for b in gtb[:B])
                z1, z2 = o_gt["z"][:n1].float(), o_gt["z"][n1:].float()
                L["cons_obj"] = (1 - F.cosine_similarity(z1.detach(), z2, dim=-1)).mean()
                p1 = torch.cat([o_gt["bgl"], o_gt["fg"]], 1)[:n1].float().softmax(1).detach()
                lp2 = torch.cat([o_gt["bgl"], o_gt["fg"]], 1)[n1:].float().log_softmax(1)
                L["cons"] = F.kl_div(lp2, p1, reduction="batchmean")
                if self.oer is not None and vis2 is not None and o_gt["r"] is not None:
                    vp = self._pad_vis(vis2, il)
                    vr = roi_align(vp, gtb[B:], output_size=7, spatial_scale=1.0, sampling_ratio=2, aligned=True).flatten(1)
                    r1, r2 = o_gt["r"][:n1], o_gt["r"][n1:]
                    L["rel"] = bce_prob(r2, vr.clamp(0, 1)) + bce_prob(r1, torch.ones_like(r1).float())
                    with torch.no_grad():
                        occl, visi = vr < 0.5, vr >= 0.5
                        if occl.any():
                            self.diag["oer_r_occluded"].append(float(r2[occl].float().mean()))
                        if visi.any():
                            self.diag["oer_r_visible"].append(float(r2[visi].float().mean()))
        # ---------- RCR: learn P(detection incorrect) ----------
        if self.risk_head is not None:
            with torch.no_grad():
                probs, o, s_k, s_t, U, ent = self.open_scores({k: v.detach() if torch.is_tensor(v) else v for k, v in out.items()}, P.detach())
                conf, pc = probs[:, 1:].max(1)
                db = self.box_coder.decode(out["reg"].detach().float(), props).reshape(-1, 4)
                iou = pair_iou(db, mg)
                correct = (lab > 0) & (pc == lab - 1) & (iou >= 0.5)
                pb = torch.cat(props)
                area = ((pb[:, 2] - pb[:, 0]) * (pb[:, 3] - pb[:, 1])).clamp(min=1).log() / 14
            rl = self.risk(out["z"].detach(), conf, U, ent, area)
            tgt = (~correct).float()
            pw = (tgt.numel() - tgt.sum()).clamp(min=1) / tgt.sum().clamp(min=1)
            L["risk"] = F.binary_cross_entropy_with_logits(rl, tgt, weight=torch.where(tgt > 0, 1.0, pw.clamp(max=20)).detach())
        return L

    def _pad_vis(self, vis, il):
        Hp, Wp = il.tensors.shape[-2:]
        out = torch.zeros(len(vis), 1, Hp, Wp, device=il.tensors.device)
        for i, (v, (h, w)) in enumerate(zip(vis, il.image_sizes[len(il.image_sizes) - len(vis):])):
            if v.shape[-2:] != (h, w):
                v = F.interpolate(v[None], size=(h, w), mode="bilinear", align_corners=False)[0]
            out[i, :, :h, :w] = v
        return out

    # ---------------- inference ----------------
    @torch.no_grad()
    def forward(self, xs):
        sizes0 = [x.shape[-2:] for x in xs]
        il, _, feats = self.features(xs)
        props, _ = self.rpn(il, feats)
        out = self.head(feats, props, il.image_sizes)
        probs, o, s_k, s_t, U, ent = self.open_scores(out)
        boxes = self.box_coder.decode(out["reg"].float(), props).reshape(-1, 4)
        pfg = probs[:, 1:]
        conf = pfg.max(1).values
        pb = torch.cat(props)
        area = ((pb[:, 2] - pb[:, 0]) * (pb[:, 3] - pb[:, 1])).clamp(min=1).log() / 14
        R = torch.sigmoid(self.risk(out["z"], conf, U, ent, area)) if self.risk_head is not None else 1 - conf
        E, Hn = self.energy_entropy(out)
        res, start = [], 0
        for i, p in enumerate(props):
            n = len(p); sl = slice(start, start + n); start += n
            (h, w), (h0, w0) = il.image_sizes[i], sizes0[i]
            b = boxes[sl].clone()
            b[:, 0::2] = b[:, 0::2].clamp(0, w); b[:, 1::2] = b[:, 1::2].clamp(0, h)
            b = b * torch.tensor([w0 / w, h0 / h, w0 / w, h0 / h], device=b.device)
            ok = ((b[:, 2] - b[:, 0]) > 1) & ((b[:, 3] - b[:, 1]) > 1)
            sc = pfg[sl]
            roi, cls = torch.where((sc > C.SCORE_THR) & ok[:, None])
            s = sc[roi, cls]
            k = batched_nms(b[roi], s, cls, 0.5)[: C.DETS_PER_IMG]
            roi, cls, s = roi[k], cls[k], s[k]
            u = U[sl]
            ru = torch.where((u > C.SCORE_THR) & ok)[0]
            ku = nms(b[ru], u[ru], 0.5)[: C.UNK_DETS_PER_IMG]
            ru = ru[ku]
            res.append(dict(boxes=b[roi].cpu(), scores=s.cpu(), labels=cls.cpu(), U=u[roi].cpu(), R=R[sl][roi].cpu(),
                            E=E[sl][roi].cpu(), H=Hn[sl][roi].cpu(),
                            ub=b[ru].cpu(), us=u[ru].cpu(), uR=R[sl][ru].cpu(), uE=E[sl][ru].cpu(), uH=Hn[sl][ru].cpu()))
        return res

    @torch.no_grad()
    def embed_boxes(self, xs, boxes):
        il, _, feats = self.features(xs)
        sc = [torch.tensor([w1 / w0, h1 / h0, w1 / w0, h1 / h0], device=b.device) for b, x, (h1, w1) in
              zip(boxes, xs, il.image_sizes) for (h0, w0) in [x.shape[-2:]]]
        out = self.head(feats, [b * s for b, s in zip(boxes, sc)], il.image_sizes)
        return out["z"].float()


class TVDetector(nn.Module):
    """Closed-set torchvision baselines (FCOS / RetinaNet / Faster R-CNN-v1) under the same pipeline.
    Unknown score = 1 - max score (MSP-style), risk = 1 - score."""
    def __init__(self, kind, known, init, tl, img_size):
        super().__init__()
        from torchvision.models.detection import fcos_resnet50_fpn, retinanet_resnet50_fpn
        from torchvision.models import ResNet50_Weights
        self.known, self.K, self.m = list(known), len(known), dict(kind=kind)
        wb = ResNet50_Weights.IMAGENET1K_V1 if (init != "none" and INTERNET) else None
        builder = dict(fcos=fcos_resnet50_fpn, retinanet=retinanet_resnet50_fpn)[kind]
        try:
            self.net = builder(weights=None, weights_backbone=wb, num_classes=self.K + 1,
                               trainable_backbone_layers=tl if wb else 5, min_size=img_size, max_size=img_size)
            self.init_used = "imagenet" if wb else "none"
        except Exception as e:
            print("[init] backbone weights failed:", e)
            self.net = builder(weights=None, weights_backbone=None, num_classes=self.K + 1, min_size=img_size, max_size=img_size)
            self.init_used = "none"
        self.net.score_thresh = C.SCORE_THR
        self.net.detections_per_img = C.DETS_PER_IMG

    def forward_train(self, xs, targets, **kw):
        L = self.net(list(xs), [dict(boxes=t["boxes"], labels=t["labels"] + 1) for t in targets])
        return {k: v for k, v in L.items()}

    @torch.no_grad()
    def forward(self, xs):
        res = []
        for d in self.net(list(xs)):
            keep = d["labels"] > 0
            b, s, l = d["boxes"][keep].cpu(), d["scores"][keep].cpu(), d["labels"][keep].cpu() - 1
            nan = torch.full_like(s, float("nan"))
            res.append(dict(boxes=b, scores=s, labels=l, U=1 - s, R=1 - s, E=nan, H=nan, ub=b, us=1 - s, uR=1 - s, uE=nan, uH=nan))
        return res


FORCE_NO_TEXT = False   # set while re-loading a model that was trained without text prototypes

_B = dict(sira_freq=0, sira_cf=0, oer=0, occ=0, stpb="none", unk_gen="none", rcr=0, aug="none")
_S, _O, _P, _U, _R = dict(sira_freq=1, sira_cf=1), dict(oer=1, occ=1), dict(stpb="vta"), dict(unk_gen="cus"), dict(rcr=1)


def _v(*parts):
    d = dict(_B)
    for p_ in parts:
        d.update(p_)
    return d


MODEL_VARIANTS = {   # component ablation matrix (review: independent + cumulative + leave-one-out)
    "base": _v(), "sira": _v(_S), "oer": _v(_O), "stpb": _v(_P),
    "sira+oer": _v(_S, _O), "sira+oer+stpb": _v(_S, _O, _P), "sira+oer+stpb+cus": _v(_S, _O, _P, _U),
    "xguard": _v(_S, _O, _P, _U, _R),
    "xguard-sira": _v(_O, _P, _U, _R), "xguard-oer": _v(_S, _P, _U, _R),
    "xguard-cus": _v(_S, _O, _P, _R), "xguard-rcr": _v(_S, _O, _P, _U),
}


def variant(base="xguard", **kw):
    d = dict(MODEL_VARIANTS[base]); d.update(kw); return d


def build_model(spec, known):
    kind = spec.get("kind", "xguard")
    if kind in ("fcos", "retinanet"):
        return TVDetector(kind, known, spec.get("init", C.INIT), spec.get("tl", C.TRAINABLE_LAYERS), C.IMG)
    return XGuard(known, spec["mcfg"], spec.get("init", C.INIT), spec.get("tl", C.TRAINABLE_LAYERS), C.IMG, C.EMB_DIM)


def count_params(m):
    return sum(p.numel() for p in m.parameters()) / 1e6


## 10 · Training & inference
Exact resume, accumulation tail, NaN policy, source-val checkpoint selection.

In [ ]:
# ==============================================================================
# 10. TRAINING + INFERENCE (reproducible & resumable)
#   * full checkpoint (model, optimizer, scaler, epoch, step, python/numpy/torch/cuda RNG) -> exact resume,
#     the deterministic per-epoch sampler restarts mid-epoch at the saved position
#   * gradient accumulation: the incomplete last window of an epoch is applied, not dropped
#   * non-finite losses are counted; the run FAILS if they exceed C.MAX_NAN_FRAC (no silent recovery)
#   * the checkpoint is selected on the SOURCE validation split (harmonized AP50); never on target data
#   * artifacts: checkpoint_best.pt, checkpoint_last.pt (fp16 weights), train_log.csv, validation_metrics.json
# ==============================================================================
if MODE == "prepare":
    raise StopExecution()   # prepare mode ends after the pack cell
class TimeUp(Exception):
    pass


_FORCE_TIMEUP_AT = None   # used only by the resume test


def loss_weight(k, m, override=None):
    W = dict(C.LOSS_W); W.update(override or {})
    table = dict(sira=W["sira"], cons_obj=W["occ"] if m.get("occ") else W["sira"], rel=W["rel"], proto=W["proto"],
                 threat=W["threat"], unknown=W["unknown"], ent=W["ent"], cons=W["cons"], risk=W["risk"], rec=W["rec"])
    return table.get(k, 1.0)


def prepare_views(xs, ts, m):
    aug = m.get("aug", "none")
    if aug == "rgb":
        xs = [rgb_jitter(x) for x in xs]
    elif aug == "fourier" and len(xs) > 1:
        xs = [a if random.random() < 0.5 else b for a, b in zip(xs, fourier_mix(xs))]
    elif aug == "cutout":
        xs, _ = synth_occlusion(xs, ts, p=C.OCC_P, frac=C.OCC_FRAC, mode="cutout")
    elif aug == "occmix":   # control: XGuard's occlusion corruption as plain augmentation  # @fp-neutral
        xs, _ = synth_occlusion(xs, ts, p=C.OCC_P, frac=C.OCC_FRAC, mode="mixed")  # @fp-neutral
    if not (m.get("sira_cf") or m.get("occ")):
        return xs, None, None
    x2 = [scanner_counterfactual(x) for x in xs] if m.get("sira_cf") else [x.clone() for x in xs]
    vis = None
    if m.get("occ"):
        x2, vis = synth_occlusion(x2, ts, p=C.OCC_P, frac=C.OCC_FRAC, mode="mixed")
    return xs, x2, vis


def make_loader(recs, cls2id, train, seed=0, epoch=0, start=0, bs=None):
    ds = XrayDet(recs, cls2id, C.IMG, train, aug_seed=seed * 1000 + epoch)
    sampler = ResumableSampler(len(ds), seed, epoch, start) if train else None
    kw = dict(persistent_workers=False, prefetch_factor=4) if C.WORKERS > 0 else {}
    return DataLoader(ds, batch_size=bs or C.BATCH, sampler=sampler, shuffle=False, num_workers=C.WORKERS,
                      collate_fn=collate, pin_memory=DEVICE.type == "cuda", drop_last=False, **kw)


def save_weights(model, path):
    if Path(path).name == "checkpoint_last.pt":     # redundant with the resume checkpoint -> not written (saves Drive space)
        return
    sd = {k: (v.half() if v.is_floating_point() else v) for k, v in model.state_dict().items()}
    atomic_write(path, lambda t: torch.save(sd, t))


def check_resume_checkpoint(run_dir):
    """a resume checkpoint written while Drive was full / during a disconnect may be truncated -> delete it so the run
    restarts cleanly instead of crashing on every Run all."""
    ck = Path(run_dir) / "checkpoint_last_full.pt"
    if ck.exists():
        try:
            torch.load(ck, map_location="cpu", weights_only=False)
        except Exception as e:
            print(f"   ⚠️  resume checkpoint unreadable ({type(e).__name__}) -> deleted, this run restarts from epoch 0")
            ck.unlink(missing_ok=True)


def load_weights(spec, known, path, meta):
    sp = dict(spec); sp["init"] = "clip" if spec.get("init") == "clip" else "none"
    global INTERNET, FORCE_NO_TEXT
    net, INTERNET = INTERNET, (INTERNET if sp["init"] == "clip" else False)
    FORCE_NO_TEXT = meta.get("text_available") is False
    try:
        model = build_model(sp, known)
    finally:
        INTERNET, FORCE_NO_TEXT = net, False
    sd = torch.load(path, map_location="cpu")
    sd = {k: v.float() if v.is_floating_point() else v for k, v in sd.items()}
    missing, unexpected = model.load_state_dict(sd, strict=False)
    bad = [k for k in missing if "num_batches_tracked" not in k]
    if bad or unexpected:
        raise RuntimeError(f"checkpoint/architecture mismatch: missing={bad[:5]} unexpected={unexpected[:5]}")
    return model.to(DEVICE).eval()


def validate(model, val_recs, known):
    """Source-validation score for checkpoint selection: harmonized AP50 (target data is never used)."""
    if not val_recs:
        return float("nan")
    p, _ = infer(model, val_recs)
    m, _ = evaluate(p, val_recs, known, dict(tau_known=0.5, tau_U=0.5, tau_R=0.5, bag=None), light=True)
    model.train()
    return float(m.get("AP50", float("nan")))


def train_model(spec, run_dir, train_recs, known, val_recs, epochs):
    run_dir.mkdir(parents=True, exist_ok=True)
    meta_p = run_dir / "train_meta.json"
    if (run_dir / "checkpoint_best.pt").exists() and meta_p.exists():
        meta = jload(meta_p)
        return load_weights(spec, known, run_dir / "checkpoint_best.pt", meta), meta
    seed = spec.get("seed", 0); seed_everything(seed)
    cls2id = {c: i for i, c in enumerate(known)}
    model = build_model(spec, known).to(DEVICE)
    m = spec.get("mcfg", {})
    params = [p for p in model.parameters() if p.requires_grad]
    base_lr = C.LR * (C.BATCH * C.ACCUM) / 8 * spec.get("lr_mult", 1.0)
    opt = torch.optim.SGD(params, lr=base_lr, momentum=0.9, weight_decay=C.WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
    ipe = math.ceil(len(train_recs) / C.BATCH)
    if C.TEST_ITERS:
        ipe, epochs = min(ipe, C.TEST_ITERS), 1
    total = max(1, epochs * math.ceil(ipe / C.ACCUM))
    warm = min(C.WARMUP_ITERS, max(1, total // 5))
    ep0, it0, gstep, log, vlog, best, nan_total, steps_total = 0, 0, 0, [], [], -1.0, 0, 0
    ck = run_dir / "checkpoint_last_full.pt"
    if ck.exists():
        s = torch.load(ck, map_location="cpu", weights_only=False)
        model.load_state_dict(s["model"]); opt.load_state_dict(s["opt"]); scaler.load_state_dict(s["scaler"])
        ep0, it0, gstep, log, vlog, best = s["epoch"], s["it"], s["gstep"], s["log"], s["vlog"], s["best"]
        nan_total, steps_total = s["nan_total"], s["steps_total"]
        set_rng_state(s["rng"])
        print(f"   resumed from epoch {ep0} iter {it0} (best val AP50 so far {best:.4f})")

    def lr_at(g):
        if g < warm:
            return base_lr * (0.001 + 0.999 * g / warm)
        p = (g - warm) / max(1, total - warm)
        return base_lr * (0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * min(1.0, p))))

    def checkpoint(ep, it):
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), scaler=scaler.state_dict(), epoch=ep, it=it,
                        gstep=gstep, log=log, vlog=vlog, best=best, nan_total=nan_total, steps_total=steps_total,
                        rng=rng_state()), ck)

    def step():
        nonlocal gstep
        for g in opt.param_groups:
            g["lr"] = lr_at(gstep)
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, C.GRAD_CLIP)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); gstep += 1

    t0 = time.time()
    steady_s, steady_n, val_s, WARM = 0.0, 0, [], 10   # speed for the cost estimate: steady-state steps only
    model.train()
    lw = spec.get("loss_w")
    for ep in range(ep0, epochs):
        loader = make_loader(train_recs, cls2id, True, seed, ep, it0 * C.BATCH)
        _rs = torch.get_rng_state(); batches = iter(loader); torch.set_rng_state(_rs)   # loader must not consume the training RNG
        run_l, pending = defaultdict(float), 0
        last_bi = it0 - 1
        t_prev = time.time()
        for bi, (xs, ts) in enumerate(batches, start=it0):
            if bi >= ipe:
                break
            last_bi = bi
            xs = [x.to(DEVICE, non_blocking=True) for x in xs]
            ts = [dict(boxes=t["boxes"].to(DEVICE), labels=t["labels"].to(DEVICE)) for t in ts]
            x1, x2, vis = prepare_views(xs, ts, m)
            with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=C.AMP):
                L = model.forward_train(x1, ts, xs2=x2, vis2=vis)
            loss = sum(loss_weight(k, m, lw) * v.float() for k, v in L.items() if torch.is_tensor(v))
            steps_total += 1
            if not torch.isfinite(loss):
                nan_total += 1; opt.zero_grad(set_to_none=True); pending = 0
                if nan_total > max(20, C.MAX_NAN_FRAC * steps_total):
                    raise RuntimeError(f"{nan_total}/{steps_total} non-finite training steps -> run FAILED (not reported)")
                continue
            scaler.scale(loss / C.ACCUM).backward(); pending += 1
            if pending == C.ACCUM:
                step(); pending = 0
            for k, v in L.items():
                if torch.is_tensor(v):
                    run_l[k] += float(v)
            run_l["total"] += float(loss); run_l["_n"] += 1          # float() synchronises -> wall time is real GPU time
            _now = time.time()
            if steps_total > WARM:
                steady_s += _now - t_prev; steady_n += 1
            t_prev = _now
            if (bi + 1) % 50 == 0 or bi + 1 == ipe:
                n = max(1, run_l.pop("_n"))
                msg = {k: round(v / n, 4) for k, v in run_l.items()}
                diag = {k: round(float(np.mean(v)), 4) for k, v in getattr(model, "diag", {}).items() if v}
                if hasattr(model, "diag"):
                    model.diag.clear()
                log.append(dict(epoch=ep, it=bi + 1, lr=opt.param_groups[0]["lr"], nan_total=nan_total,
                                unk_skipped=getattr(model, "unk_skipped", 0), **msg, **diag))
                print(f"   ep {ep + 1}/{epochs} it {bi + 1}/{ipe} lr {opt.param_groups[0]['lr']:.4f} " +
                      " ".join(f"{k}={v:.3f}" for k, v in msg.items()) +
                      (" | " + " ".join(f"{k}={v:.3f}" for k, v in diag.items()) if diag else "") +
                      f"  [{(time.time() - t0) / 60:.1f} min]")
                run_l = defaultdict(float)
            if time_left_h() < 0.3 or (_FORCE_TIMEUP_AT is not None and steps_total == _FORCE_TIMEUP_AT):
                if pending:
                    step(); pending = 0
                checkpoint(ep, bi + 1); raise TimeUp(f"session time nearly over: checkpoint saved at epoch {ep} it {bi + 1}")
        if pending:                       # apply the incomplete accumulation window
            step(); pending = 0
        it0 = 0
        if (ep + 1) % C.VAL_EVERY == 0 or ep + 1 == epochs:
            _tv = time.time(); v = validate(model, val_recs, known); val_s.append(time.time() - _tv)
            vlog.append(dict(epoch=ep + 1, val_AP50=v, minutes=(time.time() - t0) / 60))
            print(f"   epoch {ep + 1}: source-val AP50 = {v:.4f}" + ("  (best)" if v > best else ""))
            if v > best or math.isnan(best):
                best = v; save_weights(model, run_dir / "checkpoint_best.pt")
        save_weights(model, run_dir / "checkpoint_last.pt")
        checkpoint(ep + 1, 0)
    if not (run_dir / "checkpoint_best.pt").exists():
        save_weights(model, run_dir / "checkpoint_best.pt")
    pd.DataFrame(log).to_csv(run_dir / "train_log.csv", index=False)
    jdump(vlog, run_dir / "validation_metrics.json")
    meta = dict(init_used=getattr(model, "init_used", None), params_M=count_params(model), train_images=len(train_recs),
                train_minutes=(time.time() - t0) / 60, epochs=epochs, iters_per_epoch=ipe, img=C.IMG, spec=spec,
                text_available=getattr(getattr(model, "stpb", None), "text_available", None),
                prototype_sources=model.stpb.sources() if hasattr(model, "stpb") else None,
                prototype_mix=model.stpb.mix_weights() if hasattr(model, "stpb") else {},
                text_info=dict(TEXT_INFO), nan_steps=nan_total, steps=steps_total,
                unk_skipped=getattr(model, "unk_skipped", 0), best_val_AP50=best,
                sec_per_iter=(steady_s / steady_n) if steady_n >= 5 else (time.time() - t0) / max(1, steps_total),
                val_sec_per_epoch=float(np.mean(val_s)) if val_s else None)
    jdump(meta, meta_p)
    if not C.KEEP_FULL_CKPT and ck.exists():
        ck.unlink()
    return load_weights(spec, known, run_dir / "checkpoint_best.pt", meta), meta


@torch.no_grad()
def infer(model, recs, bs=None, occ_ratio=0.0):
    """occ_ratio > 0: deterministic synthetic occlusion of every GT object (evaluation-time occlusion sweep)."""
    model.eval()
    allc = {c: i for i, c in enumerate(sorted({c for r in recs for c in r["cls"]}))} if occ_ratio > 0 else {}
    loader = make_loader(recs, allc, False, bs=bs or max(2, 2 * C.BATCH))   # inference needs no gradients: bigger batches
    preds, t_model, n = [None] * len(recs), 0.0, 0
    for bi, (xs, ts) in enumerate(loader):
        xs = [x.to(DEVICE, non_blocking=True) for x in xs]
        if occ_ratio > 0:
            xs, _ = synth_occlusion(xs, [dict(boxes=t["boxes"].to(DEVICE)) for t in ts], p=1.0, frac=(occ_ratio, occ_ratio),
                                    mode="mixed", rng=random.Random(7919 * (bi + 1)))
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        t = time.time()
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=C.AMP):
            outs = model(xs)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        t_model += time.time() - t; n += len(xs)
        for o, tg in zip(outs, ts):
            s = tg["scale"]
            preds[tg["idx"]] = {k: (v.float().numpy() / (s if k in ("boxes", "ub") else 1.0)).astype(np.float32)
                                if k != "labels" else v.numpy().astype(np.int64) for k, v in o.items()}
    return preds, dict(ms_per_img=1000 * t_model / max(1, n), fps=n / max(1e-6, t_model))


def model_flops(model):
    try:
        from torch.utils.flop_counter import FlopCounterMode
        x = [torch.rand(3, C.IMG, C.IMG, device=DEVICE)]
        fc = FlopCounterMode(display=False)
        with fc, torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=C.AMP):
            model(x)
        return fc.get_total_flops() / 2e9   # GMACs-style convention (FLOPs/2)
    except Exception:
        return float("nan")


## 11 · Evaluation
Three AP tracks, open-world metrics, reliability, bag-level referral, metric status.

In [ ]:
# ==============================================================================
# 11. EVALUATION
#   closed-set (3 tracks, never mixed): HARMONIZED AP (primary: AP, AP50, AP75, AR100, APs/m/l),
#                                       EXACT AP (xAP, xAP50), HIERARCHICAL superclass AP (hAP, hAP50)
#   open-world : U_AP, U_AP50, U_R50 (one-to-one matching, top-50 unknown dets), U_R50_op (at tau_U),
#                AOSE, WI (at the source-val operating point)
#                locAUROC / locFPR95: known-vs-unknown separability of U for LOCALIZED objects only
#                (reported with locCoverage = fraction of GT objects that were localized at all)
#   reliability: ECE / Brier / NLL; AURC + E-AURC for risk scores {msp, entropy, energy, U, rcr}
#   operational: bag-level referral at fixed budgets (thresholds from SOURCE val): referral rate,
#                missed threats after referral, false alarms per bag after referral
#   every block reports metric_status = ok | not_applicable:<reason> | evaluator_failure:<error>
# ==============================================================================
if MODE == "prepare":
    raise StopExecution()   # prepare mode ends after the pack cell
from sklearn.metrics import roc_auc_score, roc_curve

RISK_TYPES = ["msp", "entropy", "energy", "U", "rcr"]


def np_iou(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), np.float32)
    lt = np.maximum(a[:, None, :2], b[None, :, :2]); rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.clip(rb - lt, 0, None).prod(2)
    ar = lambda x: (x[:, 2] - x[:, 0]) * (x[:, 3] - x[:, 1])
    return inter / np.maximum(ar(a)[:, None] + ar(b)[None] - inter, 1e-6)


def greedy_match(db, ds, gb, thr=0.5):
    """one-to-one greedy matching (highest score first). returns tp flags per det and matched gt index."""
    n = len(db); tp = np.zeros(n, bool); gi = -np.ones(n, int)
    if n == 0 or len(gb) == 0:
        return tp, gi
    order = np.argsort(-ds, kind="mergesort"); iou = np_iou(db[order], gb); used = np.zeros(len(gb), bool)
    for r, k in enumerate(order):
        c = np.where(used, -1, iou[r]); j = int(c.argmax())
        if c[j] >= thr:
            tp[k], gi[k], used[j] = True, j, True
    return tp, gi


def ap_from(scores, tp, n_gt, w=None):
    """COCO-style 101-point interpolated AP; optional per-detection weights (bootstrap)."""
    if n_gt <= 0:
        return np.nan
    if len(scores) == 0:
        return 0.0
    o = np.argsort(-scores, kind="mergesort"); tp = tp[o].astype(np.float64)
    w = np.ones_like(tp) if w is None else w[o]
    ctp, cfp = np.cumsum(w * tp), np.cumsum(w * (1 - tp))
    rec, prec = ctp / n_gt, ctp / np.maximum(ctp + cfp, 1e-9)
    prec = np.maximum.accumulate(prec[::-1])[::-1]
    rs = np.linspace(0, 1, 101); idx = np.searchsorted(rec, rs, side="left")
    return float(np.mean([prec[i] if i < len(prec) else 0 for i in idx]))


COCO_KEYS = ["AP", "AP50", "AP75", "APs", "APm", "APl", "AR1", "AR10", "AR100", "ARs", "ARm", "ARl"]


def coco_eval(gts, dets, cats):
    """gts: {img: [(box, cat)]}, dets: [(img, box, score, cat)] -> (stats, per-class AP50)."""
    if not cats or sum(len(v) for v in gts.values()) == 0:
        return {k: np.nan for k in COCO_KEYS}, {}
    cid = {c: i + 1 for i, c in enumerate(cats)}
    if HAVE_COCO:
        images, anns = [], []
        for im, g in gts.items():
            images.append(dict(id=im, width=1, height=1))
            for b, c in g:
                if c in cid:
                    anns.append(dict(id=len(anns) + 1, image_id=im, category_id=cid[c],
                                     bbox=[float(b[0]), float(b[1]), float(b[2] - b[0]), float(b[3] - b[1])],
                                     area=float((b[2] - b[0]) * (b[3] - b[1])), iscrowd=0))
        with contextlib.redirect_stdout(io.StringIO()):
            gt = COCO(); gt.dataset = dict(images=images, annotations=anns, categories=[dict(id=v, name=k) for k, v in cid.items()]); gt.createIndex()
            dl = [dict(image_id=im, category_id=cid[c], bbox=[float(b[0]), float(b[1]), float(b[2] - b[0]), float(b[3] - b[1])], score=float(s))
                  for im, b, s, c in dets if c in cid and im in gts]
            if not dl:
                return {k: 0.0 for k in COCO_KEYS}, {c: 0.0 for c in cats}
            dt = gt.loadRes(dl)
            E = COCOeval(gt, dt, "bbox"); E.params.imgIds = list(gts.keys()); E.evaluate(); E.accumulate(); E.summarize()
        stats = {k: float(v) if v >= 0 else np.nan for k, v in zip(COCO_KEYS, E.stats)}
        pc = {}
        for c, i in cid.items():
            p = E.eval["precision"][0, :, i - 1, 0, 2]; p = p[p > -1]
            pc[c] = float(p.mean()) if len(p) else np.nan
        return stats, pc
    res = {k: np.nan for k in COCO_KEYS}; pc = {}; aps = defaultdict(list)
    for c in cats:
        for t in np.arange(0.5, 0.96, 0.05):
            S, TP, ng = [], [], 0
            for im, g in gts.items():
                gb = np.array([b for b, cc in g if cc == c]).reshape(-1, 4); ng += len(gb)
                d = [(b, s) for i2, b, s, cc in dets if i2 == im and cc == c]
                if d:
                    db = np.array([x[0] for x in d]); dsc = np.array([x[1] for x in d])
                    tp, _ = greedy_match(db, dsc, gb, t); S += list(dsc); TP += list(tp)
            aps[round(t, 2)].append(ap_from(np.array(S), np.array(TP, bool), ng))
        pc[c] = aps[0.5][-1]
    res["AP"] = float(np.nanmean([np.nanmean(v) for v in aps.values()])); res["AP50"] = float(np.nanmean(aps[0.5])); res["AP75"] = float(np.nanmean(aps[0.75]))
    return res, pc


def ece_score(conf, correct, bins=15):
    if len(conf) == 0:
        return np.nan
    e, edges = 0.0, np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(conf[m].mean() - correct[m].mean())
    return float(e)


def risk_coverage(conf, correct):
    """detection-level selective risk; returns (AURC, E-AURC, curve on a 100-point coverage grid)."""
    if len(conf) == 0:
        return np.nan, np.nan, None
    o = np.argsort(-conf, kind="mergesort"); err = np.cumsum(~correct[o]) / np.arange(1, len(o) + 1)
    cov = np.arange(1, len(o) + 1) / len(o)
    aurc = float(err.mean())
    srt = np.sort(~correct)                                        # perfect ranking: all correct detections first
    opt = float((np.cumsum(srt) / np.arange(1, len(srt) + 1)).mean())   # exact (discrete) optimal AURC
    curve = np.interp(np.linspace(0.01, 1, 100), cov, err)
    return aurc, float(aurc - opt), curve


def det_risk(p, kind):
    """per-known-detection risk (higher = more likely wrong)."""
    if kind == "msp":
        return 1 - p["scores"]
    if kind == "entropy":
        return p.get("H", np.full(len(p["scores"]), np.nan))
    if kind == "energy":
        return p.get("E", np.full(len(p["scores"]), np.nan))
    if kind == "U":
        return p["U"]
    return p["R"]


def build_eval_items(preds, recs, known, subset=None):
    items = []
    for i, r in enumerate(recs):
        if r.get("excluded") or preds[i] is None:
            continue
        if subset and subset != "all" and subset not in r["subsets"]:
            continue
        kb, kc, xb, xc, ub = [], [], [], [], []
        for b, c in zip(r["boxes"], r["cls"]):
            hk = resolve_known(c, known, "harmonized")
            if hk is None:
                ub.append(b)
            else:
                kb.append(b); kc.append(hk)
            if c in known:
                xb.append(b); xc.append(c)
        p = preds[i]
        names = np.array([known[l] for l in p["labels"]], dtype=object)
        tp = np.zeros(len(names), bool)
        kbn, kcn = np.array(kb, np.float32).reshape(-1, 4), np.array(kc, dtype=object)
        for c in set(names.tolist()):
            di, gi = np.where(names == c)[0], np.where(kcn == c)[0]
            t, _ = greedy_match(p["boxes"][di], p["scores"][di], kbn[gi], 0.5); tp[di] = t
        ubn = np.array(ub, np.float32).reshape(-1, 4)
        hit_unk = np_iou(p["boxes"], ubn).max(1) >= 0.5 if len(ubn) and len(names) else np.zeros(len(names), bool)
        items.append(dict(i=i, r=r, p=p, names=names, tp=tp, hit_unk=hit_unk, kb=kbn, kc=kcn,
                          xb=np.array(xb, np.float32).reshape(-1, 4), xc=np.array(xc, dtype=object), ub=ubn,
                          all_cls=list(r["cls"]), all_b=r["boxes"], neg=bool(r["neg"])))
    return items


def _block(M, name, fn):
    try:
        reason = fn()
        M["metric_status"][name] = "ok" if reason is None else f"not_applicable:{reason}"
    except Exception as e:
        M["metric_status"][name] = f"evaluator_failure:{type(e).__name__}: {str(e)[:120]}"


def bag_risk(p, kind):
    m = p["scores"] >= 0.05
    if not m.any():
        return 0.0
    v = det_risk(p, kind)[m]
    v = v[np.isfinite(v)]
    return float(v.max()) if len(v) else np.nan


def evaluate(preds, recs, known, thr, subset=None, fps=None, light=False):
    items = build_eval_items(preds, recs, known, subset)
    M = dict(n_img=len(items), n_known_gt=int(sum(len(it["kb"]) for it in items)),
             n_exact_gt=int(sum(len(it["xb"]) for it in items)), n_unk_gt=int(sum(len(it["ub"]) for it in items)),
             n_neg_img=int(sum(it["neg"] for it in items)), metric_status={})
    curves = {}
    if not items:
        M["metric_status"]["all"] = "not_applicable:no_images"
        return M, curves
    evcats = order_classes({c for it in items for c in it["kc"]})
    M["eval_classes"] = evcats
    dets = [(it["i"], b, s, c) for it in items for b, s, c in zip(it["p"]["boxes"], it["p"]["scores"], it["names"])]

    def closed():
        if M["n_known_gt"] == 0:
            for k in COCO_KEYS:
                M[k] = np.nan
            return "no_shared_known_gt"
        st, pc = coco_eval({it["i"]: list(zip(it["kb"], it["kc"])) for it in items}, dets, evcats)
        M.update(st); M["per_class_AP50"] = pc
    _block(M, "closed_harmonized", closed)
    if light:
        return M, curves

    def exact():
        xc = order_classes({c for it in items for c in it["xc"]})
        if not xc:
            M["xAP"] = M["xAP50"] = np.nan
            return "no_exact_shared_gt"
        st, _ = coco_eval({it["i"]: list(zip(it["xb"], it["xc"])) for it in items}, dets, xc)
        M["xAP"], M["xAP50"] = st["AP"], st["AP50"]
    _block(M, "closed_exact", exact)

    def hier():
        hg = {it["i"]: [(b, superclass(c)) for b, c in zip(it["all_b"], it["all_cls"])] for it in items}
        hcats = sorted({superclass(c) for c in known} & {c for v in hg.values() for _, c in v} - {"other"})
        M["hier_classes"] = hcats
        if not hcats:
            M["hAP"] = M["hAP50"] = np.nan
            return "no_shared_superclass"
        st, pc = coco_eval(hg, [(i, b, s, superclass(c)) for i, b, s, c in dets], hcats)
        M["hAP"], M["hAP50"], M["per_super_AP50"] = st["AP"], st["AP50"], pc
    _block(M, "closed_hierarchical", hier)

    def unknown():
        if M["n_unk_gt"] == 0:
            for k in ["U_AP", "U_AP50", "U_R50", "U_R50_op"]:
                M[k] = np.nan
            return "no_unknown_gt"
        st, _ = coco_eval({it["i"]: [(b, "unknown") for b in it["ub"]] for it in items},
                          [(it["i"], b, s, "unknown") for it in items for b, s in zip(it["p"]["ub"], it["p"]["us"])], ["unknown"])
        M["U_AP"], M["U_AP50"] = st["AP"], st["AP50"]
        hits = hits_op = 0
        for it in items:
            if len(it["ub"]) and len(it["p"]["ub"]):
                t, _ = greedy_match(it["p"]["ub"], it["p"]["us"], it["ub"], 0.5); hits += int(t.sum())
                m = it["p"]["us"] >= thr["tau_U"]
                if m.any():
                    t, _ = greedy_match(it["p"]["ub"][m], it["p"]["us"][m], it["ub"], 0.5); hits_op += int(t.sum())
        M["U_R50"], M["U_R50_op"] = hits / M["n_unk_gt"], hits_op / M["n_unk_gt"]
    _block(M, "unknown", unknown)

    def loc_auroc():
        ys, us, n_gt = [], [], 0
        for it in items:
            B = np.concatenate([it["p"]["boxes"], it["p"]["ub"]]); Uv = np.concatenate([it["p"]["U"], it["p"]["us"]])
            for gb, lab in [(it["kb"], 0), (it["ub"], 1)]:
                n_gt += len(gb)
                if len(gb) and len(B):
                    for row in np_iou(gb, B):
                        if row.max() >= 0.5:
                            ys.append(lab); us.append(float(Uv[row.argmax()]))
        M["locCoverage"] = len(ys) / max(1, n_gt)
        ys, us = np.array(ys), np.array(us)
        if len(set(ys.tolist())) < 2:
            M["locAUROC"] = M["locFPR95"] = M["locAUROC_flipped"] = np.nan
            return "needs_localized_known_and_unknown"
        assert np.isfinite(us).all()
        M["locAUROC"] = float(roc_auc_score(ys, us)); M["locAUROC_flipped"] = float(roc_auc_score(ys, -us))  # diagnostic only
        fpr, tpr, _ = roc_curve(ys, us); M["locFPR95"] = float(fpr[min(len(fpr) - 1, np.searchsorted(tpr, 0.95))])
    _block(M, "loc_auroc", loc_auroc)

    tk, tu, tr = thr["tau_known"], thr["tau_U"], thr["tau_R"]
    S = np.concatenate([it["p"]["scores"] for it in items]); TP = np.concatenate([it["tp"] for it in items])
    NM = np.concatenate([it["names"] for it in items]); Uall = np.concatenate([it["p"]["U"] for it in items])
    HU = np.concatenate([it["hit_unk"] for it in items])

    def openset_op():
        if M["n_unk_gt"] == 0:
            M["AOSE"] = M["WI"] = np.nan
            return "no_unknown_gt"
        sel = S >= tk
        M["AOSE"] = int((sel & ~TP & HU & (Uall < tu)).sum())
        nk_only = np.concatenate([np.full(len(it["p"]["scores"]), len(it["ub"]) == 0) for it in items])
        P_KU = TP[sel].mean() if sel.any() else np.nan
        P_K = TP[sel & nk_only].mean() if (sel & nk_only).any() else np.nan
        M["WI"] = float(P_K / P_KU - 1) if P_KU and np.isfinite(P_KU) and np.isfinite(P_K) else np.nan
    _block(M, "openset_operating_point", openset_op)

    evset = set(evcats)
    cm = (np.array([n in evset for n in NM], bool) & (S >= 0.05)) if len(NM) else np.zeros(0, bool)

    def reliability():
        if not cm.any():
            return "no_detections_on_shared_classes"
        conf, corr = S[cm], TP[cm]
        M["ECE"] = ece_score(conf, corr)
        M["Brier"] = float(np.mean((conf - corr) ** 2))
        M["NLL"] = float(-np.mean(corr * np.log(np.clip(conf, 1e-6, 1)) + (~corr) * np.log(np.clip(1 - conf, 1e-6, 1))))
        for kind in RISK_TYPES:
            rk = np.concatenate([det_risk(it["p"], kind) for it in items])[cm]
            if not np.isfinite(rk).all():
                M[f"AURC_{kind}"] = M[f"EAURC_{kind}"] = np.nan; continue
            a, e, cv = risk_coverage(-rk, corr)
            M[f"AURC_{kind}"], M[f"EAURC_{kind}"] = a, e
            curves[f"rc_{kind}"] = cv
            if kind == "rcr":
                M["ECE_R"] = ece_score(1 - rk, corr)
        M["AURC"], M["AURC_R"] = M.get("AURC_msp"), M.get("AURC_rcr")
    _block(M, "reliability", reliability)

    def operational():
        fa = miss = ngt = 0
        per_bag = []
        for it in items:
            p = it["p"]
            ab = np.concatenate([p["boxes"][p["scores"] >= tk], p["ub"][p["us"] >= tu]])
            asc = np.concatenate([p["scores"][p["scores"] >= tk], p["us"][p["us"] >= tu]])
            if len(ab):
                ab = ab[nms(torch.from_numpy(ab), torch.from_numpy(asc), 0.5).numpy()]
            G = np.concatenate([it["kb"], it["ub"]]) if (len(it["kb"]) or len(it["ub"])) else np.zeros((0, 4), np.float32)
            iou = np_iou(ab, G)
            f = int((iou.max(1) < 0.5).sum()) if len(G) else len(ab)
            mi = int((iou.max(0) < 0.5).sum()) if len(ab) else len(G)
            fa += f; miss += mi; ngt += len(G)
            per_bag.append(dict(fa=f, miss=mi, ngt=len(G), neg=it["neg"], **{k: bag_risk(p, k) for k in RISK_TYPES}))
        M["FA_per_img"] = fa / len(items); M["miss_rate"] = miss / ngt if ngt else np.nan
        negs = [b for b in per_bag if b["neg"]]
        M["FA_per_clean_bag"] = float(np.mean([b["fa"] for b in negs])) if negs else np.nan
        bag_thr = thr.get("bag") or {}
        for kind in RISK_TYPES:
            for q in C.REFERRAL_BUDGETS:
                t = bag_thr.get(kind, {}).get(str(q))
                key = f"{kind}@{int(q * 100)}"
                if t is None or not np.isfinite(t):
                    M[f"ref_rate_{key}"] = M[f"miss_after_ref_{key}"] = M[f"fa_after_ref_{key}"] = np.nan; continue
                ref = np.array([np.isfinite(b[kind]) and b[kind] > t for b in per_bag])
                M[f"ref_rate_{key}"] = float(ref.mean())
                M[f"miss_after_ref_{key}"] = float(sum(b["miss"] for b, r in zip(per_bag, ref) if not r) / ngt) if ngt else np.nan
                M[f"fa_after_ref_{key}"] = float(np.mean([b["fa"] for b, r in zip(per_bag, ref) if not r])) if (~ref).any() else np.nan
        M["referral_rate"] = M.get("ref_rate_rcr@10", np.nan)
    _block(M, "operational", operational)
    if fps:
        M.update(fps)
    return M, curves


def compute_thresholds(preds, recs, known):
    """All operating points come from the SOURCE validation split (no target labels are ever used)."""
    items = build_eval_items(preds, recs, known)
    thr = dict(tau_known=0.5, tau_U=0.5, tau_R=0.5, bag={})
    if not items:
        return thr
    S = np.concatenate([it["p"]["scores"] for it in items]); TP = np.concatenate([it["tp"] for it in items])
    ngt = sum(len(it["kb"]) for it in items)
    if len(S) and ngt:
        o = np.argsort(-S); ctp = np.cumsum(TP[o]); prec = ctp / np.arange(1, len(o) + 1); rec = ctp / ngt
        f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-9); thr["tau_known"] = float(np.clip(S[o][f1.argmax()], 0.05, 0.95))
    U = np.concatenate([it["p"]["U"] for it in items]); R = np.concatenate([it["p"]["R"] for it in items])
    m = (S >= thr["tau_known"]) & TP
    thr["tau_U"] = max(float(np.quantile(U[m], 0.95)) if m.sum() >= 5 else 0.5, 1e-3)   # 5% of correct known dets flagged
    sel = S >= thr["tau_known"]
    thr["tau_R"] = float(np.quantile(R[sel], 0.90)) if sel.sum() >= 5 else 0.5
    for kind in RISK_TYPES:
        br = np.array([bag_risk(it["p"], kind) for it in items], dtype=float)
        br = br[np.isfinite(br)]
        thr["bag"][kind] = {str(q): (float(np.quantile(br, 1 - q)) if len(br) >= 10 else None) for q in C.REFERRAL_BUDGETS}
    return thr


def paired_bootstrap(pA, pB, recs, known, metric="AP50", n=None, seed=0):
    """95% CI of Delta = metric(B) - metric(A); image-level resampling of the SAME images for both models."""
    n = n or C.BOOTSTRAP
    iA, iB = build_eval_items(pA, recs, known), build_eval_items(pB, recs, known)
    ids = sorted({it["i"] for it in iA} & {it["i"] for it in iB})
    if not ids:
        return dict(delta=np.nan, lo=np.nan, hi=np.nan, p_le_0=np.nan)
    pos = {i: k for k, i in enumerate(ids)}

    def pack(items):
        D = defaultdict(lambda: [[], [], []]); G = defaultdict(lambda: np.zeros(len(ids)))
        for it in items:
            if it["i"] not in pos:
                continue
            k = pos[it["i"]]
            if metric == "U_AP50":
                p = it["p"]
                if len(p["ub"]):
                    t, _ = greedy_match(p["ub"], p["us"], it["ub"], 0.5)
                    D["u"][0] += list(p["us"]); D["u"][1] += list(t); D["u"][2] += [k] * len(t)
                G["u"][k] += len(it["ub"])
            else:
                for s, t, c in zip(it["p"]["scores"], it["tp"], it["names"]):
                    D[c][0].append(s); D[c][1].append(t); D[c][2].append(k)
                for c in it["kc"]:
                    G[c][k] += 1
        return {c: (np.array(v[0]), np.array(v[1], bool), np.array(v[2], int)) for c, v in D.items()}, G

    DA, GA = pack(iA); DB, GB = pack(iB)
    cats = [c for c in GA if GA[c].sum() > 0]
    if not cats:
        return dict(delta=np.nan, lo=np.nan, hi=np.nan, p_le_0=np.nan)

    def score(D, G, w):
        vals = []
        for c in cats:
            ng = float((G[c] * w).sum())
            if ng <= 0:
                continue
            if c in D:
                s, t, im = D[c]; vals.append(ap_from(s, t, ng, w[im]))
            else:
                vals.append(0.0)
        return np.mean(vals) if vals else np.nan

    one = np.ones(len(ids))
    d0 = score(DB, GB, one) - score(DA, GA, one)
    rng = np.random.RandomState(seed); ds = []
    for _ in range(n):
        w = np.bincount(rng.randint(0, len(ids), len(ids)), minlength=len(ids)).astype(float)
        ds.append(score(DB, GB, w) - score(DA, GA, w))
    ds = np.array(ds)
    return dict(delta=float(d0), lo=float(np.nanpercentile(ds, 2.5)), hi=float(np.nanpercentile(ds, 97.5)),
                p_le_0=float(np.nanmean(ds <= 0)))


## 12 · Correctness tests
Metric unit tests (stop on failure), CUS/ontology/CLIP integrity, reproducibility.

In [ ]:
# ==============================================================================
# 12. CORRECTNESS TESTS (run before any experiment; metric failures STOP the notebook)
#   * metric unit tests on hand-made toy cases (AP, one-to-one unknown recall, AUROC, E-AURC, zero-GT)
#   * CUS / pair-sampling integrity, AMP-dtype quantile, ontology integrity, CLIP text integrity
#   * reproducibility (test mode, GPU): same seed twice + interrupted/resumed run vs uninterrupted run
# ==============================================================================
if MODE == "prepare":
    raise StopExecution()   # prepare mode ends after the pack cell
TESTS = OrderedDict()


def _toy_rec(boxes, cls, neg=False, subsets=()):
    return dict(boxes=np.array(boxes, np.float32).reshape(-1, 4), cls=list(cls), neg=neg, subsets=list(subsets), excluded=False)


def _toy_pred(boxes, scores, labels, ub=(), us=(), U=None):
    b = np.array(boxes, np.float32).reshape(-1, 4); s = np.array(scores, np.float32)
    ubb = np.array(ub, np.float32).reshape(-1, 4); uss = np.array(us, np.float32)
    U = np.zeros(len(s), np.float32) if U is None else np.array(U, np.float32)
    return dict(boxes=b, scores=s, labels=np.array(labels, np.int64), U=U, R=1 - s, E=-s, H=1 - s,
                ub=ubb, us=uss, uR=np.zeros(len(uss), np.float32), uE=np.zeros(len(uss), np.float32), uH=np.zeros(len(uss), np.float32))


def run_metric_tests():
    T = {}
    # 1. one-to-one matching: ONE detection covering TWO ground truths may match only one
    tp, gi = greedy_match(np.array([[0, 0, 10, 10]], np.float32), np.array([0.9]), np.array([[0, 0, 10, 10], [0, 0, 10, 9.5]], np.float32))
    T["greedy_one_to_one"] = bool(tp.sum() == 1)
    # 2. AP edge cases
    T["ap_perfect"] = abs(ap_from(np.array([0.9, 0.8]), np.array([True, True]), 2) - 1.0) < 1e-9
    T["ap_no_dets"] = ap_from(np.array([]), np.array([], bool), 3) == 0.0
    T["ap_zero_gt_is_nan"] = bool(np.isnan(ap_from(np.array([0.5]), np.array([False]), 0)))
    # 3. full evaluate() on a toy open-world benchmark (known = knife; gun is unknown)
    known = ["knife"]
    recs = [_toy_rec([[10, 10, 50, 50]], ["knife"]),
            _toy_rec([[100, 100, 150, 150], [100, 100, 150, 148]], ["gun", "gun"]),   # two unknown GTs, near-identical boxes
            _toy_rec([], [], neg=True, subsets=["negative"])]
    preds = [_toy_pred([[10, 10, 50, 50]], [0.9], [0], U=[0.05]),
             _toy_pred([], [], [], ub=[[100, 100, 150, 150]], us=[0.8]),                # ONE unknown detection
             _toy_pred([], [], [])]
    thr = dict(tau_known=0.5, tau_U=0.5, tau_R=0.5, bag={})
    M, _ = evaluate(preds, recs, known, thr)
    T["toy_AP50_is_1"] = abs(M["AP50"] - 1.0) < 1e-6
    T["toy_unknown_recall_one_to_one"] = abs(M["U_R50"] - 0.5) < 1e-9       # 1 det can hit only 1 of 2 GTs
    T["toy_status_ok"] = M["metric_status"]["closed_harmonized"] == "ok" and M["metric_status"]["unknown"] == "ok"
    T["toy_locAUROC_perfect"] = abs(M["locAUROC"] - 1.0) < 1e-9 and abs(M["locAUROC_flipped"]) < 1e-9
    # 4. zero-GT behaviour: only a clean bag -> not_applicable, never a crash / silent zero
    M0, _ = evaluate([preds[2]], [recs[2]], known, thr)
    T["zero_gt_not_applicable"] = M0["metric_status"]["closed_harmonized"].startswith("not_applicable") and \
        M0["metric_status"]["unknown"].startswith("not_applicable")
    # 5. selective risk: perfect ranking -> E-AURC = 0; random -> > 0
    corr = np.array([True] * 8 + [False] * 2)
    a, e, _ = risk_coverage(np.linspace(1, 0, 10), corr)
    T["eaurc_perfect_zero"] = abs(e) < 1e-6
    a2, e2, _ = risk_coverage(np.linspace(0, 1, 10), corr)
    T["eaurc_worst_positive"] = e2 > 0.1
    # 6. different-class pair sampling + CUS empty handling + fp32 quantile on AMP tensors
    y = torch.tensor([0, 0, 1, 1, 2])
    i, j = different_class_pairs(y, 200)
    T["pairs_always_different_class"] = bool((y[i] != y[j]).all())
    T["pairs_single_class_returns_None"] = different_class_pairs(torch.tensor([3, 3, 3]), 10) is None
    cus = CUS(16)
    T["cus_single_class_returns_None"] = cus.synth(torch.randn(4, 16), torch.tensor([1, 1, 1, 1]), 8) is None
    zs = cus.synth(torch.randn(6, 16), torch.tensor([0, 0, 1, 1, 2, 2]), 8)
    T["cus_shape"] = zs is not None and tuple(zs.shape) == (8, 16)
    try:
        torch.quantile(torch.randn(10).to(torch.bfloat16).float(), 0.25); T["quantile_fp32_cast"] = True
    except Exception:
        T["quantile_fp32_cast"] = False
    # 7. ontology integrity
    T["aliases_point_to_canonical"] = all(v in CLASS_ATTR for v in SEM_ALIAS.values()) and not (set(SEM_ALIAS) & set(SEM_ALIAS.values()))
    T["exact_track_ignores_alias"] = resolve_known("folding_knife", ["knife"], "exact") is None and \
        resolve_known("folding_knife", ["knife"], "harmonized") == "knife"
    data_classes = {c for r in ALL_REC for c in r["cls"]}
    T["all_data_classes_have_superclass"] = all(superclass(c) != "other" for c in data_classes)
    return T


TESTS.update(run_metric_tests())
failed = [k for k, v in TESTS.items() if not v]
print("metric / component tests:", "ALL PASSED" if not failed else f"FAILED: {failed}")
if failed:
    jdump(TESTS, WORK / "logs" / "unit_tests.json")
    raise AssertionError(f"correctness tests failed: {failed} -> fix before running experiments")

# CLIP text integrity (needed by STPB 't'): embeddings present, version + hash recorded
_tp = text_prototypes(["knife", "gun"])
TESTS["clip_text_available"] = _tp is not None
if _tp is not None:
    TESTS["clip_text_shapes"] = _tp["cls"].dim() == 2 and _tp["threat"].dim() == 1 and _tp["cls"].shape[1] == _tp["threat"].shape[0]
    assert TESTS["clip_text_shapes"], f"CLIP text prototypes have wrong shapes {tuple(_tp['cls'].shape)} / {tuple(_tp['threat'].shape)}"
# the text path of STPB must work even where no CLIP is available (synthetic 512-d text features)
_st = STPB(["knife", "gun"], 32, "vta", dict(cls=F.normalize(torch.randn(2, 512), dim=-1), threat=F.normalize(torch.randn(512), dim=0)))
_lg = _st.logits(torch.randn(5, 32)); _tp_ = _st.threat_proto()
TESTS["stpb_text_path"] = tuple(_lg.shape) == (5, 2) and tuple(_tp_.shape) == (32,)
assert TESTS["stpb_text_path"], "STPB text path broken"
TESTS["clip_text_info"] = dict(TEXT_INFO)
print("CLIP text prototypes:", "available" if _tp is not None else f"UNAVAILABLE ({TEXT_INFO['error']})")
if C.REQUIRE_TEXT and _tp is None:
    raise RuntimeError("REQUIRE_TEXT=True but CLIP text prototypes are unavailable (Internet off?)")

_FORCE_TIMEUP_AT = None


def reproducibility_test():
    """same seed twice -> parameter difference = hardware nondeterminism floor; interrupted+resumed vs continuous."""
    global _FORCE_TIMEUP_AT
    src = [r for r in REC.get("sixray", next(iter(REC.values()))) if r["split"] == "train" and not r["neg"]][:24]
    val = [r for r in REC.get("sixray", next(iter(REC.values()))) if r["split"] == "val"][:8]
    known = order_classes({c for r in src for c in r["cls"]})
    spec = dict(name="_repro", kind="xguard", mcfg=variant("xguard"), seed=123, init=C.INIT)
    old = (C.TEST_ITERS, C.BATCH)
    C.TEST_ITERS, C.BATCH = 6, 2
    outs = []
    try:
        for tag, stop in [("a", None), ("b", None), ("c", 3)]:
            rd = WORK / "logs" / f"_repro_{tag}"
            shutil.rmtree(rd, ignore_errors=True)
            _FORCE_TIMEUP_AT = stop
            try:
                m, _ = train_model(spec, rd, src, known, val, 1)
            except TimeUp:
                _FORCE_TIMEUP_AT = None
                m, _ = train_model(spec, rd, src, known, val, 1)     # resume from the saved full checkpoint
            outs.append(torch.cat([p.detach().float().flatten().cpu() for p in m.parameters()]))
            shutil.rmtree(rd, ignore_errors=True)
    finally:
        _FORCE_TIMEUP_AT = None
        C.TEST_ITERS, C.BATCH = old
    d_seed = float((outs[0] - outs[1]).abs().max()); d_resume = float((outs[0] - outs[2]).abs().max())
    return dict(same_seed_max_param_diff=d_seed, resume_vs_continuous_max_param_diff=d_resume,
                passed=bool(d_resume <= max(10 * d_seed, 1e-3)))


if MODE == "test" or SMOKE:
    TESTS["reproducibility"] = reproducibility_test()
    print("reproducibility:", TESTS["reproducibility"])
jdump(TESTS, WORK / "logs" / "unit_tests.json")


## 13 · Protocols & experiment registry

In [ ]:
# ==============================================================================
# 13. PROTOCOLS + EXPERIMENT REGISTRY + RUNNER
#   P1 in-domain (SIXray)             P2 SIXray -> PIDray / OPIXray / HiXray (no target data, no target tuning)
#   P3 leave-one-domain-out           P4 leave-one-class-out on SIXray (folds = classes with enough objects)
#   P5 SIXray -> PIDray novel classes (never named)   P6 OPIXray OL1-3 (+ synthetic occlusion sweep)
#   P7 compound: PIDray-hidden (deliberately concealed) novel classes from a SIXray-only model
#   TIERS (budget): 1 = paper-critical, 2 = supporting ablations / baselines, 3 = extra studies
#   Every run folder holds fingerprint.json; a run whose training inputs changed is moved to _stale/ and
#   retrained, a run whose evaluation inputs changed is re-evaluated (weights kept). No stale caches.
# ==============================================================================
if MODE == "prepare":
    raise StopExecution()   # prepare mode ends after the pack cell
AVAILABLE = set(REC)
RUNS_DIR = WORK / ("runs" if MODE in ("full", "report") else f"runs_{MODE}")
RUNS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIRS = [RUNS_DIR]
if MODE == "report":     # + the other platform's finished runs
    for d in EXTRA_RESULT_DIRS:
        for pat in ["runs", "*/runs", "*/*/runs", "*/*/*/runs"]:
            RUNS_DIRS += [Path(x) for x in glob.glob(str(Path(d) / pat)) if Path(x) not in RUNS_DIRS]
    print("result folders merged:", [str(d) for d in RUNS_DIRS])


def all_runs():
    seen, out = set(), []
    for d in RUNS_DIRS:
        for rd in sorted(d.glob("*")):
            if rd.is_dir() and not rd.name.startswith("_") and rd.name not in seen:
                seen.add(rd.name); out.append(rd)
    return out


def run_path(name):
    for d in RUNS_DIRS:
        if (d / name / "DONE").exists():
            return d / name
    return RUNS_DIR / name
SIX_CLASSES = CLASS_AUDIT[(CLASS_AUDIT.dataset == "sixray")].cls.tolist() if len(CLASS_AUDIT) else []
LOCO_CLASSES = CLASS_AUDIT[(CLASS_AUDIT.dataset == "sixray") & CLASS_AUDIT.valid_loco].cls.tolist() if len(CLASS_AUDIT) else []
print("SIXray classes in the data:", SIX_CLASSES, "| valid LOCO folds:", LOCO_CLASSES)
if "hammer" not in SIX_CLASSES and "sixray" in AVAILABLE:
    print("⚠️  this SIXray mirror has NO 'hammer' -> it is a 5-class subset; do not present it as the canonical 6-class SIXray.")


def split_recs(ds, split):
    return [r for r in REC.get(ds, []) if r["split"] == split]


def cap(recs, n, salt=""):
    if n is None or len(recs) <= n:
        return recs
    return sorted(recs, key=lambda r: stable_hash(salt + r["uid"]))[:n]


def train_set(spec):
    out = []
    for ds in spec["train"]:
        rs = split_recs(ds, "train")
        if spec.get("loco"):
            rs = [r for r in rs if spec["loco"] not in r["cls"]]       # every image containing the held-out class is removed
        pos = sorted([r for r in rs if not r["neg"]], key=train_cap_order)
        pos = pos[: C.MAX_TRAIN_PER_DS] if C.MAX_TRAIN_PER_DS else pos
        out += pos + [r for r in rs if r["neg"]][: C.N_NEG_TRAIN]
    known = order_classes({c for r in out for c in r["cls"]})
    return out, known


def val_set(spec):
    rs = []
    n = C.TEST_VAL or C.MAX_VAL
    for ds in spec["train"]:
        v = [r for r in split_recs(ds, "val") if not r["neg"] and not r.get("excluded")]
        if spec.get("loco"):
            v = [r for r in v if spec["loco"] not in r["cls"]]
        rs += cap(v, max(1, n // len(spec["train"])), "val")
    return rs


EVAL_CACHE = {}


def eval_set(ds, full=True):
    """main runs: the complete test split; ablations: a FIXED md5-ordered subset (same images for every ablation)."""
    key = (ds, full)
    if key not in EVAL_CACHE:
        n = C.TEST_EVAL or (C.MAX_EVAL_MAIN if full else C.MAX_EVAL_ABL)
        t = split_recs(ds, "test")
        EVAL_CACHE[key] = cap([r for r in t if not r["neg"]], n, "ev") + [r for r in t if r["neg"]][: (C.TEST_EVAL or C.N_NEG_EVAL)]
    return EVAL_CACHE[key]


def S(name, group, tier, train, mcfg=None, kind="xguard", seed=0, targets=None, full_eval=False, epochs=None, **kw):
    return dict(name=f"{name}_s{seed}", tag=name, group=group, tier=tier, kind=kind, train=train, mcfg=mcfg or {}, seed=seed,
                targets=targets or sorted(AVAILABLE), full_eval=full_eval, epochs=epochs or (C.EPOCHS_MAIN if full_eval else C.EPOCHS_ABL), **kw)


def registry():
    R, six, X = [], ["sixray"], (lambda **k: variant("xguard", **k))
    for s in C.SEEDS_MAIN:      # ---------------- tier 1 (Colab A100)
        R += [S("six_base", "core", 1, six, variant("base"), seed=s, full_eval=True, analyze=True, occ_sweep=True),
              S("six_xguard", "core", 1, six, variant("xguard"), seed=s, full_eval=True, analyze=True, occ_sweep=True),
              # fair control for the occlusion claim: the plain detector trained with XGuard's occlusion corruption
              S("six_base-occaug", "control", 1, six, variant("base", aug="occmix"), seed=s, full_eval=True, occ_sweep=True)]
    for g in ["pseudo", "vos", "gaussian"]:   # unknown supervision compared under the SAME protocol as XGuard (12 ep, full test)
        R.append(S(f"unkF-{g}", "unknown_main", 1, six, variant("xguard", unk_gen=g), seed=0, full_eval=True))
    for s in C.SEEDS_ABL:
        for v in ["sira", "oer", "stpb", "sira+oer", "sira+oer+stpb", "sira+oer+stpb+cus", "xguard-sira", "xguard-oer", "xguard-cus"]:
            R.append(S(f"six_{v}", "ablation", 1, six, variant(v), seed=s, analyze=v == "sira",
                       occ_sweep=v in ("oer", "sira+oer")))
        for c in LOCO_CLASSES:
            for v in ["base", "xguard"]:
                R.append(S(f"loco-{c}_{v}", "loco", 2, six, variant(v), seed=s, loco=c,
                           targets=[t for t in ["sixray", "opixray", "pidray"] if t in AVAILABLE]))
        # ---------------- tier 2
        for src in ["v", "t", "a", "vt", "va"]:
            R.append(S(f"proto-{src}", "prototype", 2, six, X(stpb=src), seed=s))
        for g in ["none", "gaussian", "mixup", "slerp"]:
            R.append(S(f"unk-{g}", "unknown_gen", 2, six, X(unk_gen=g), seed=s))
        R.append(S("unk-cus-noent", "unknown_gen", 2, six, X(), seed=s, loss_w=dict(ent=0.0)))
        R += [S("scan-rgb", "sira_validation", 2, six, variant("base", aug="rgb"), seed=s, analyze=True),
              S("scan-fourier", "sira_validation", 2, six, variant("base", aug="fourier"), seed=s, analyze=True),
              S("occ-cutout", "occlusion", 2, six, variant("base", aug="cutout"), seed=s, occ_sweep=True)]
        R += [S("six_fcos", "baselines", 2, six, kind="fcos", seed=s, lr_mult=0.5, full_eval=True),
              S("six_retinanet", "baselines", 2, six, kind="retinanet", seed=s, lr_mult=0.5, full_eval=True),
              S("six_owlvit", "zeroshot", 2, six, kind="owlvit", seed=s), S("six_gdino", "zeroshot", 2, six, kind="gdino", seed=s),
              S("six_owlvit-oracle", "zeroshot", 2, six, kind="owlvit", oracle=True, seed=s),
              S("six_gdino-oracle", "zeroshot", 2, six, kind="gdino", oracle=True, seed=s)]
        # ---------------- tier 3
        R += [S("six_yolo11s", "ultralytics", 3, six, kind="yolo", weights="yolo11s.pt", seed=s, full_eval=True),
              S("six_rtdetr-l", "ultralytics", 3, six, kind="yolo", weights="rtdetr-l.pt", seed=s, full_eval=True)]
        for ds in ["opixray", "pidray", "hixray"]:
            for v in ["base", "xguard"]:
                R.append(S(f"in-{ds}_{v}", "indomain", 3, [ds], variant(v), seed=s, targets=[ds]))
        four = [d for d in ["sixray", "opixray", "pidray", "hixray"] if d in AVAILABLE]
        for held in four:
            for v in ["base", "xguard"]:
                R.append(S(f"lodo-{held}_{v}", "lodo", 3, [d for d in four if d != held], variant(v), seed=s, targets=[held]))
        R += [S("enc-imagenet", "encoder", 3, six, X(), seed=s, init="imagenet"), S("enc-dino", "encoder", 3, six, X(), seed=s, init="dino"),
              S("enc-clip", "encoder", 3, six, X(), seed=s, init="clip"), S("enc-frozen", "encoder", 3, six, X(), seed=s, tl=0),
              S("enc-unfrozen", "encoder", 3, six, X(), seed=s, tl=5),
              S("src2", "num_sources", 3, ["sixray", "opixray"], X(), seed=s, targets=[t for t in ["pidray", "hixray"] if t in AVAILABLE]),
              S("src3", "num_sources", 3, ["sixray", "opixray", "hixray"], X(), seed=s, targets=[t for t in ["pidray"] if t in AVAILABLE])]
    return R


REGISTRY = registry()
PRIORITY = ["core", "control", "unknown_main", "ablation", "loco", "sira_validation", "unknown_gen", "prototype", "occlusion", "baselines", "zeroshot",
            "indomain", "lodo", "num_sources", "encoder", "ultralytics"]
SELECTED = sorted([s for s in REGISTRY if s["tier"] in TIERS], key=lambda s: (PRIORITY.index(s["group"]), s["seed"], s["name"]))
if MODE == "test":   # one representative per configuration type of THIS platform's tiers exercises every code path
    keep_tags = {"six_base", "six_xguard", "six_sira", "six_oer", "six_sira+oer", "six_sira+oer+stpb+cus",       # tier 1
                 "proto-t", "proto-a", "unk-gaussian", "unk-slerp", "unk-cus-noent", "scan-fourier", "occ-cutout",  # tier 2
                 "six_fcos", "six_owlvit", "six_gdino-oracle", "in-opixray_xguard", "six_yolo11s", "src2"}          # tier 2-3
    keep_tags |= {f"loco-{LOCO_CLASSES[0]}_base", f"loco-{LOCO_CLASSES[0]}_xguard"} if LOCO_CLASSES else set()
    SELECTED = [s for s in SELECTED if s["tag"] in keep_tags and s["seed"] == 0]
if MODE == "full" and C.DEADLINE:   # deadline plan: the runs that strengthen the paper most, in order of value
    _by = {s["name"]: s for s in REGISTRY}
    SELECTED = [_by[n] for n in C.DEADLINE_RUNS if n in _by]
    print("DEADLINE plan:", [s["name"] for s in SELECTED])
if MODE == "report":
    SELECTED = []
print(f"{len(REGISTRY)} experiments registered; {len(SELECTED)} selected for MODE={MODE}, TIERS={TIERS}")
print(Counter(f"T{s['tier']}:{s['group']}" for s in SELECTED))


# ------------------------- fingerprints (stale-cache protection) -------------------------
def _src_hash(objs):
    parts = []
    for o in objs:
        try:
            # lines tagged "@fp-neutral" only add NEW options; they do not change existing configurations,
            # so they are left out of the hash and finished runs stay valid
            parts.append("".join(l for l in inspect.getsource(o).splitlines(keepends=True) if "@fp-neutral" not in l))
        except Exception:
            parts.append(getattr(o, "__name__", str(o)))
    return sha(parts)


def fingerprints(spec, train_recs, val_recs, known):
    TRAIN_CODE = _src_hash([XGuard, STPB, CUS, OER, SIRAAdapter, different_class_pairs, train_model, prepare_views,
                            scanner_counterfactual, synth_occlusion, XrayDet, build_backbone_rpn, TVDetector])
    EVAL_CODE = _src_hash([evaluate, compute_thresholds, build_eval_items, coco_eval, infer, greedy_match, risk_coverage])
    cfg = {k: getattr(C, k) for k in ["IMG", "BATCH", "ACCUM", "LR", "WARMUP_ITERS", "WEIGHT_DECAY", "TEST_ITERS", "OCC_P", "OCC_FRAC",
                                     "VIEW2_DET", "LOSS_W", "EMB_DIM", "AMP", "GRAD_CLIP", "MAX_TRAIN_PER_DS", "PACK_LONG"]}
    train_fp = sha(dict(code=CODE_VERSION, src=TRAIN_CODE, spec={k: v for k, v in spec.items() if k not in ("targets", "full_eval", "analyze", "occ_sweep")},
                        cfg=cfg, train=sha(sorted(r["uid"] for r in train_recs)), val=sha(sorted(r["uid"] for r in val_recs)),
                        known=known, ontology=ONTOLOGY_VERSION, amp=str(AMP_DTYPE)))
    evals = {t: sha(sorted(r["uid"] for r in eval_set(t, spec["full_eval"]))) for t in spec["targets"] if t in AVAILABLE}
    eval_fp = sha(dict(src=EVAL_CODE, evals=evals, thr=[C.REFERRAL_BUDGETS, C.SCORE_THR, C.DETS_PER_IMG], occ=C.OCC_SWEEP,
                       alias=C.USE_SEMANTIC_ALIASES, ontology=ONTOLOGY_VERSION, train_fp=train_fp))
    return dict(train_fp=train_fp, eval_fp=eval_fp, train_code=TRAIN_CODE, eval_code=EVAL_CODE)


def check_stale(rd, fp):
    old = jload(rd / "fingerprint.json")
    if old is None:
        return
    if old["train_fp"] != fp["train_fp"]:
        dst = RUNS_DIR / "_stale" / f"{rd.name}_{time.strftime('%Y%m%d-%H%M%S')}"
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(rd), str(dst)); strip_heavy(dst)        # keep only its logs: saves Drive space
        print(f"   ♻️  training inputs/code changed -> old run moved to {dst} and retrained")
    elif old["eval_fp"] != fp["eval_fp"]:
        for f in list(rd.glob("preds_*.pkl")) + list(rd.glob("metrics_*.json")) + list(rd.glob("curves_*.json")) + \
                [rd / "thresholds.json", rd / "DONE", rd / "emb.npz"]:
            if f.exists():
                f.unlink()
        print("   ♻️  evaluation code/sets changed -> predictions and metrics recomputed (weights kept)")


# ------------------------- zero-shot & ultralytics adapters -------------------------
class ZeroShotOV(nn.Module):
    """OWL-ViT / Grounding-DINO. Name-free protocol: known-class prompts + ONE generic prompt for unknowns.
    oracle=True: the unseen class NAMES are also given (vocabulary disclosure) -> an upper-bound REFERENCE only."""
    GENERIC = "a dangerous object"

    def __init__(self, kind, known, oracle_names=None):
        super().__init__()
        from transformers import AutoProcessor
        self.kind, self.known, self.m = kind, list(known), dict(kind=kind, oracle=bool(oracle_names))
        name = "google/owlvit-base-patch32" if kind == "owlvit" else "IDEA-Research/grounding-dino-tiny"
        self.proc = AutoProcessor.from_pretrained(name)
        if kind == "owlvit":
            from transformers import OwlViTForObjectDetection as M
        else:
            from transformers import AutoModelForZeroShotObjectDetection as M
        self.net = M.from_pretrained(name).to(DEVICE).eval()
        self.init_used = name
        extra = [TEXT_NAME.get(c, c.replace("_", " ")) for c in (oracle_names or [])] or [self.GENERIC]
        self.queries = [TEXT_NAME.get(c, c.replace("_", " ")) for c in self.known] + extra

    @torch.no_grad()
    def forward(self, xs):
        res = []
        for x in xs:
            im = Image.fromarray((x.permute(1, 2, 0).float().cpu().numpy() * 255).astype(np.uint8))
            H, W = x.shape[-2:]
            if self.kind == "owlvit":
                inp = self.proc(text=[self.queries], images=im, return_tensors="pt").to(DEVICE)
                out = self.net(**inp)
                r = self.proc.post_process_object_detection(out, threshold=C.SCORE_THR, target_sizes=torch.tensor([[H, W]], device=DEVICE))[0]
                b, s, l = r["boxes"].float().cpu(), r["scores"].float().cpu(), r["labels"].cpu()
            else:
                inp = self.proc(images=im, text=". ".join(self.queries) + ".", return_tensors="pt").to(DEVICE)
                out = self.net(**inp)
                try:
                    r = self.proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=0.05, text_threshold=0.05, target_sizes=[(H, W)])[0]
                except TypeError:
                    r = self.proc.post_process_grounded_object_detection(out, inp.input_ids, box_threshold=0.05, text_threshold=0.05, target_sizes=[(H, W)])[0]
                b, s = r["boxes"].float().cpu(), r["scores"].float().cpu()
                txt = r.get("text_labels", r.get("labels"))
                l = torch.tensor([next((i for i, q in enumerate(self.queries) if q in str(t)), len(self.queries) - 1) for t in txt], dtype=torch.long)
            unk = l >= len(self.known)
            kb, ks, kl = b[~unk], s[~unk], l[~unk]
            nan_k, nan_u = torch.full_like(ks, float("nan")), torch.full_like(s[unk], float("nan"))
            res.append(dict(boxes=kb, scores=ks, labels=kl, U=1 - ks, R=1 - ks, E=nan_k, H=nan_k,
                            ub=b[unk], us=s[unk], uR=1 - s[unk], uE=nan_u, uH=nan_u))
        return res


def run_ultralytics(spec, train_recs, known, rd, epochs):
    if not INTERNET:
        raise RuntimeError("ultralytics baseline needs Internet (pip + weights)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=False)
    from ultralytics import YOLO, RTDETR
    root = CACHE / "yolo_export" / "_".join(spec["train"])
    cls2id = {c: i for i, c in enumerate(known)}
    for split, recs in [("train", train_recs), ("val", val_set(spec))]:
        (root / "images" / split).mkdir(parents=True, exist_ok=True); (root / "labels" / split).mkdir(parents=True, exist_ok=True)
        for r in recs:
            name = r["uid"].replace("/", "_") + os.path.splitext(r["path"])[1]
            dst = root / "images" / split / name
            if not dst.exists():
                os.symlink(r["path"], dst)
            with open(root / "labels" / split / (os.path.splitext(name)[0] + ".txt"), "w") as f:
                for b, c in zip(r["boxes"], r["cls"]):
                    if c in cls2id:
                        x1, y1, x2, y2 = b
                        f.write(f"{cls2id[c]} {(x1 + x2) / 2 / r['w']:.6f} {(y1 + y2) / 2 / r['h']:.6f} {(x2 - x1) / r['w']:.6f} {(y2 - y1) / r['h']:.6f}\n")
    (root / "data.yaml").write_text(f"path: {root}\ntrain: images/train\nval: images/val\nnames: {json.dumps(known)}\n")
    M_ = RTDETR if "rtdetr" in spec["weights"] else YOLO
    model = M_(spec["weights"])
    model.train(data=str(root / "data.yaml"), epochs=epochs, imgsz=C.IMG, batch=C.BATCH * 2, seed=spec["seed"], project=str(rd),
                name="ul", exist_ok=True, verbose=False, plots=False, fraction=0.05 if C.TEST_ITERS else 1.0)

    class Wrap(nn.Module):
        def __init__(s):
            super().__init__(); s.init_used = spec["weights"]; s.m = dict(kind="ultralytics")

        @torch.no_grad()
        def forward(s, xs):
            ims = [(x.permute(1, 2, 0).float().cpu().numpy()[:, :, ::-1] * 255).astype(np.uint8) for x in xs]
            out = []
            for r in model.predict(ims, imgsz=C.IMG, conf=C.SCORE_THR, verbose=False, max_det=C.DETS_PER_IMG):
                b = torch.from_numpy(r.boxes.xyxy.cpu().numpy()); sc = torch.from_numpy(r.boxes.conf.cpu().numpy())
                l = torch.from_numpy(r.boxes.cls.cpu().numpy()).long(); nan = torch.full_like(sc, float("nan"))
                out.append(dict(boxes=b, scores=sc, labels=l, U=1 - sc, R=1 - sc, E=nan, H=nan, ub=b, us=1 - sc, uR=1 - sc, uE=nan, uH=nan))
            return out
    return Wrap()


# ------------------------- runner -------------------------
@torch.no_grad()
def extract_embeddings(model, n_img=150):
    """GT-object embeddings of every test set (+ scanner-counterfactual consistency on SIXray, CUS samples)."""
    if not hasattr(model, "embed_boxes"):
        return None
    Z, D, Y, SYN, CONS = [], [], [], None, []
    allc = {c: i for i, c in enumerate(GLOBAL_ORDER + sorted({c for r in ALL_REC for c in r["cls"]} - set(GLOBAL_ORDER)))}
    inv = {i: c for c, i in allc.items()}
    for ds in sorted(AVAILABLE):
        recs = cap([r for r in split_recs(ds, "test") if not r["neg"] and not r.get("excluded")], n_img, "emb")
        for bi, (xs, ts) in enumerate(make_loader(recs, allc, False, bs=4)):
            xs = [x.to(DEVICE) for x in xs]
            bx = [t["boxes"].to(DEVICE) for t in ts]
            if sum(len(b) for b in bx) == 0:
                continue
            xcf = None
            if ds == "sixray":       # SIRA claim: same object under a different virtual scanner -> same embedding
                random.seed(1000 + bi)
                xcf = [scanner_counterfactual(x) for x in xs]      # fp32, outside autocast
            with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=C.AMP):
                z = model.embed_boxes(xs, bx)
                if xcf is not None:
                    z2 = model.embed_boxes(xcf, bx)
                    CONS += F.cosine_similarity(z.float(), z2.float(), dim=-1).cpu().tolist()
            Z.append(z.float().cpu().numpy()); D += [ds] * len(z); Y += [inv[int(l)] for t in ts for l in t["labels"]]
    if not Z:
        return None
    Z = np.concatenate(Z)
    if getattr(model, "cus", None) is not None:
        known = model.known
        m = np.array([y in known for y in Y]) & (np.array(D) == "sixray")
        if m.sum() > 4:
            zr = torch.from_numpy(Z[m]).to(DEVICE)
            yr = torch.tensor([known.index(y) for y, k in zip(Y, m) if k], device=DEVICE)
            out = model.cus.synth(zr, yr, 400)
            if out is not None:
                SYN = F.normalize(out.float(), dim=-1).cpu().numpy()
    return dict(z=Z, ds=np.array(D), cls=np.array(Y), synth=SYN, scanner_cos=np.array(CONS))


def write_run_artifacts(rd, spec, fp, known):
    jdump(fp, rd / "fingerprint.json")
    jdump(dict(spec=spec, C={k: v for k, v in vars(C).items()}, MODE=MODE, known=known), rd / "config.json")
    (rd / "dataset_manifest_hash.txt").write_text(str(MANIFEST_HASH))
    (rd / "ontology_version.txt").write_text(ONTOLOGY_VERSION)
    (rd / "git_commit.txt").write_text(ENV_INFO.get("git_commit", "n/a"))
    (rd / "seed.txt").write_text(str(spec["seed"]))
    jdump(ENV_INFO, rd / "environment.json")


def load_preds(pf, recs):
    try:
        d, sp = pickle.load(open(pf, "rb"))
    except Exception as e:          # half-written file (full Drive / disconnect) -> predictions are recomputed
        print(f"   ⚠️  {pf.name} unreadable ({type(e).__name__}) -> recomputing"); pf.unlink(missing_ok=True)
        return None, None
    return [d.get(r["uid"]) for r in recs], sp


def strip_heavy(rd, patterns=("*.pt", "*.pkl", "*.tmp")):
    """delete weights / raw predictions of a folder that is kept only for its logs (test runs, stale runs)."""
    n = 0
    for f in [f for pat in patterns for f in Path(rd).rglob(pat)]:
        try:
            n += f.stat().st_size; f.unlink()
        except Exception:
            pass
    return n / 1e9


def run_experiment(spec):
    rd = RUNS_DIR / spec["name"]
    miss = [d for d in spec["train"] if d not in AVAILABLE]
    targets = [t for t in spec["targets"] if t in AVAILABLE]
    if miss or not targets:
        return f"skipped (missing datasets {miss or spec['targets']})"
    if spec["kind"] in ("owlvit", "gdino", "yolo") and not INTERNET:
        return "skipped (needs Internet for weights)"
    train_recs, known = train_set(spec)
    vrecs = val_set(spec)
    fp = fingerprints(spec, train_recs, vrecs, known)
    check_stale(rd, fp)
    if (rd / "DONE").exists():
        return "done (fingerprint verified)"
    rd.mkdir(parents=True, exist_ok=True)
    write_run_artifacts(rd, spec, fp, known)
    t0 = time.time()
    print(f"\n▶ {spec['name']} [T{spec['tier']} {spec['group']}] train={spec['train']} images={len(train_recs)} known={known} epochs={spec['epochs']}")
    if spec["kind"] in ("owlvit", "gdino"):
        oracle = sorted({c for t in targets for r in eval_set(t, spec["full_eval"]) for c in r["cls"]
                         if resolve_known(c, known) is None}) if spec.get("oracle") else None
        model, meta = ZeroShotOV(spec["kind"], known, oracle), dict(init_used="zero-shot", oracle_vocabulary=oracle)
        jdump(meta, rd / "train_meta.json")
    elif spec["kind"] == "yolo":
        model, meta = run_ultralytics(spec, train_recs, known, rd, 1 if C.TEST_ITERS else spec["epochs"]), dict(init_used=spec["weights"])
        jdump(meta, rd / "train_meta.json")
    else:
        check_resume_checkpoint(rd)
        model, meta = train_model(spec, rd, train_recs, known, vrecs, spec["epochs"])
    thr = jload(rd / "thresholds.json")
    if thr is None:
        vp, _ = infer(model, vrecs)
        thr = compute_thresholds(vp, vrecs, known); jdump(thr, rd / "thresholds.json")
    dep = dict(train_minutes=meta.get("train_minutes"), sec_per_iter=meta.get("sec_per_iter"), val_sec_per_epoch=meta.get("val_sec_per_epoch"), init_used=meta.get("init_used"),
               gpu=GPU_NAME, platform=PLATFORM, amp=str(AMP_DTYPE), batch=C.BATCH, accum=C.ACCUM)
    eval_minutes = 0.0
    for tgt in targets:
        recs = eval_set(tgt, spec["full_eval"])
        pf = rd / f"preds_{tgt}.pkl"
        wait_pack(tgt)
        te = time.time()
        preds, sp = load_preds(pf, recs) if pf.exists() else (None, None)
        if preds is None:
            preds, sp = infer(model, recs)
            _d = ({r["uid"]: p for r, p in zip(recs, preds)}, sp)
            atomic_write(pf, lambda t: pickle.dump(_d, open(t, "wb")))
        eval_minutes += (time.time() - te) / 60
        dep.setdefault("fps", sp["fps"]); dep.setdefault("ms_per_img", sp["ms_per_img"]); dep.setdefault("eval_images", 0)
        dep["eval_images"] += len(recs)
        subsets = ["all"] + sorted({s for r in recs for s in r["subsets"]})
        out, curves = {}, {}
        for sub in subsets:
            out[sub], cv = evaluate(preds, recs, known, thr, None if sub == "all" else sub)
            curves[sub] = {k: (v.tolist() if v is not None else None) for k, v in cv.items()}
        jdump(out, rd / f"metrics_{tgt}.json"); jdump(curves, rd / f"curves_{tgt}.json")
        a = out["all"]
        print(f"   {tgt:8s} AP50={a.get('AP50', np.nan):.3f} xAP50={a.get('xAP50', np.nan):.3f} hAP50={a.get('hAP50', np.nan):.3f} "
              f"U-AP50={a.get('U_AP50', np.nan):.3f} U-R={a.get('U_R50', np.nan):.3f} locAUROC={a.get('locAUROC', np.nan):.3f} "
              f"AURC msp/rcr={a.get('AURC_msp', np.nan):.3f}/{a.get('AURC_rcr', np.nan):.3f}")
        bad = {k: v for k, v in a.get("metric_status", {}).items() if v.startswith("evaluator_failure")}
        if bad:
            print("   ❌ evaluator failures:", bad)
    if spec.get("occ_sweep") and "sixray" in targets and spec["kind"] == "xguard":
        recs = [r for r in eval_set("sixray", False) if not r["neg"]]
        sweep = {}
        for r_ in C.OCC_SWEEP:
            p_, _ = infer(model, recs, occ_ratio=r_)
            mm, _ = evaluate(p_, recs, known, thr, light=True)
            sweep[str(r_)] = mm.get("AP50")
        jdump(sweep, rd / "metrics_occsweep.json")
        print("   occlusion sweep AP50:", {k: round(v, 3) if v == v else v for k, v in sweep.items()})
    dep["eval_minutes"] = eval_minutes
    if isinstance(model, nn.Module) and spec["kind"] not in ("owlvit", "gdino", "yolo"):
        dep["params_M"] = count_params(model); dep["GFLOPs"] = model_flops(model)
    jdump(dep, rd / "deploy.json")
    if spec.get("analyze") and not (rd / "emb.npz").exists():
        wait_pack_all()
        e = extract_embeddings(model, 40 if C.TEST_ITERS else 150)
        if e is not None:
            atomic_write(rd / "emb.npz", lambda t: np.savez_compressed(open(t, "wb"), **{k: v for k, v in e.items() if v is not None}))
    if MODE == "test":
        strip_heavy(rd, ("*.pt", "*.tmp"))   # test runs: weights not needed (predictions kept for the test tables)
    (rd / "DONE").write_text(f"{(time.time() - t0) / 60:.1f} min")
    del model; gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return "finished"


## 14 · Run
Long cell. Resumable: re-run the notebook to continue. Test mode also prints the full-plan time / compute-unit estimate.

In [ ]:
# ==============================================================================
# 14. RUN (resumable; finished runs with a matching fingerprint are skipped)
#   test mode: every selected configuration runs C.TEST_ITERS iterations on REAL data, then the measured
#   speeds are projected to the full experiment plan (hours + Colab compute units per tier).
# ==============================================================================
if MODE == "prepare":
    raise StopExecution()   # prepare mode ends after the pack cell
STATUS = OrderedDict()
need_six = [s for s in SELECTED if "sixray" in s["train"]]
if "sixray" not in AVAILABLE and need_six:
    raise RuntimeError("SIXray has no box annotations in the pack -> rebuild the pack with khanhbtq99/sixray attached.")


def pilot_check():
    """Seed-0 pilot: XGuard must beat the plain detector BEFORE the remaining seeds / ablations spend GPU time.
    Compares mean harmonized AP50 over targets with known classes and mean U-AP50 over targets with unknowns."""
    b_, x_ = run_path("six_base_s0"), run_path("six_xguard_s0")
    if not ((b_ / "DONE").exists() and (x_ / "DONE").exists()):
        return None
    def summary(rd):
        ap, uap = [], []
        for mf in sorted(rd.glob("metrics_*.json")):
            if mf.stem == "metrics_occsweep":
                continue
            a = (jload(mf, {}) or {}).get("all", {})
            if isinstance(a.get("AP50"), (int, float)) and not math.isnan(a["AP50"]):
                ap.append(a["AP50"])
            if isinstance(a.get("U_AP50"), (int, float)) and not math.isnan(a["U_AP50"]):
                uap.append(a["U_AP50"])
        return float(np.mean(ap)) if ap else float("nan"), float(np.mean(uap)) if uap else float("nan")
    (ba, bu), (xa, xu) = summary(b_), summary(x_)
    res = dict(base_AP50=ba, xguard_AP50=xa, base_U_AP50=bu, xguard_U_AP50=xu,
               known_ok=bool(xa >= ba - 0.01), unknown_ok=bool(xu > bu))
    res["passed"] = res["known_ok"] and res["unknown_ok"]
    jdump(res, WORK / "logs" / "pilot_check.json")
    print(f"\n🧪 PILOT (seed 0): mean AP50 base {100 * ba:.1f} vs XGuard {100 * xa:.1f} | "
          f"mean U-AP50 base {100 * bu:.1f} vs XGuard {100 * xu:.1f} -> {'PASS' if res['passed'] else 'NOT PASSED'}")
    return res


def drive_guard(need_gb):
    """stop BEFORE a run if Drive is almost full: a full Drive breaks checkpoint writes and wastes GPU time."""
    free = drive_free_gb()
    if free is None or free >= need_gb:
        return True
    print(f"⛔ Google Drive has only {free:.1f} GB free (need ≥ {need_gb} GB). Free space: delete MyDrive/xguard/work/runs_test, "
          "then drive.google.com -> Trash -> Empty trash. Then Run all again (finished runs are kept).")
    return False


if MODE == "full":
    _freed = 0.0
    for _d in [WORK / "runs_test"] + [RUNS_DIR / "_stale", WORK / "runs_test" / "_stale"]:
        if _d.exists():
            _freed += strip_heavy(_d) if _d.name == "_stale" else strip_heavy(_d, ("*.pt", "*.tmp"))
    _freed += strip_heavy(RUNS_DIR, ("*.tmp",))
    _free = drive_free_gb()
    print(f"Drive: {_freed:.2f} GB of test / stale weights removed" + (f"; {_free:.1f} GB free" if _free is not None else "")
          + " (deleted files sit in the Drive trash until it is emptied)")
    if not drive_guard(C.MIN_DRIVE_GB):
        release_runtime("Google Drive is full")
        raise StopExecution()

PILOT = None
_gate_on = MODE == "full" and C.PILOT_GATE and not C.PILOT_FORCE and \
    any(s["group"] == "core" and s["seed"] == 0 for s in SELECTED)
for spec in SELECTED:
    if _gate_on and not (spec["group"] == "core" and spec["seed"] == 0):
        PILOT = PILOT or pilot_check() or dict(passed=False, reason="seed-0 base / XGuard run did not finish")
        if not PILOT["passed"]:
            print("⛔ pilot not passed -> the remaining runs are NOT started, so no GPU time is wasted.\n"
                  "   Send me the PILOT line above (logs/pilot_check.json). Set C.PILOT_FORCE = True only to continue anyway.")
            break
    if MODE == "full" and not drive_guard(C.MIN_DRIVE_GB):
        break
    if time_left_h() < 0.5:
        print(f"⏱  {elapsed_h():.1f} h used -> stopping. Re-run the notebook to continue (finished runs are kept on Drive).")
        break
    try:
        STATUS[spec["name"]] = run_experiment(spec)
    except TimeUp as e:
        STATUS[spec["name"]] = "paused"; print("⏱ ", e); break
    except Exception as e:
        traceback.print_exc()
        STATUS[spec["name"]] = f"error: {type(e).__name__}: {str(e)[:200]}"
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
RUN_STATUS = pd.DataFrame([dict(run=k, status=v) for k, v in STATUS.items()])
display(RUN_STATUS)
RUN_STATUS.to_csv(WORK / "logs" / f"run_status_{MODE}.csv", index=False)
errors = RUN_STATUS[RUN_STATUS.status.str.startswith("error")] if len(RUN_STATUS) else RUN_STATUS


def estimate_full_plan():
    """Project measured test-mode speeds to the full plan (all tiers) on this GPU."""
    rows, speed = [], {}
    for rd in all_runs():
        sp, dep = jload(rd / "config.json", {}).get("spec"), jload(rd / "deploy.json", {})
        if sp and dep.get("sec_per_iter"):
            two_view = bool(sp.get("mcfg", {}).get("sira_cf") or sp.get("mcfg", {}).get("occ"))
            speed.setdefault((sp["kind"], two_view), []).append(dep["sec_per_iter"])
        if sp and dep.get("val_sec_per_epoch"):
            speed.setdefault("val", []).append(dep["val_sec_per_epoch"] * C.MAX_VAL / max(1, C.TEST_VAL or C.MAX_VAL))
        if sp and dep.get("fps"):
            speed.setdefault(("fps", sp["kind"]), []).append(dep["fps"])
    if not speed:
        return None
    sp_ = {k: float(np.median(v)) for k, v in speed.items()}
    old_seeds = (C.SEEDS_MAIN, C.SEEDS_ABL)
    C.SEEDS_MAIN, C.SEEDS_ABL = [0, 1, 2], [0]       # the FULL plan: 3 seeds for the core runs
    try:
        full_reg = [s for s in registry() if s["tier"] in TIERS]
    finally:
        C.SEEDS_MAIN, C.SEEDS_ABL = old_seeds
    for s in full_reg:
        tr, _ = train_set(s)
        ipe = math.ceil(len(tr) / C.BATCH)
        two = bool(s.get("mcfg", {}).get("sira_cf") or s.get("mcfg", {}).get("occ"))
        spi = sp_.get((s["kind"], two)) or sp_.get(("xguard", two)) or np.nan
        n_eval = sum(len([r for r in split_recs(t, "test")]) if s["full_eval"] else min(C.MAX_EVAL_ABL, len(split_recs(t, "test")))
                     for t in s["targets"] if t in AVAILABLE)
        fps = sp_.get(("fps", s["kind"])) or sp_.get(("fps", "xguard")) or np.nan
        val_h = s["epochs"] * sp_.get("val", 60.0) / 3600 / max(1, C.VAL_EVERY)
        train_h = 0 if s["kind"] in ("owlvit", "gdino") else s["epochs"] * ipe * spi / 3600 * 1.05 + val_h
        rows.append(dict(tier=s["tier"], group=s["group"], run=s["name"], train_h=train_h, eval_h=n_eval / fps / 3600 * 1.3))
    df = pd.DataFrame(rows); df["total_h"] = df.train_h + df.eval_h
    cu = next((v for k, v in C.CU_PER_HOUR.items() if k in GPU_NAME), np.nan)
    summ = df.groupby(["tier", "group"]).agg(runs=("run", "count"), hours=("total_h", "sum")).reset_index()
    summ["compute_units"] = summ.hours * cu
    tot = df.groupby("tier").total_h.sum()
    print(f"\nProjected FULL plan for TIERS={TIERS} on {GPU_NAME} (measured speeds; Colab ≈ {cu} CU/h — verify on your Resources page):")
    display(summ)
    for t, h in tot.items():
        print(f"   tier {t}: {h:.1f} GPU-hours ≈ {h * cu:.0f} compute units")
    df.to_csv(WORK / "logs" / "full_plan_estimate.csv", index=False)
    return summ


if MODE == "test":
    PLAN = estimate_full_plan()
if MODE == "full" and PLATFORM == "colab":
    # every result is already on Drive; tables / figures / gates are built later with MODE='report' on a CPU runtime
    release_runtime("all selected Colab runs are finished or paused")


## 15 · Tables T0–T14

In [ ]:
# ==============================================================================
# 15. TABLES T1-T12 (+ LOCO, LODO, literature) -> tables/*.csv and *.tex
#   cells: mean±sd over finished seeds (n shown in T11); "n/a" = not applicable (metric_status says why);
#   "—" = run not finished / not selected. Nothing is ever filled in by hand.
# ==============================================================================
if MODE == "prepare":
    raise StopExecution()   # prepare mode ends after the pack cell
def load_results():
    rows = []
    for rd in all_runs():
        if not (rd / "DONE").exists():
            continue
        cfg, dep = jload(rd / "config.json", {}), jload(rd / "deploy.json", {})
        spec = cfg.get("spec", {})
        for mf in rd.glob("metrics_*.json"):
            tgt = mf.stem.replace("metrics_", "")
            if tgt == "occsweep":
                for r_, v in jload(mf, {}).items():
                    rows.append(dict(run=rd.name, tag=spec.get("tag"), group=spec.get("group"), seed=spec.get("seed"),
                                     target="sixray", subset=f"occ{r_}", AP50=v))
                continue
            for sub, M in jload(mf, {}).items():
                row = dict(run=rd.name, tag=spec.get("tag"), group=spec.get("group"), seed=spec.get("seed"), target=tgt, subset=sub)
                row.update({k: v for k, v in M.items() if not isinstance(v, (dict, list))})
                row.update({f"dep_{k}": v for k, v in dep.items() if not isinstance(v, (dict, list))})
                rows.append(row)
    return pd.DataFrame(rows)


RES = load_results()
print(f"{RES.run.nunique() if len(RES) else 0} finished runs in {[str(d) for d in RUNS_DIRS]}")
RES.to_csv(WORK / "tables" / f"all_results_long_{MODE}.csv", index=False)
TAB = WORK / "tables" / MODE
TAB.mkdir(parents=True, exist_ok=True)


def val(tag, target, metric, subset="all", fmt=True, scale=100.0):
    if not len(RES) or "tag" not in RES:
        return "—" if fmt else np.nan
    d = RES[(RES.tag == tag) & (RES.target == target) & (RES.subset == subset)]
    if not len(d):
        return "—" if fmt else np.nan
    if metric not in d or d[metric].dropna().empty:
        return "n/a" if fmt else np.nan
    v = d[metric].dropna().astype(float) * scale
    if not fmt:
        return float(v.mean())
    return f"{v.mean():.1f}±{v.std(ddof=1):.1f}" if len(v) > 1 else f"{v.mean():.1f}"


def num(x):
    try:
        return float(str(x).split("±")[0])
    except ValueError:
        return np.nan


def merge_external(df, table):
    if C.EXTERNAL_RESULTS and os.path.exists(C.EXTERNAL_RESULTS):
        ext = pd.read_csv(C.EXTERNAL_RESULTS)
        ext = ext[ext.table == table] if "table" in ext else ext.iloc[0:0]
        for m, g in ext.groupby("method"):
            row = {c: "—" for c in df.columns}; row.update({r.column: f"{r.value}" for r in g.itertuples() if r.column in df.columns})
            df.loc[f"{m} (reported)"] = row
    return df


def save_table(df, name, caption):
    df = merge_external(df, name)
    df.to_csv(TAB / f"{name}.csv")
    try:
        tex = df.to_latex(escape=True, caption=caption, label=f"tab:{name}")
    except Exception:
        tex = "% " + caption + "\n" + df.to_string()
    (TAB / f"{name}.tex").write_text(tex)
    display(Markdown(f"**{name}** — {caption}")); display(df)
    return df


NICE = {"six_base": "Faster R-CNN R50-FPN (base)", "six_fcos": "FCOS", "six_retinanet": "RetinaNet", "six_yolo11s": "YOLO11-s",
        "six_rtdetr-l": "RT-DETR-L", "six_owlvit": "OWL-ViT (zero-shot, name-free)", "six_gdino": "Grounding-DINO (zero-shot, name-free)",
        "six_owlvit-oracle": "OWL-ViT (ORACLE vocabulary)†", "six_gdino-oracle": "Grounding-DINO (ORACLE vocabulary)†",
        "six_xguard": "XGuard (composite)", "six_base-occaug": "Faster R-CNN + occlusion aug (control)"}
MAIN = ["six_base", "six_base-occaug", "six_fcos", "six_retinanet", "six_yolo11s", "six_rtdetr-l", "six_owlvit", "six_gdino", "six_owlvit-oracle",
        "six_gdino-oracle", "six_xguard"]
TG = [t for t in ["pidray", "opixray", "hixray", "clcxray"] if t in AVAILABLE]

# ---- T0 literature (qualitative; VERIFY every cell against the cited papers before submission) ----
save_table(pd.DataFrame([
    ["CHR (SIXray)", "–", "–", "partial", "–", "–"], ["DOAM (OPIXray)", "–", "–", "✓", "–", "–"],
    ["LIM (HiXray)", "–", "–", "partial", "–", "–"], ["SDANet (PIDray)", "–", "–", "✓ (hidden)", "–", "–"],
    ["OVXD", "partial", "✓ (named novel classes)", "–", "–", "–"], ["STING-BEE", "✓", "multimodal / zero-shot", "–", "–", "–"],
    ["XGuard (ours)", "✓", "✓ (name-free)", "✓", "✓", "✓"]],
    columns=["Method", "Domain shift", "Unknown class", "Occlusion", "Risk / referral", "Compound"]).set_index("Method"),
    "T0_literature", "Qualitative positioning (verify each cell against the papers).")

# ---- T1 dataset statistics, splits, ontology ----
rows = {}
for k, v in STATS.items():
    ver = VERIFICATION[VERIFICATION.dataset == k] if len(VERIFICATION) else VERIFICATION
    rows[k] = {"split origin": SPLIT_ORIGIN.get(k, "as shipped"),
               "classes": len(CLASS_AUDIT[CLASS_AUDIT.dataset == k]),
               **{f"{sp} img": v["splits"][sp]["images"] for sp in ["train", "val", "test"]},
               **{f"{sp} obj": v["splits"][sp]["instances"] for sp in ["train", "test"]},
               "test subsets": ", ".join(f"{a}:{b}" for a, b in v["splits"]["test"]["subsets"].items()),
               "excluded (leakage)": AUDIT_SUMMARY.get("excluded_by_dataset", {}).get(k, 0),
               "official-size check": ", ".join(f"{r.quantity}:{r.status}" for r in ver.itertuples()) or "—",
               "lineage": f"{LINEAGE_LOG.get(k, {}).get('files', '?')} files → {LINEAGE_LOG.get(k, {}).get('unique_ids', '?')} ids"}
save_table(pd.DataFrame(rows).T, "T1_datasets", "Dataset statistics, split provenance, leakage exclusions and verification against official sizes.")

# ---- T2 in-domain ----
rows = {NICE.get(t, t): {m: val(t, "sixray", m) for m in ["AP", "AP50", "AP75", "AR100", "APs", "APm", "APl"]} for t in MAIN}
for ds in ["opixray", "pidray", "hixray"]:
    for v in ["base", "xguard"]:
        rows[f"[{ds}] {v}"] = {m: val(f"in-{ds}_{v}", ds, m) for m in ["AP", "AP50", "AP75", "AR100", "APs", "APm", "APl"]}
save_table(pd.DataFrame(rows).T, "T2_in_domain", "P1 in-domain known-object detection on the SIXray (mirror) test split, COCO metrics (%).")

# ---- T3 cross-dataset generalization (three tracks) ----
rows = {}
for t in MAIN + ["six_sira"]:
    r = {}
    for g in TG:
        r[f"{g} xAP50"], r[f"{g} AP50"], r[f"{g} hAP50"] = val(t, g, "xAP50"), val(t, g, "AP50"), val(t, g, "hAP50")
    a = [val(t, g, "AP50", fmt=False) for g in TG]
    r["mean AP50 (harmonized)"] = f"{np.nanmean(a):.1f}" if np.isfinite(a).any() else "n/a"
    rows[NICE.get(t, t)] = r
save_table(pd.DataFrame(rows).T, "T3_cross_domain", "P2 single-source generalization (train SIXray only). xAP50 exact names, AP50 harmonized aliases, hAP50 superclass.")

# ---- T4 unknown detection and false alarms ----
rows = {}
for t in MAIN + ["six_sira+oer+stpb+cus"]:
    r = {}
    for g in [x for x in ["pidray", "hixray", "opixray"] if x in AVAILABLE]:
        for m, lab in [("U_AP50", "U-AP50"), ("U_R50", "U-R50"), ("U_R50_op", "U-R50@τ"), ("locAUROC", "locAUROC"),
                       ("locCoverage", "loc-cov"), ("FA_per_img", "FA/img")]:
            r[f"{g} {lab}"] = val(t, g, m, scale=1 if m == "FA_per_img" else 100)
        r[f"{g} AOSE"] = val(t, g, "AOSE", scale=1)
    r["SIXray FA/clean bag"] = val(t, "sixray", "FA_per_clean_bag", "negative", scale=1)
    rows[NICE.get(t, t)] = r
save_table(pd.DataFrame(rows).T, "T4_unknown", "P5 unknown threats (never named in training). locAUROC only over localized objects (loc-cov). †oracle = unseen names given.")

# LOCO
rows = {}
for c in LOCO_CLASSES:
    for v in ["base", "xguard"]:
        rows[f"{c} | {v}"] = {m: val(f"loco-{c}_{v}", "sixray", k) for m, k in
                             [("Known AP50", "AP50"), ("U-AP50", "U_AP50"), ("U-R50", "U_R50"), ("locAUROC", "locAUROC"), ("AOSE", "AOSE")]}
if rows:
    df = pd.DataFrame(rows).T
    for v in ["base", "xguard"]:
        sub = df[[i.endswith(v) for i in df.index]].apply(lambda col: col.map(num))
        df.loc[f"mean | {v}"] = [f"{x:.1f}" if np.isfinite(x) else "n/a" for x in sub.mean()]
    save_table(df, "T4b_loco", "P4 leave-one-class-out on SIXray (folds = classes with enough train AND test objects).")

# ---- T5 occlusion robustness ----
rows = {}
for t in ["six_base", "six_base-occaug", "six_oer", "six_sira+oer", "occ-cutout", "six_xguard"]:
    r = {f"OL{i}": val(t, "opixray", "hAP50", f"OL{i}") for i in (1, 2, 3)}
    r.update({f"PIDray {sub} AP50": val(t, "pidray", "AP50", sub) for sub in ("easy", "hard", "hidden")})   # REAL concealment
    for r_ in C.OCC_SWEEP:
        r[f"synthetic occ {int(r_ * 100)}% AP50"] = val(t, "sixray", "AP50", f"occ{r_}")
    a0, a5 = val(t, "sixray", "AP50", f"occ{C.OCC_SWEEP[0]}", fmt=False), val(t, "sixray", "AP50", f"occ{C.OCC_SWEEP[-1]}", fmt=False)
    r["degradation 0→max ↓"] = f"{(a0 - a5) / a0:.3f}" if np.isfinite(a0) and np.isfinite(a5) and a0 else "n/a"
    rows[NICE.get(t, t)] = r
save_table(pd.DataFrame(rows).T, "T5_occlusion", "P6 occlusion: official OPIXray OL1-3 (if lists provided) + deterministic synthetic occlusion sweep on SIXray test.")

# ---- T6 compound shift ----
rows = {}
for t in ["six_base", "six_base-occaug", "six_xguard", "unkF-pseudo", "unkF-vos", "unkF-gaussian", "six_rtdetr-l", "six_gdino", "six_owlvit"]:
    rows[f"PIDray-hidden, unseen classes | {NICE.get(t, t)}"] = {
        "U-AP50": val(t, "pidray", "U_AP50", "hidden"), "U-R50": val(t, "pidray", "U_R50", "hidden"),
        "miss-rate ↓": val(t, "pidray", "miss_rate", "hidden"), "FA/img ↓": val(t, "pidray", "FA_per_img", "hidden", scale=1)}
for v in ["base", "xguard"]:
    rows[f"OPIXray, knife never seen (LOCO-knife) | {v}"] = {
        "U-AP50": val(f"loco-knife_{v}", "opixray", "U_AP50"), "U-R50": val(f"loco-knife_{v}", "opixray", "U_R50"),
        "miss-rate ↓": val(f"loco-knife_{v}", "opixray", "miss_rate"), "FA/img ↓": val(f"loco-knife_{v}", "opixray", "FA_per_img", scale=1)}
save_table(pd.DataFrame(rows).T, "T6_compound", "P7 compound shift: unseen scanner × unseen class × concealment (PIDray hidden).")

# ---- T7 component ablation (independent, cumulative, leave-one-out) ----
ABL = [("six_base", "", "", "", "", ""), ("six_sira", "✓", "", "", "", ""), ("six_oer", "", "✓", "", "", ""),
       ("six_stpb", "", "", "✓", "", ""), ("six_sira+oer", "✓", "✓", "", "", ""), ("six_sira+oer+stpb", "✓", "✓", "✓", "", ""),
       ("six_sira+oer+stpb+cus", "✓", "✓", "✓", "✓", ""), ("six_xguard", "✓", "✓", "✓", "✓", "✓"),
       ("six_xguard-sira", "", "✓", "✓", "✓", "✓"), ("six_xguard-oer", "✓", "", "✓", "✓", "✓"),
       ("six_xguard-cus", "✓", "✓", "✓", "", "✓"), ("six_sira+oer+stpb+cus", "✓", "✓", "✓", "✓", "– (=full−RCR)")]


def abl_metrics(t):
    base = lambda m, g, sub="all": val("six_base", g, m, sub, fmt=False)
    me = lambda m, g, sub="all": val(t, g, m, sub, fmt=False)
    d_dom = me("AP50", "pidray") - base("AP50", "pidray")
    d_unk = me("U_AP50", "pidray") - base("U_AP50", "pidray")
    d_rc = me("AURC_rcr", "pidray") - base("AURC_msp", "pidray")
    f = lambda x: f"{x:+.1f}" if np.isfinite(x) else "n/a"
    return {"SIXray AP50": val(t, "sixray", "AP50"), "PIDray AP50 (domain)": val(t, "pidray", "AP50"),
            "PIDray U-AP50": val(t, "pidray", "U_AP50"), "occ 50% AP50": val(t, "sixray", "AP50", "occ0.5"),
            "AURC rcr ↓": val(t, "pidray", "AURC_rcr"), "ΔAP_domain": f(d_dom), "ΔU-AP": f(d_unk), "ΔAURC ↓": f(d_rc)}


rows = {}
for t, *flags in ABL:
    key = f"{'+'.join(n for n, f in zip(['SIRA', 'OER', 'STPB', 'CUS', 'RCR'], flags) if f == '✓') or 'base'}" + (" (full−RCR)" if flags[-1].startswith("–") else "")
    rows[key] = {"SIRA": flags[0], "OER": flags[1], "STPB": flags[2], "CUS": flags[3], "RCR": flags[4], **abl_metrics(t)}
save_table(pd.DataFrame(rows).T, "T7_ablation", "Component ablation (single seed, fixed eval subsets). Δ relative to the matched base detector.")

# ---- T8 prototype sources ----
rows = {}
for t, lab in [("proto-v", "visual"), ("proto-t", "text"), ("proto-a", "attribute"), ("proto-vt", "visual+text"),
               ("proto-va", "visual+attribute"), ("six_xguard", "visual+text+attribute (full)")]:
    rows[lab] = {"SIXray AP50": val(t, "sixray", "AP50"), "PIDray AP50": val(t, "pidray", "AP50"), "PIDray U-AP50": val(t, "pidray", "U_AP50"),
                 "HiXray U-AP50": val(t, "hixray", "U_AP50"), "locAUROC (PIDray)": val(t, "pidray", "locAUROC")}
save_table(pd.DataFrame(rows).T, "T8_prototype", "STPB prototype-source ablation inside the full model (text = CLIP; see CLIP integrity in gates).")

# ---- T9 synthetic unknown generation ----
rows = {}
for t, lab in [("unk-none", "none (U = o·s_t·(1−s_k))"), ("unk-gaussian", "Gaussian feature noise"), ("unk-mixup", "feature mixup"),
               ("unk-slerp", "manifold (slerp) interpolation"), ("six_xguard", "CUS (factorized, ours)"), ("unk-cus-noent", "CUS without entropy loss")]:
    log_ = pd.read_csv(run_path(f"{t}_s0") / "train_log.csv") if (run_path(f"{t}_s0") / "train_log.csv").exists() else pd.DataFrame()
    rows[lab] = {"SIXray known AP50": val(t, "sixray", "AP50"), "PIDray U-AP50": val(t, "pidray", "U_AP50"), "PIDray U-R50": val(t, "pidray", "U_R50"),
                 "HiXray U-AP50": val(t, "hixray", "U_AP50"), "locAUROC": val(t, "pidray", "locAUROC"),
                 "synthetic max-known cos": f"{log_.synth_max_known_cos.dropna().iloc[-5:].mean():.3f}" if "synth_max_known_cos" in log_ and log_.synth_max_known_cos.notna().any() else "n/a",
                 "real max-known cos": f"{log_.real_max_known_cos.dropna().iloc[-5:].mean():.3f}" if "real_max_known_cos" in log_ and log_.real_max_known_cos.notna().any() else "n/a"}
save_table(pd.DataFrame(rows).T, "T9_unknown_generation", "Synthetic-unknown strategies under the same protocol (+ known-support similarity of generated features).")

# ---- T9b unknown supervision under the MAIN protocol (12 epochs, full test sets, same detector) ----
rows = {}
for t, lab in [("six_base", "none (detector score only)"), ("unkF-gaussian", "Gaussian feature noise"),
               ("unkF-vos", "VOS-style low-likelihood sampling"), ("unkF-pseudo", "ORE/OW-DETR-style pseudo-unknowns"),
               ("six_xguard", "factor mixing (XGuard)")]:
    log_ = pd.read_csv(run_path(f"{t}_s0") / "train_log.csv") if (run_path(f"{t}_s0") / "train_log.csv").exists() else pd.DataFrame()
    last = lambda k: (f"{log_[k].dropna().iloc[-5:].mean():.3f}" if k in log_ and log_[k].notna().any() else "n/a")
    rows[lab] = {"SIXray AP50": val(t, "sixray", "AP50"), "PIDray AP50": val(t, "pidray", "AP50"),
                 "PIDray-hidden U-R50": val(t, "pidray", "U_R50", "hidden"), "PIDray U-AP50": val(t, "pidray", "U_AP50"),
                 "PIDray locAUROC": val(t, "pidray", "locAUROC"), "HiXray U-AP50": val(t, "hixray", "U_AP50"),
                 "HiXray AOSE ↓": val(t, "hixray", "AOSE", scale=1), "final unknown loss": last("unknown"),
                 "synthetic max-known cos": last("synth_max_known_cos"), "real max-known cos": last("real_max_known_cos")}
save_table(pd.DataFrame(rows).T, "T9b_unknown_supervision", "Unknown supervision recipes in the same detector, 12 epochs, full test sets, seed 0.")

# ---- T10 reliability / referral ----
rows = {}
for t in ["six_base", "six_xguard"]:
    for k in RISK_TYPES:
        rows[f"{NICE.get(t, t)} | {k}"] = {
            "AURC ↓": val(t, "pidray", f"AURC_{k}"), "E-AURC ↓": val(t, "pidray", f"EAURC_{k}"),
            **{f"referral @{int(q * 100)}%": val(t, "pidray", f"ref_rate_{k}@{int(q * 100)}") for q in C.REFERRAL_BUDGETS},
            **{f"missed threats after ref @{int(q * 100)}% ↓": val(t, "pidray", f"miss_after_ref_{k}@{int(q * 100)}") for q in C.REFERRAL_BUDGETS},
            **{f"FA/bag after ref @{int(q * 100)}% ↓": val(t, "pidray", f"fa_after_ref_{k}@{int(q * 100)}", scale=1) for q in C.REFERRAL_BUDGETS}}
    rows[f"{NICE.get(t, t)} | calibration"] = {"AURC ↓": f"ECE {val(t, 'pidray', 'ECE')} / ECE(1−R) {val(t, 'pidray', 'ECE_R')}",
                                               "E-AURC ↓": f"Brier {val(t, 'pidray', 'Brier')}"}
save_table(pd.DataFrame(rows).T.fillna(""), "T10_reliability", "Risk scores compared on PIDray (unseen scanner): detection-level AURC/E-AURC and bag-level referral at budgets fixed on SIXray val.")

# ---- T11 multi-seed statistics + paired bootstrap ----
rows = {}
for t in ["six_base", "six_base-occaug", "six_xguard"]:
    n = RES[(RES.tag == t)].seed.nunique() if len(RES) else 0
    rows[f"{NICE.get(t, t)} (n={n} seeds)"] = {f"{g} {m}": val(t, g, m) for g in ["sixray"] + TG for m in ["AP50", "U_AP50"]}
save_table(pd.DataFrame(rows).T, "T11_seeds", "Mean ± SD over independent seeds.")
rows = {}
for t in ["sixray"] + TG:
    pa, pb = run_path("six_base_s0") / f"preds_{t}.pkl", run_path("six_xguard_s0") / f"preds_{t}.pkl"
    if pa.exists() and pb.exists():
        known = jload(run_path("six_xguard_s0") / "config.json")["known"]
        rec_t = eval_set(t, True)
        for metric in ["AP50", "U_AP50"]:
            ci = paired_bootstrap(load_preds(pa, rec_t)[0], load_preds(pb, rec_t)[0], rec_t, known, metric)
            rows[f"{t} {metric}"] = {"Δ (XGuard−base)": f"{100 * ci['delta']:+.1f}" if np.isfinite(ci["delta"]) else "n/a",
                                     "95% CI": f"[{100 * ci['lo']:+.1f}, {100 * ci['hi']:+.1f}]" if np.isfinite(ci["lo"]) else "n/a",
                                     "P(Δ≤0)": f"{ci['p_le_0']:.3f}" if np.isfinite(ci["p_le_0"]) else "n/a"}
save_table(pd.DataFrame(rows).T, "T11b_bootstrap", f"Paired image-level bootstrap ({C.BOOTSTRAP} resamples), same test images for both models.")

# ---- T12 efficiency ----
rows = {}
for t in MAIN + ["six_sira", "six_oer", "six_sira+oer+stpb+cus"]:
    d = RES[(RES.tag == t)] if len(RES) else pd.DataFrame()
    g = lambda k, f="{:.1f}": (f.format(d[k].dropna().astype(float).mean()) if len(d) and k in d and d[k].dropna().size else "—")
    rows[NICE.get(t, t)] = {"Params (M)": g("dep_params_M"), "GFLOPs": g("dep_GFLOPs"), "FPS": g("dep_fps"), "ms/img": g("dep_ms_per_img"),
                            "train min": g("dep_train_minutes", "{:.0f}")}
save_table(pd.DataFrame(rows).T, "T12_efficiency", f"Efficiency on {GPU_NAME}, input {C.IMG}px, AMP {AMP_DTYPE}.")

# ---- LODO ----
rows = {}
for held in ["sixray", "opixray", "pidray", "hixray"]:
    for v in ["base", "xguard"]:
        rows[f"→ {held} | {v}"] = {m: val(f"lodo-{held}_{v}", held, k) for m, k in [("AP50", "AP50"), ("hAP50", "hAP50"), ("U-AP50", "U_AP50"), ("locAUROC", "locAUROC")]}
save_table(pd.DataFrame(rows).T, "T13_lodo", "P3 leave-one-domain-out (tier 3).")


## 16 · Representation evidence (SIRA validation)

In [ ]:
# ==============================================================================
# 16. REPRESENTATION EVIDENCE (SIRA validation)
#   domain probe ↓ (dataset identity from GT-object embeddings), threat probe ↑ (superclass), scanner-
#   counterfactual consistency ↑ (cos(z(x), z(T_phi x))), feature geometry D_same / D_diff, together with
#   source AP and target AP -- a lower domain probe alone is NOT evidence; threat information must survive.
# ==============================================================================
if MODE == "prepare":
    raise StopExecution()   # prepare mode ends after the pack cell
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE


def load_emb(tag):
    p = run_path(f"{tag}_s0") / "emb.npz"
    return dict(np.load(p, allow_pickle=True)) if p.exists() else None


def probe(X, y, seed=0):
    y = np.asarray(y); cnt = Counter(y)
    keep_cls = [c for c, n in cnt.items() if n >= 5]
    if len(keep_cls) < 2:
        return np.nan
    m = np.isin(y, keep_cls); X, y = X[m], y[m]
    n = min(Counter(y).values()); rng = np.random.RandomState(seed)
    idx = np.concatenate([rng.choice(np.where(y == c)[0], n, replace=False) for c in keep_cls])
    k = min(5, n)
    return float(cross_val_score(LogisticRegression(max_iter=2000), X[idx], y[idx], cv=StratifiedKFold(k, shuffle=True, random_state=seed)).mean())


def geometry(Z, D, Y):
    Zn = Z / np.linalg.norm(Z, axis=1, keepdims=True).clip(1e-6)
    shared = [c for c in set(Y) if len(set(D[Y == c])) >= 2]
    same, diff, rng = [], [], np.random.RandomState(0)
    for c in shared:
        idx = np.where(Y == c)[0]
        for _ in range(min(2000, len(idx) ** 2)):
            i, j = rng.choice(idx, 2)
            if D[i] != D[j]:
                same.append(1 - Zn[i] @ Zn[j])
    for _ in range(4000):
        i, j = rng.randint(0, len(Y), 2)
        if Y[i] != Y[j]:
            diff.append(1 - Zn[i] @ Zn[j])
    return (np.mean(same) if same else np.nan), (np.mean(diff) if diff else np.nan), shared


REP = {}
for tag, lab in [("six_base", "base"), ("scan-rgb", "+ RGB jitter"), ("scan-fourier", "+ Fourier (FDA)"),
                 ("six_sira", "+ SIRA"), ("six_xguard", "XGuard (full)")]:
    e = load_emb(tag)
    if e is None:
        continue
    Z, D, Y = e["z"], e["ds"], e["cls"]
    ds_, dd_, shared = geometry(Z, D, Y)
    REP[lab] = {"domain-probe acc ↓": probe(Z, D), "threat-probe acc ↑": probe(Z, np.array([superclass(c) for c in Y])),
                "scanner-cf cos ↑": float(np.mean(e["scanner_cos"])) if "scanner_cos" in e and len(e["scanner_cos"]) else np.nan,
                "D_same/D_diff ↓": ds_ / dd_ if dd_ else np.nan,
                "source AP50 (SIXray)": val(tag, "sixray", "AP50"), "target AP50 (PIDray)": val(tag, "pidray", "AP50"),
                "n objects": len(Z)}
if REP:
    save_table(pd.DataFrame(REP).T, "T14_sira_validation",
               "SIRA validation: less scanner information AND preserved threat information AND target gains (all required).")
else:
    print("no embeddings yet (analyze=True runs not finished)")


## 17 · Figures 1–8

In [ ]:
# ==============================================================================
# 17. FIGURES 1-8 (PNG + PDF). Only Fig. 1 (architecture) is a drawing; every other figure is generated
#     from real model outputs of this run (no illustrative / fabricated detections).
#   1 architecture · 2 feature space by dataset · 3 qualitative known vs unknown · 4 occlusion curves
#   5 risk-coverage + referral · 6 CUS validity · 7 failure cases · 8 per-class cross-domain AP heatmap
# ==============================================================================
if MODE == "prepare":
    raise StopExecution()   # prepare mode ends after the pack cell
wait_pack_all()          # qualitative figures read test images of every dataset
FIG = WORK / "figures" / MODE
FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False})


def savefig(name):
    fig_ = plt.gcf()           # paper figures carry their caption in LaTeX: drop the "Fig. N ..." titles from the image
    if getattr(fig_, "_suptitle", None) is not None and fig_._suptitle.get_text().startswith("Fig."):
        fig_._suptitle.set_text("")
    for ax_ in fig_.axes:
        if ax_.get_title().startswith("Fig."):
            ax_.set_title("")
    plt.savefig(FIG / f"{name}.png", dpi=200, bbox_inches="tight"); plt.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    plt.show(); plt.close()


def box(ax, x, y, w, h, text, fc="#E8EEF7", ec="#34495E", fs=8, bold=False):
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.02", fc=fc, ec=ec, lw=1.2))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=fs, weight="bold" if bold else "normal")


def arrow(ax, x1, y1, x2, y2, c="#34495E"):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", color=c, lw=1.3))


def run_file(tag, name, seed=0):
    p = run_path(f"{tag}_s{seed}") / name
    return p if p.exists() else None


# ---------- Fig. 1 architecture ----------
fig, ax = plt.subplots(figsize=(10, 4.2)); ax.axis("off"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
box(ax, 0.01, 0.42, 0.09, 0.16, "X-ray\nimage", fc="#EEEEEE")
box(ax, 0.13, 0.40, 0.12, 0.20, "ResNet-50\n+ FPN", fc="#DDEBF7")
box(ax, 0.28, 0.66, 0.16, 0.22, "SIRA\nF_L=LP(F), F_H=F−F_L\nG=σ(MLP[F_L,F_H])\n+ T_φ consistency", fc="#FCE4D6", fs=7)
box(ax, 0.28, 0.12, 0.16, 0.22, "RPN\n→ RoI tokens (7×7)", fc="#EEEEEE", fs=7)
box(ax, 0.48, 0.12, 0.16, 0.22, "OER\nA=qkᵀ/√d+λ·log r_i\nvisibility-supervised r_i", fc="#FCE4D6", fs=7)
box(ax, 0.48, 0.66, 0.16, 0.22, "STPB\np_c = Σ w_k p_c^k\nk ∈ {visual, text, attribute}", fc="#E2F0D9", fs=7)
box(ax, 0.68, 0.40, 0.13, 0.20, "RoI embedding z\nbox · class\nU=o·u(z)", fc="#DDEBF7", fs=7)
box(ax, 0.68, 0.72, 0.13, 0.18, "CUS (train only)\n[s_A, m_B, λg_A+(1−λ)g_B]", fc="#E2F0D9", fs=7)
box(ax, 0.84, 0.40, 0.15, 0.20, "RCR\nR=P(wrong|z)\nDetect / Refer", fc="#FFF2CC", fs=8, bold=True)
for a in [(0.10, 0.5, 0.13, 0.5), (0.25, 0.55, 0.28, 0.75), (0.25, 0.45, 0.28, 0.25), (0.44, 0.23, 0.48, 0.23),
          (0.64, 0.23, 0.68, 0.45), (0.64, 0.77, 0.68, 0.55), (0.745, 0.72, 0.745, 0.60), (0.81, 0.5, 0.84, 0.5), (0.36, 0.66, 0.36, 0.34)]:
    arrow(ax, *a)
ax.set_title("Fig. 1  XGuard architecture", y=-0.02)
savefig("fig1_architecture")

# ---------- Fig. 2 feature space coloured by dataset ----------
embs = {t: load_emb(t) for t in ["six_base", "six_sira", "six_xguard"]}
embs = {k: v for k, v in embs.items() if v is not None}
if embs:
    fig, axes = plt.subplots(1, len(embs), figsize=(4.6 * len(embs), 4.2), squeeze=False)
    for ax, (t, e) in zip(axes[0], embs.items()):
        Z = e["z"]; sel = np.random.RandomState(0).choice(len(Z), min(len(Z), 3000), replace=False)
        P2 = TSNE(2, init="pca", perplexity=min(30, max(5, len(sel) // 4)), random_state=0).fit_transform(
            Z[sel] / np.linalg.norm(Z[sel], axis=1, keepdims=True).clip(1e-6))
        for d in sorted(set(e["ds"])):
            m = e["ds"][sel] == d; ax.scatter(P2[m, 0], P2[m, 1], s=4, label=d, alpha=0.6)
        ax.set_title(NICE.get(t, t)); ax.legend(fontsize=7, markerscale=3); ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle("Fig. 2  GT-object embeddings (t-SNE) coloured by dataset / scanner")
    savefig("fig2_feature_space")


# ---------- helpers for prediction-based figures ----------
def preds_of(tag, t, full=True):
    pf = run_file(tag, f"preds_{t}.pkl")
    return load_preds(pf, eval_set(t, full))[0] if pf else None


def draw(ax, r, p, kn, thr, title=""):
    im = np.asarray(load_rgb(r["path"], 1024).resize((r["w"], r["h"])))
    ax.imshow(im); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title, fontsize=8)
    if p is None:
        return
    for b, s, l in zip(p["boxes"], p["scores"], p["labels"]):
        if s >= thr["tau_known"]:
            ax.add_patch(mpatches.Rectangle(b[:2], b[2] - b[0], b[3] - b[1], fill=False, ec="lime", lw=1.3))
            ax.text(b[0], b[1], f"{kn[l]} {s:.2f}", color="lime", fontsize=6, va="bottom")
    for b, s in zip(p["ub"], p["us"]):
        if s >= thr["tau_U"]:
            ax.add_patch(mpatches.Rectangle(b[:2], b[2] - b[0], b[3] - b[1], fill=False, ec="magenta", lw=1.3, ls="--"))
            ax.text(b[0], b[3], f"UNKNOWN {s:.2f}", color="magenta", fontsize=6, va="top")


def draw_gt(ax, r):
    for b, c in zip(r["boxes"], r["cls"]):
        ax.add_patch(mpatches.Rectangle(b[:2], b[2] - b[0], b[3] - b[1], fill=False, ec="yellow", lw=1.5))
        ax.text(b[0], b[1], c, color="yellow", fontsize=6, va="bottom")


def pick(tag_x, t, kind, known_x, thr, sub=None, k=1):
    px = preds_of(tag_x, t)
    if px is None:
        return []
    scored = []
    for i, r in enumerate(eval_set(t, True)):
        if px[i] is None or r.get("excluded") or (sub and sub not in r["subsets"]):
            continue
        p = px[i]
        A = np.concatenate([p["boxes"][p["scores"] >= thr["tau_known"]], p["ub"][p["us"] >= thr["tau_U"]]])
        G = r["boxes"]
        hit = np_iou(G, A).max(1) >= 0.5 if len(G) and len(A) else np.zeros(len(G), bool)
        unk = np.array([resolve_known(c, known_x) is None for c in r["cls"]], bool)
        if kind == "unknown_found":
            s = (hit & unk).sum() if unk.any() else -1
        elif kind == "known_found":
            s = (hit & ~unk).sum() - 0.1 * len(A) if (~unk).any() else -1
        elif kind == "missed":
            s = (~hit).sum() if len(G) else -1
        elif kind == "false_alarm":
            s = int((np_iou(A, G).max(1) < 0.5).sum()) if len(G) else len(A)
        else:
            s = -1
        if s > 0:
            scored.append((s, i))
    return [(t, i) for s, i in sorted(scored, reverse=True)[:k]]


thr_x = jload(run_file("six_xguard", "thresholds.json") or "_none_")
cfg_x, cfg_b = jload(run_file("six_xguard", "config.json") or "_none_"), jload(run_file("six_base", "config.json") or "_none_")
if thr_x and cfg_x and cfg_b:
    kx, kb = cfg_x["known"], cfg_b["known"]
    thr_b = jload(run_file("six_base", "thresholds.json"))
    # ---------- Fig. 3 qualitative known vs unknown ----------
    cases = []
    for t, kind, sub in [("sixray", "known_found", None), ("pidray", "known_found", None), ("pidray", "unknown_found", None),
                         ("pidray", "unknown_found", "hidden"), ("hixray", "unknown_found", None), ("opixray", "known_found", None)]:
        if t in AVAILABLE:
            cases += [(kind, sub) + c for c in pick("six_xguard", t, kind, kx, thr_x, sub)]
    if cases:
        fig, axes = plt.subplots(len(cases), 3, figsize=(11, 3.2 * len(cases)), squeeze=False)
        for row, (kind, sub, t, i) in zip(axes, cases):
            r = eval_set(t, True)[i]
            draw(row[0], r, None, kx, thr_x, "ground truth"); draw_gt(row[0], r)
            draw(row[1], r, preds_of("six_base", t)[i], kb, thr_b, "base")
            draw(row[2], r, preds_of("six_xguard", t)[i], kx, thr_x, "XGuard")
            row[0].set_ylabel(f"{t}{'/' + sub if sub else ''}\n{kind}", fontsize=8)
        fig.suptitle("Fig. 3  Known (green) and unknown (magenta) threat detection on real test images")
        plt.tight_layout(); savefig("fig3_qualitative")
    # ---------- Fig. 7 failure cases ----------
    fails = []
    for t in [x for x in ["sixray", "pidray", "hixray"] if x in AVAILABLE]:
        fails += [("missed",) + c for c in pick("six_xguard", t, "missed", kx, thr_x, k=1)]
        fails += [("false alarm",) + c for c in pick("six_xguard", t, "false_alarm", kx, thr_x, k=1)]
    if fails:
        fig, axes = plt.subplots(len(fails), 2, figsize=(8, 3.2 * len(fails)), squeeze=False)
        for row, (kind, t, i) in zip(axes, fails):
            r = eval_set(t, True)[i]
            draw(row[0], r, None, kx, thr_x, "ground truth"); draw_gt(row[0], r)
            draw(row[1], r, preds_of("six_xguard", t)[i], kx, thr_x, f"XGuard — {kind}")
            row[0].set_ylabel(t, fontsize=8)
        fig.suptitle("Fig. 7  Failure cases of XGuard (largest misses / false alarms)")
        plt.tight_layout(); savefig("fig7_failures")

# ---------- Fig. 4 occlusion robustness curves ----------
fig, ax = plt.subplots(figsize=(5, 3.6)); any_ = False
for t in ["six_base", "six_base-occaug", "six_oer", "six_sira+oer", "occ-cutout", "six_xguard"]:
    f = run_file(t, "metrics_occsweep.json")
    if f:
        d = jload(f); xs = sorted(float(k) for k in d)
        ax.plot([100 * x for x in xs], [100 * (d[str(x)] if d[str(x)] == d[str(x)] else np.nan) for x in xs], "o-", label=NICE.get(t, t)); any_ = True
if any_:
    ax.set_xlabel("synthetic occlusion of each object (%)"); ax.set_ylabel("AP50 (%)"); ax.legend(fontsize=7)
    ax.set_title("Fig. 4  Occlusion robustness (SIXray test, deterministic)")
    savefig("fig4_occlusion")
else:
    plt.close()

# ---------- Fig. 5 risk-coverage + referral ----------
tg5 = [t for t in ["pidray", "hixray", "sixray"] if run_file("six_xguard", f"curves_{t}.json")]
if tg5:
    fig, axes = plt.subplots(1, len(tg5) + 1, figsize=(3.6 * (len(tg5) + 1), 3.2), squeeze=False)
    grid = np.linspace(0.01, 1, 100)
    for ax, t in zip(axes[0], tg5):
        for tag, key, style, lab in [("six_base", "rc_msp", "--", "base · MSP"), ("six_xguard", "rc_msp", ":", "XGuard · MSP"),
                                     ("six_xguard", "rc_energy", "-.", "XGuard · energy"), ("six_xguard", "rc_rcr", "-", "XGuard · RCR")]:
            cv = jload(run_file(tag, f"curves_{t}.json") or "_none_", {}).get("all", {}).get(key)
            if cv:
                ax.plot(grid, cv, style, label=f"{lab} ({np.mean(cv) * 100:.1f})")
        ax.set_title(t); ax.set_xlabel("coverage"); ax.set_ylabel("selective risk"); ax.legend(fontsize=6)
    ax = axes[0][-1]
    for i, k in enumerate(RISK_TYPES):
        ax.bar(i, val("six_xguard", "pidray", f"miss_after_ref_{k}@10", fmt=False), color="#4C72B0")
    ax.set_xticks(range(len(RISK_TYPES))); ax.set_xticklabels(RISK_TYPES, rotation=30); ax.set_ylabel("missed threats after 10% referral (%)")
    ax.set_title("bag-level referral (PIDray)")
    fig.suptitle("Fig. 5  Risk–coverage (AURC in legend, lower is better) and operational referral", y=1.06)
    savefig("fig5_risk_referral")

# ---------- Fig. 6 CUS validity ----------
e = load_emb("six_xguard")
if e is not None and "synth" in e and len(e.get("synth", [])):
    known_x = (cfg_x or {}).get("known", [])
    Zn = e["z"] / np.linalg.norm(e["z"], axis=1, keepdims=True).clip(1e-6)
    mk = (e["ds"] == "sixray") & np.isin(e["cls"], known_x)
    mu = (e["ds"] == "pidray") & ~np.array([resolve_known(c, known_x) is not None for c in e["cls"]])
    pca = PCA(2).fit(np.concatenate([Zn[mk], e["synth"]]))
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
    for i, c in enumerate(sorted(set(e["cls"][mk]))):
        p = pca.transform(Zn[mk & (e["cls"] == c)]); axes[0].scatter(p[:, 0], p[:, 1], s=6, label=f"known {c}", alpha=0.6)
    p = pca.transform(e["synth"]); axes[0].scatter(p[:, 0], p[:, 1], s=8, marker="x", c="gray", label="CUS synthetic", alpha=0.5)
    if mu.any():
        p = pca.transform(Zn[mu]); axes[0].scatter(p[:, 0], p[:, 1], s=10, marker="^", c="crimson", label="REAL unseen (PIDray)", alpha=0.5)
    axes[0].legend(fontsize=6); axes[0].set_title("PCA of RoI embeddings")
    cen = np.stack([Zn[mk & (e["cls"] == c)].mean(0) for c in sorted(set(e["cls"][mk]))])
    cen /= np.linalg.norm(cen, axis=1, keepdims=True)
    for arr, lab in [(Zn[mk], "known (SIXray)"), (e["synth"], "CUS synthetic"), (Zn[mu] if mu.any() else None, "real unseen (PIDray)")]:
        if arr is not None and len(arr):
            axes[1].hist((arr @ cen.T).max(1), bins=40, alpha=0.5, density=True, label=lab)
    axes[1].set_xlabel("max cosine to a known-class centroid"); axes[1].legend(fontsize=7)
    axes[1].set_title("known-support overlap")
    fig.suptitle("Fig. 6  Are synthetic unknowns off the known support, like real unseen threats?")
    savefig("fig6_cus_validity")

# ---------- Fig. 8 per-class cross-domain AP heatmap ----------
rows = []
for tag in ["six_base", "six_xguard"]:
    for t in TG:
        f = run_file(tag, f"metrics_{t}.json")
        if f:
            for c, v in (jload(f)["all"].get("per_class_AP50") or {}).items():
                rows.append(dict(model=NICE.get(tag, tag), target=t, cls=c, AP50=100 * v if v == v else np.nan))
if rows:
    df = pd.DataFrame(rows)
    fig, axes = plt.subplots(1, df.model.nunique(), figsize=(4.5 * df.model.nunique(), 3.6), squeeze=False)
    for ax, (mname, g) in zip(axes[0], df.groupby("model")):
        piv = g.pivot_table(index="cls", columns="target", values="AP50")
        im_ = ax.imshow(piv.values, cmap="viridis", vmin=0, vmax=100)
        ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns); ax.set_yticks(range(len(piv.index))); ax.set_yticklabels(piv.index)
        for (i, j), v in np.ndenumerate(piv.values):
            ax.text(j, i, "n/a" if v != v else f"{v:.0f}", ha="center", va="center", fontsize=7, color="w")
        ax.set_title(mname, fontsize=9)
    fig.colorbar(im_, ax=axes[0].tolist(), shrink=0.8)
    fig.suptitle("Fig. 8  Per-class AP50 under cross-dataset shift (trained on SIXray only)")
    savefig("fig8_perclass_heatmap")


## 18 · Release gates

In [ ]:
# ==============================================================================
# 18. SCIENTIFIC RELEASE GATES (automatic). A failed gate is logged with its reason; results that fail a
#     gate must not go into the manuscript until the cause is fixed.
# ==============================================================================
if MODE == "prepare":
    raise StopExecution()   # prepare mode ends after the pack cell
jdump(TESTS, WORK / "logs" / f"unit_tests_{MODE}.json")
GATES = OrderedDict()


def gate(name, ok, detail, warn=False):
    GATES[name] = dict(status="PASS" if ok else ("WARN" if warn else "FAIL"), detail=detail)


# 1 dataset integrity
mism = VERIFICATION[(VERIFICATION.status == "MISMATCH")] if len(VERIFICATION) else VERIFICATION
mism_official = mism[mism.dataset != "sixray"] if len(mism) else mism
gate("dataset_integrity", len(mism_official) == 0,
     "train/val/test disjoint by ORIGINAL image id (asserted); official-size check: "
     + ("; ".join(f"{r.dataset}.{r.quantity} {r.observed}≠{r.official}" for r in mism.itertuples()) or "all match")
     + ". SIXray is a mirror split (documented), not the official one.")
# 2 ontology integrity
gate("ontology_integrity", TESTS.get("aliases_point_to_canonical") and TESTS.get("exact_track_ignores_alias") and TESTS.get("all_data_classes_have_superclass"),
     f"ontology {ONTOLOGY_VERSION}; aliases {sorted(SEM_ALIAS)}; exact / harmonized / hierarchical tracks separated")
# 3 CLIP integrity
text_runs = []
for rd in all_runs():
    sp = jload(rd / "config.json", {}).get("spec", {})
    st_ = STPB_ALIASES.get(sp.get("mcfg", {}).get("stpb", "none"), sp.get("mcfg", {}).get("stpb", "none"))
    if "t" in str(st_) and st_ != "none" and (rd / "train_meta.json").exists():
        text_runs.append((rd.name, jload(rd / "train_meta.json", {}).get("text_available")))
bad_text = [n for n, a in text_runs if a is False]
gate("clip_integrity", TESTS.get("clip_text_available") and not bad_text,
     f"CLIP {TEXT_INFO.get('source')} transformers={TEXT_INFO.get('transformers')} sha={TEXT_INFO.get('embedding_sha')}; "
     f"runs requesting text but trained WITHOUT it: {bad_text or 'none'}")
# 4 CUS integrity
skips = {}
for rd in all_runs():
    m_ = jload(rd / "train_meta.json", {})
    if m_.get("steps"):
        skips[rd.name] = m_.get("unk_skipped", 0) / max(1, m_["steps"])
heavy = {k: round(v, 3) for k, v in skips.items() if v > 0.2}
gate("cus_integrity", all(TESTS.get(k) for k in ["pairs_always_different_class", "pairs_single_class_returns_None",
                                                 "cus_single_class_returns_None", "cus_shape", "quantile_fp32_cast"]) and not heavy,
     f"different-class pairs guaranteed, fp32 quantiles, empty batches handled; runs skipping >20% of unknown steps: {heavy or 'none'}")
# 5 metric correctness
mt = [k for k in ["greedy_one_to_one", "ap_perfect", "ap_no_dets", "ap_zero_gt_is_nan", "toy_AP50_is_1", "toy_unknown_recall_one_to_one",
                  "toy_locAUROC_perfect", "zero_gt_not_applicable", "eaurc_perfect_zero", "eaurc_worst_positive"]]
evfail = []
for rd in all_runs():
    for mf in rd.glob("metrics_*.json"):
        for sub, M in (jload(mf, {}) or {}).items():
            if isinstance(M, dict):
                evfail += [f"{rd.name}/{mf.stem}/{sub}:{k}" for k, v in M.get("metric_status", {}).items() if str(v).startswith("evaluator_failure")]
gate("metric_correctness", all(TESTS.get(k) for k in mt) and not evfail, f"toy-case unit tests passed; evaluator failures: {evfail[:5] or 'none'}")
# 6 training reproducibility
rep = TESTS.get("reproducibility") or (jload(WORK / "logs" / "unit_tests_test.json", {}) or {}).get("reproducibility")
gate("training_reproducibility", bool(rep and rep.get("passed")),
     f"{rep}" if rep else "run MODE='test' first (it runs the same-seed and interrupted-resume checks)", warn=not rep)
# 7 fairness documentation
undoc = [rd.name for rd in all_runs() if (rd / "DONE").exists() and not (rd / "config.json").exists()]
gate("fairness_documented", not undoc,
     "every run stores init/pretraining, epochs, resolution, prompts (oracle vocabulary flagged), thresholds from SOURCE val only"
     + (f"; undocumented: {undoc}" if undoc else ""))
# 8 evidence completeness
_need_tiers = [1, 2] if MODE == "report" else TIERS
need = [s for s in REGISTRY if s["tier"] in _need_tiers and s["group"] in ("core", "ablation", "loco")]
done = [s for s in need if (run_path(s["name"]) / "DONE").exists()]
seeds_core = len({s["seed"] for s in need if s["tag"] == "six_xguard" and (run_path(s["name"]) / "DONE").exists()})
gate("evidence_complete", MODE in ("full", "report") and len(done) == len(need) and seeds_core >= 3,
     f"paper-critical runs (core, ablation, LOCO) finished {len(done)}/{len(need)}; core seeds {seeds_core}/3"
     + ("" if MODE in ("full", "report") else " (test mode: not applicable)"), warn=MODE == "test")
# 9 artifact integrity: every finished run's fingerprint matches the CURRENT code/data
stale = []
for s in SELECTED:
    rd = run_path(s["name"])
    if (rd / "DONE").exists() and s["kind"] not in ("owlvit", "gdino", "yolo"):
        tr, kn = train_set(s)
        fp = fingerprints(s, tr, val_set(s), kn); old = jload(rd / "fingerprint.json", {})
        if old.get("train_fp") != fp["train_fp"] or old.get("eval_fp") != fp["eval_fp"]:
            stale.append(s["name"])
errs = RUN_STATUS[RUN_STATUS.status.str.startswith("error")].run.tolist() if len(RUN_STATUS) else []
gate("artifact_integrity", not stale and not errs, f"stale runs: {stale or 'none'}; errored runs: {errs or 'none'}")

GATE_DF = pd.DataFrame(GATES).T
jdump(GATES, WORK / f"gates_{MODE}.json")
display(GATE_DF)
n_fail = (GATE_DF.status == "FAIL").sum()
print(f"\n{'✅ all gates passed' if n_fail == 0 else f'❌ {n_fail} gate(s) FAILED'} (MODE={MODE})")
if MODE == "test":
    if n_fail == 0:
        jdump(dict(passed=time.strftime("%Y-%m-%d %H:%M"), gates=GATES), WORK / "logs" / "test_passed.json")
        print("✅ test passed -> next time you press Run all on a GPU runtime, MODE 'auto' starts the FULL experiments.")
    else:
        print("❌ test did not pass -> MODE 'auto' will run the test again; send me the gate table above.")


## 19 · Package results

In [ ]:
# ==============================================================================
# 19. PACKAGE deliverables (tables, figures, audit, manifests, metrics, gates; no weights / predictions)
# ==============================================================================
if MODE == "prepare":
    raise StopExecution()   # prepare mode ends after the pack cell
pkg = CACHE / f"package_{MODE}"
if pkg.exists():
    shutil.rmtree(pkg)
pkg.mkdir(parents=True)
for f in [f"config_{MODE}.json", "dataset_stats.json", "class_mapping.json", f"gates_{MODE}.json"]:
    if (WORK / f).exists():
        shutil.copy(WORK / f, pkg / f)
for d in ["audit", "logs"]:
    if (WORK / d).exists():
        shutil.copytree(WORK / d, pkg / d, ignore=shutil.ignore_patterns("_repro_*"))
shutil.copytree(TAB, pkg / "tables"); shutil.copytree(FIG, pkg / "figures")
for rd in all_runs():
    if rd.is_dir() and not rd.name.startswith("_"):
        files = list(rd.glob("*.json")) + list(rd.glob("*.txt")) + list(rd.glob("*.csv")) + list(rd.glob("*.npz")) + list(rd.glob("DONE"))
        if files:
            (pkg / "runs" / rd.name).mkdir(parents=True, exist_ok=True)
            for f in files:
                shutil.copy(f, pkg / "runs" / rd.name / f.name)
zp = shutil.make_archive(str(WORK / f"xguard_results_{MODE}"), "zip", pkg)
print("results archive:", zp, f"({os.path.getsize(zp) / 1e6:.1f} MB)")
todo = [s["name"] for s in SELECTED if not (run_path(s["name"]) / "DONE").exists()]
print(f"session time used: {elapsed_h():.2f} h | selected runs still to do: {len(todo)}")
if todo:
    print("➡  re-run the notebook (same MODE / TIERS) to continue; finished runs are kept on Drive and skipped.")
if PLATFORM == "kaggle" and MODE == "full":
    print("Kaggle: download", zp, "and put it into MyDrive/xguard/kaggle_results/ -> Colab MODE='report' merges everything.")
if PLATFORM == "colab" and MODE == "test":
    release_runtime("test finished — read the gates and the time estimate above, if the test passed, just Connect a GPU runtime and Run all again: MODE 'auto' starts the full experiments")
